# 205 — TCGA Epigenetic-Transcriptomic Program Discovery

## Objective

Identify candidate epigenetic-transcriptomic programs across the frozen TCGA primary-tumor multi-omic cohort.

This notebook will:

* characterize the dominant structure of the transcriptomic and DNA-methylation data using PCA;
* perform primary candidate-program discovery using ICA;
* use NMF as a complementary sensitivity analysis;
* generate sample-level program scores and feature-level loadings for downstream robustness assessment.

The analysis uses the authoritative multi-omic cohort and confounder artifacts frozen in notebooks 203 and 204.

Programs identified here are considered **candidate programs**. Their stability, cross-lineage recurrence, confounder sensitivity, and methodological robustness will be evaluated separately in notebook 206.


In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Load and inspect authoritative multi-omic metadata path
# =============================================================================

MULTIOMIC_METADATA_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_final_consumables_metadata.json"
)

with MULTIOMIC_METADATA_PATH.open(encoding="utf-8") as file:
    multiomic_metadata = json.load(file)

sorted(multiomic_metadata)

['analysis_layer',
 'artifact_type',
 'cohort_definition',
 'final_artifacts',
 'matrix_policy',
 'notebook',
 'probe_mapping',
 'recorded_at_utc',
 'selection_policy',
 'source_artifacts',
 'status']

In [3]:
# =============================================================================
# Inspect final multi-omic artifacts
# =============================================================================

multiomic_metadata["final_artifacts"]

{'hm27_matrix': {'column_axis': 'hm27_matrix_column_index',
  'dtype': 'float32',
  'fortran_order': True,
  'path': 'data/interim/methylation/tcga_primary_tumor_methylation_hm27_final_case_level_beta_values.npy',
  'row_axis': 'hm27_probe_mapping.matrix_row_index',
  'selected_missing_beta_count': 271731,
  'selected_nonmissing_beta_count': 39099129,
  'sha256': '9bbdde971ac47048508e04a5ac0523b98f622d8330155532cdfa61069ce0377f',
  'shape': [24303, 1620],
  'size_bytes': 157483568},
 'hm450_matrix': {'build_metadata_path': 'data/interim/methylation/tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.build_metadata.json',
  'column_axis': 'hm450_matrix_column_index',
  'dtype': 'float32',
  'fortran_order': True,
  'path': 'data/interim/methylation/tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.npy',
  'row_axis': 'hm450_probe_mapping.matrix_row_index',
  'selected_missing_beta_count': 26490075,
  'selected_nonmissing_beta_count': 3375733045,
  'sha256':

In [4]:
# =============================================================================
# Select authoritative discovery artifacts
# =============================================================================

final_artifacts = multiomic_metadata["final_artifacts"]

rna_artifact = final_artifacts["rna_matrix"]
methylation_artifact = final_artifacts["shared_methylation_matrix"]
rna_feature_artifact = final_artifacts["rna_feature_index"]
probe_mapping_artifact = final_artifacts["probe_mapping"]
sample_mapping_artifact = final_artifacts["sample_mapping"]

In [5]:
# =============================================================================
# Resolve authoritative artifact paths
# =============================================================================

RNA_MATRIX_PATH = Paths.expression / rna_artifact["path"].rsplit("/", 1)[-1]
METHYLATION_MATRIX_PATH = (
    Paths.methylation / methylation_artifact["path"].rsplit("/", 1)[-1]
)

RNA_FEATURE_INDEX_PATH = (
    Paths.expression / rna_feature_artifact["path"].rsplit("/", 1)[-1]
)
PROBE_MAPPING_PATH = (
    Paths.metadata / probe_mapping_artifact["path"].rsplit("/", 1)[-1]
)
SAMPLE_MAPPING_PATH = (
    Paths.metadata / sample_mapping_artifact["path"].rsplit("/", 1)[-1]
)

In [6]:
# ======================================================
# Load final sample mapping and RNA-seq feature index
# ======================================================

sample_mapping = pd.read_csv(SAMPLE_MAPPING_PATH)

rna_feature_index = pd.read_csv(RNA_FEATURE_INDEX_PATH)

In [7]:
# =====================================================
# Load and inspect methylation probe mapping
# =====================================================

probe_mapping = pd.read_csv(PROBE_MAPPING_PATH, low_memory=False)

probe_mapping["representation"].value_counts(dropna=False)

representation
hm450                407696
hm27                  24303
shared_hm27_hm450     23356
Name: count, dtype: int64

In [8]:
# ====================================================
# Prepare shared HM27–HM450 probe mapping
# ====================================================

shared_probe_mapping = (
    probe_mapping.loc[
        probe_mapping["representation"].eq("shared_hm27_hm450")
    ]
    .sort_values("matrix_row_index")
    .reset_index(drop=True)
)

del probe_mapping

shared_probe_mapping.shape

(23356, 14)

In [9]:
# =======================================================
# Load authoritative multi-omic matrices
# =======================================================

rna_counts = np.load(RNA_MATRIX_PATH, mmap_mode="r")
methylation_betas = np.load(METHYLATION_MATRIX_PATH, mmap_mode="r")

rna_counts.shape, methylation_betas.shape

((60660, 9965), (23356, 9965))

In [10]:
# =======================================================
# Prepare sample-level discovery metadata
# =======================================================

sample_metadata = (
    sample_mapping[
        [
            "final_sample_column_index",
            "case_submitter_id",
            "sample_submitter_id",
            "project_id",
            "methylation_platform",
        ]
    ]
    .sort_values("final_sample_column_index")
    .reset_index(drop=True)
)

sample_metadata.shape

(9965, 5)

In [11]:
# =======================================================
# Prepare and inspect RNA-seq feature metadata
# =======================================================

rna_feature_metadata = (
    rna_feature_index
    .sort_values("matrix_row_index")
    .reset_index(drop=True)
)

rna_feature_metadata["gene_type"].value_counts(dropna=False)

gene_type
protein_coding                        19962
lncRNA                                16901
processed_pseudogene                  10167
unprocessed_pseudogene                 2614
misc_RNA                               2212
snRNA                                  1901
miRNA                                  1881
TEC                                    1057
snoRNA                                  943
transcribed_unprocessed_pseudogene      939
transcribed_processed_pseudogene        500
rRNA_pseudogene                         497
IG_V_pseudogene                         187
IG_V_gene                               145
transcribed_unitary_pseudogene          138
TR_V_gene                               106
unitary_pseudogene                       98
TR_J_gene                                79
scaRNA                                   49
polymorphic_pseudogene                   48
rRNA                                     47
IG_D_gene                                37
TR_V_pseudogene       

In [12]:
# =======================================================
# Select protein-coding genes
# =======================================================

protein_coding_mask = rna_feature_metadata["gene_type"].eq(
    "protein_coding"
)

rna_discovery_features = (
    rna_feature_metadata.loc[protein_coding_mask]
    .reset_index(drop=True)
)

rna_discovery_features.shape

(19962, 6)

In [13]:
# =======================================================
# Define RNA-seq discovery row indices
# =======================================================

rna_discovery_row_indices = (
    rna_discovery_features["matrix_row_index"]
    .to_numpy(dtype=np.int64)
)

rna_discovery_row_indices.shape

(19962,)

In [14]:
# =======================================================
# Compute RNA-seq library sizes
# =======================================================

rna_library_sizes = np.sum(
    rna_counts,
    axis=0,
    dtype=np.uint64,
)

pd.Series(
    rna_library_sizes,
    name="rna_library_size",
).describe(percentiles=[0.05, 0.50, 0.95])

count    9.965000e+03
mean     5.213251e+07
std      1.745014e+07
min      1.456498e+06
5%       2.332328e+07
50%      5.110139e+07
95%      8.185867e+07
max      1.608347e+08
Name: rna_library_size, dtype: float64

In [15]:
# =======================================================
# Inspect project sample counts
# =======================================================

project_sample_counts = (
    sample_metadata["project_id"]
    .value_counts()
    .sort_values()
)

project_sample_counts.describe(), project_sample_counts.head(10)

(count      33.000000
 mean      301.969697
 std       225.163203
 min        35.000000
 25%       120.000000
 50%       259.000000
 75%       487.000000
 max      1089.000000
 Name: count, dtype: float64,
 project_id
 TCGA-CHOL     35
 TCGA-DLBC     48
 TCGA-UCS      57
 TCGA-KICH     66
 TCGA-ACC      79
 TCGA-UVM      80
 TCGA-MESO     87
 TCGA-SKCM    103
 TCGA-THYM    120
 TCGA-LAML    134
 Name: count, dtype: int64)

In [16]:
# =======================================================
# Define RNA-seq expression-filtering policy
# =======================================================

MIN_CPM = 1.0
MIN_EXPRESSED_SAMPLES = int(project_sample_counts.min())

MIN_CPM, MIN_EXPRESSED_SAMPLES

(1.0, 35)

In [17]:
# =======================================================
# Count RNA-seq expression prevalence
# =======================================================

RNA_FILTER_CHUNK_SIZE = 512

minimum_counts_per_sample = (
    rna_library_sizes * MIN_CPM / 1_000_000
)

rna_expressed_sample_counts = np.empty(
    len(rna_discovery_row_indices),
    dtype=np.int32,
)

for start in range(
    0,
    len(rna_discovery_row_indices),
    RNA_FILTER_CHUNK_SIZE,
):
    stop = start + RNA_FILTER_CHUNK_SIZE
    row_indices = rna_discovery_row_indices[start:stop]

    rna_expressed_sample_counts[start:stop] = np.sum(
        rna_counts[row_indices, :] >= minimum_counts_per_sample,
        axis=1,
    )

pd.Series(rna_expressed_sample_counts).describe()

count    19962.000000
mean      6695.147931
std       4062.413678
min          0.000000
25%       2122.250000
50%       9644.500000
75%       9965.000000
max       9965.000000
dtype: float64

In [18]:
# =======================================================
# Apply RNA-seq expression filter
# =======================================================

rna_expression_eligible = (
    rna_expressed_sample_counts >= MIN_EXPRESSED_SAMPLES
)

rna_filtered_features = (
    rna_discovery_features.loc[rna_expression_eligible]
    .copy()
    .reset_index(drop=True)
)

rna_filtered_features["expressed_sample_count"] = (
    rna_expressed_sample_counts[rna_expression_eligible]
)

pd.Series(
    {
        "protein_coding_genes": len(rna_discovery_features),
        "retained_genes": int(rna_expression_eligible.sum()),
        "excluded_genes": int((~rna_expression_eligible).sum()),
    }
)

protein_coding_genes    19962
retained_genes          18123
excluded_genes           1839
dtype: int64

In [19]:
# =======================================================
# Define filtered RNA-seq row indices
# =======================================================

rna_filtered_row_indices = (
    rna_filtered_features["matrix_row_index"]
    .to_numpy(dtype=np.int64)
)

rna_filtered_row_indices.shape

(18123,)

In [20]:
# =======================================================
# Define Python–R exchange artifact paths
# =======================================================

RNA_COUNTS_H5_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_filtered_raw_counts.h5"
)

RNA_GENE_METADATA_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_filtered_gene_metadata.csv"
)

RNA_SAMPLE_METADATA_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_sample_metadata.csv"
)

RNA_TMM_LOGCPM_H5_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_tmm_logcpm.h5"
)

In [21]:
# =======================================================
# Write Python–R exchange metadata
# =======================================================

rna_filtered_features.to_csv(
    RNA_GENE_METADATA_PATH,
    index=False,
)

sample_metadata.to_csv(
    RNA_SAMPLE_METADATA_PATH,
    index=False,
)

project_relative_path(RNA_GENE_METADATA_PATH), project_relative_path(RNA_SAMPLE_METADATA_PATH)

('data/interim/expression/tcga_primary_tumor_rnaseq_program_discovery_filtered_gene_metadata.csv',
 'data/interim/expression/tcga_primary_tumor_rnaseq_program_discovery_sample_metadata.csv')

In [22]:
# =======================================================
# Write filtered RNA-seq counts to HDF5
# =======================================================

import h5py

RNA_H5_CHUNK_SIZE = 256

with h5py.File(RNA_COUNTS_H5_PATH, "w") as h5_file:
    counts_dataset = h5_file.create_dataset(
        "counts",
        shape=(
            len(rna_filtered_row_indices),
            rna_counts.shape[1],
        ),
        dtype=np.uint32,
        chunks=(RNA_H5_CHUNK_SIZE, 256),
        compression="gzip",
        compression_opts=4,
        shuffle=True,
    )

    for start in range(
        0,
        len(rna_filtered_row_indices),
        RNA_H5_CHUNK_SIZE,
    ):
        stop = min(
            start + RNA_H5_CHUNK_SIZE,
            len(rna_filtered_row_indices),
        )

        counts_dataset[start:stop, :] = rna_counts[
            rna_filtered_row_indices[start:stop],
            :,
        ]

    counts_dataset.attrs["orientation"] = "genes_x_samples"

project_relative_path(RNA_COUNTS_H5_PATH)

'data/interim/expression/tcga_primary_tumor_rnaseq_program_discovery_filtered_raw_counts.h5'

In [23]:
# =======================================================
# Verify Python–R exchange artifacts
# =======================================================

with h5py.File(RNA_COUNTS_H5_PATH, "r") as h5_file:
    counts_dataset = h5_file["counts"]

    exchange_summary = {
        "counts_shape": counts_dataset.shape,
        "counts_dtype": str(counts_dataset.dtype),
        "orientation": counts_dataset.attrs["orientation"],
        "gene_metadata_rows": len(
            pd.read_csv(RNA_GENE_METADATA_PATH)
        ),
        "sample_metadata_rows": len(
            pd.read_csv(RNA_SAMPLE_METADATA_PATH)
        ),
    }

exchange_summary

{'counts_shape': (18123, 9965),
 'counts_dtype': 'uint32',
 'orientation': 'genes_x_samples',
 'gene_metadata_rows': 18123,
 'sample_metadata_rows': 9965}

In [24]:
# =======================================================
# Load TMM-normalized RNA-seq artifacts
# =======================================================

RNA_TMM_FACTORS_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_tmm_factors.csv"
)

rna_tmm_factors = pd.read_csv(RNA_TMM_FACTORS_PATH)

rna_tmm_logcpm_h5 = h5py.File(
    RNA_TMM_LOGCPM_H5_PATH,
    "r",
)

rna_tmm_logcpm = rna_tmm_logcpm_h5["logcpm"]

{
    "logcpm_shape": rna_tmm_logcpm.shape,
    "logcpm_dtype": str(rna_tmm_logcpm.dtype),
    "orientation": rna_tmm_logcpm.attrs["orientation"],
    "tmm_factor_rows": len(rna_tmm_factors),
}

{'logcpm_shape': (18123, 9965),
 'logcpm_dtype': 'float64',
 'orientation': array([b'genes_x_samples'], dtype='|S16'),
 'tmm_factor_rows': 9965}

In [25]:
# =======================================================
# Load normalized RNA-seq matrix as float32
# =======================================================

rna_logcpm = np.empty(
    rna_tmm_logcpm.shape,
    dtype=np.float32,
)

rna_tmm_logcpm.read_direct(rna_logcpm)
rna_tmm_logcpm_h5.close()

rna_logcpm.shape, rna_logcpm.dtype

((18123, 9965), dtype('float32'))

In [26]:
# =======================================================
# Prepare sample-by-gene RNA-seq representation
# =======================================================

rna_sample_by_gene = rna_logcpm.T

rna_sample_by_gene.shape, rna_sample_by_gene.dtype

((9965, 18123), dtype('float32'))

In [27]:
# =======================================================
# Compute RNA-seq gene-level variability
# =======================================================

rna_gene_means = np.mean(
    rna_logcpm,
    axis=1,
    dtype=np.float64,
)

rna_gene_variances = np.var(
    rna_logcpm,
    axis=1,
    dtype=np.float64,
)

pd.DataFrame(
    {
        "mean_logcpm": rna_gene_means,
        "variance_logcpm": rna_gene_variances,
    }
).describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,mean_logcpm,variance_logcpm
count,18123.000000,18123.000000
mean,2.661672,2.561298
std,3.416335,3.025388
min,-4.614917,0.119279
25%,-0.041975,0.561575
50%,3.589718,1.395429
75%,5.300844,3.410242
90%,6.396015,6.356638
95%,7.066560,8.541976
99%,8.578937,14.311763


In [28]:
# =======================================================
# Select variable RNA-seq features
# =======================================================

N_VARIABLE_GENES = 5_000

rna_feature_statistics = rna_filtered_features.copy()

rna_feature_statistics["filtered_matrix_row_index"] = np.arange(
    len(rna_feature_statistics),
    dtype=np.int32,
)

rna_feature_statistics["mean_logcpm"] = rna_gene_means
rna_feature_statistics["variance_logcpm"] = rna_gene_variances

rna_variable_row_indices = np.sort(
    np.argsort(rna_gene_variances)[-N_VARIABLE_GENES:]
)

rna_variable_features = (
    rna_feature_statistics
    .iloc[rna_variable_row_indices]
    .reset_index(drop=True)
)

{
    "selected_genes": len(rna_variable_features),
    "minimum_selected_variance": (
        rna_variable_features["variance_logcpm"].min()
    ),
}

{'selected_genes': 5000,
 'minimum_selected_variance': np.float64(3.1001446128434447)}

In [29]:
# =======================================================
# Prepare variable-gene RNA-seq matrix
# =======================================================

rna_variable_matrix = np.ascontiguousarray(
    rna_sample_by_gene[:, rna_variable_row_indices],
    dtype=np.float32,
)

rna_variable_matrix.shape, rna_variable_matrix.dtype

((9965, 5000), dtype('float32'))

In [30]:
# =======================================================
# Fit diagnostic RNA-seq PCA
# =======================================================

from sklearn.decomposition import PCA

RNA_PCA_COMPONENTS = 100
RANDOM_STATE = 42

rna_pca = PCA(
    n_components=RNA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=RANDOM_STATE,
)

rna_pca_scores = rna_pca.fit_transform(
    rna_sample_by_gene
)

{
    "score_shape": rna_pca_scores.shape,
    "loading_shape": rna_pca.components_.shape,
    "explained_variance_100_pcs": float(
        rna_pca.explained_variance_ratio_.sum()
    ),
}

{'score_shape': (9965, 100),
 'loading_shape': (100, 18123),
 'explained_variance_100_pcs': 0.7437160611152649}

In [31]:
# =======================================================
# Summarize RNA-seq PCA explained variance
# =======================================================

rna_pca_variance = pd.DataFrame(
    {
        "principal_component": np.arange(
            1,
            RNA_PCA_COMPONENTS + 1,
        ),
        "explained_variance_percent": (
            rna_pca.explained_variance_ratio_ * 100
        ),
        "cumulative_variance_percent": (
            np.cumsum(
                rna_pca.explained_variance_ratio_
            )
            * 100
        ),
    }
)

rna_pca_variance.loc[
    rna_pca_variance["principal_component"].isin(
        [1, 2, 5, 10, 20, 50, 100]
    )
]

,principal_component,explained_variance_percent,cumulative_variance_percent
0,1,13.522717,13.522717
1,2,7.147131,20.669847
4,5,4.168949,34.232815
9,10,2.176897,48.105850
19,20,0.673307,59.306984
49,50,0.168421,69.154144
99,100,0.068477,74.371582


In [32]:
# =======================================================
# Identify RNA-seq PCA variance thresholds
# =======================================================

rna_pca_cumulative_variance = np.cumsum(
    rna_pca.explained_variance_ratio_
)

pd.Series(
    {
        f"pcs_for_{threshold}%": (
            np.searchsorted(
                rna_pca_cumulative_variance,
                threshold / 100,
            )
            + 1
        )
        for threshold in [50, 60, 70]
    }
)

pcs_for_50%    11
pcs_for_60%    22
pcs_for_70%    56
dtype: int64

In [33]:
# =======================================================
# Combine RNA-seq PCA scores with sample metadata
# =======================================================

rna_pca_score_columns = [
    f"PC{component}"
    for component in range(1, RNA_PCA_COMPONENTS + 1)
]

rna_pca_score_metadata = pd.concat(
    [
        sample_metadata.reset_index(drop=True),
        pd.DataFrame(
            rna_pca_scores,
            columns=rna_pca_score_columns,
        ),
    ],
    axis=1,
)

rna_pca_score_metadata.shape

(9965, 105)

In [34]:
# =======================================================
# Quantify project-associated PCA structure
# =======================================================

project_counts = (
    rna_pca_score_metadata
    .groupby("project_id", observed=True)
    .size()
)

project_pc_means = (
    rna_pca_score_metadata
    .groupby("project_id", observed=True)[rna_pca_score_columns]
    .mean()
)

pc_grand_means = rna_pca_scores.mean(axis=0)

pc_total_sum_squares = np.sum(
    (rna_pca_scores - pc_grand_means) ** 2,
    axis=0,
)

pc_between_project_sum_squares = np.sum(
    project_counts.to_numpy()[:, None]
    * (
        project_pc_means.to_numpy()
        - pc_grand_means
    ) ** 2,
    axis=0,
)

rna_pca_project_association = pd.DataFrame(
    {
        "principal_component": np.arange(
            1,
            RNA_PCA_COMPONENTS + 1,
        ),
        "project_eta_squared": (
            pc_between_project_sum_squares
            / pc_total_sum_squares
        ),
        "explained_variance_percent": (
            rna_pca.explained_variance_ratio_ * 100
        ),
    }
)

rna_pca_project_association.head(20)

,principal_component,project_eta_squared,explained_variance_percent
0,1,0.918351,13.522717
1,2,0.882182,7.147131
2,3,0.776759,4.900112
3,4,0.719076,4.493908
4,5,0.717676,4.168949
5,6,0.789685,3.388629
6,7,0.799746,3.195613
7,8,0.685154,2.719055
8,9,0.661838,2.392840
9,10,0.670198,2.176897


In [35]:
# =======================================================
# Summarize project-associated PCA variance
# =======================================================

rna_pca_project_variance_summary = pd.DataFrame(
    [
        {
            "n_components": n_components,
            "captured_variance_percent": (
                rna_pca.explained_variance_ratio_[:n_components].sum()
                * 100
            ),
            "project_associated_variance_percent": (
                np.sum(
                    rna_pca.explained_variance_ratio_[:n_components]
                    * rna_pca_project_association[
                        "project_eta_squared"
                    ].to_numpy()[:n_components]
                )
                * 100
            ),
        }
        for n_components in [10, 20, 50, 100]
    ]
)

rna_pca_project_variance_summary[
    "weighted_project_eta_squared"
] = (
    rna_pca_project_variance_summary[
        "project_associated_variance_percent"
    ]
    / rna_pca_project_variance_summary[
        "captured_variance_percent"
    ]
)

rna_pca_project_variance_summary

,n_components,captured_variance_percent,project_associated_variance_percent,weighted_project_eta_squared
0,10,48.105850,38.890524,0.808436
1,20,59.306992,46.345607,0.781453
2,50,69.154144,49.651115,0.717977
3,100,74.371605,49.917874,0.671195


In [36]:
# =======================================================
# Rank genes by within-project variability
# =======================================================

project_labels = sample_metadata["project_id"].to_numpy()
project_ids = np.sort(sample_metadata["project_id"].unique())

rna_project_variances = np.empty(
    (len(project_ids), rna_logcpm.shape[0]),
    dtype=np.float32,
)

for project_index, project_id in enumerate(project_ids):
    project_sample_indices = np.flatnonzero(
        project_labels == project_id
    )

    rna_project_variances[project_index] = np.var(
        rna_sample_by_gene[project_sample_indices],
        axis=0,
        ddof=1,
        dtype=np.float64,
    )

rna_median_within_project_variance = np.median(
    rna_project_variances,
    axis=0,
)

pd.Series(
    rna_median_within_project_variance
).describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    18123.000000
mean         1.123409
std          1.136433
min          0.000000
25%          0.325481
50%          0.690400
75%          1.577014
90%          2.631338
95%          3.345187
99%          4.887442
max         23.561235
dtype: float64

In [37]:
# =======================================================
# Select lineage-aware variable RNA-seq features
# =======================================================

N_ICA_GENES = 5_000

rna_ica_row_indices = np.sort(
    np.argsort(
        rna_median_within_project_variance
    )[-N_ICA_GENES:]
)

rna_ica_features = (
    rna_feature_statistics
    .iloc[rna_ica_row_indices]
    .copy()
    .reset_index(drop=True)
)

rna_ica_features["median_within_project_variance"] = (
    rna_median_within_project_variance[
        rna_ica_row_indices
    ]
)

global_selection_overlap = np.intersect1d(
    rna_ica_row_indices,
    rna_variable_row_indices,
).size

{
    "selected_genes": len(rna_ica_features),
    "minimum_selected_median_variance": float(
        rna_ica_features[
            "median_within_project_variance"
        ].min()
    ),
    "overlap_with_global_selection": int(
        global_selection_overlap
    ),
    "overlap_percent": (
        global_selection_overlap
        / N_ICA_GENES
        * 100
    ),
}

{'selected_genes': 5000,
 'minimum_selected_median_variance': 1.4517713785171509,
 'overlap_with_global_selection': 4265,
 'overlap_percent': 85.3}

In [38]:
# =======================================================
# Prepare lineage-centered RNA-seq ICA input
# =======================================================

rna_ica_input = np.ascontiguousarray(
    rna_sample_by_gene[:, rna_ica_row_indices],
    dtype=np.float32,
)

for project_id in project_ids:
    project_sample_indices = np.flatnonzero(
        project_labels == project_id
    )

    project_gene_means = np.mean(
        rna_ica_input[project_sample_indices],
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    rna_ica_input[project_sample_indices] -= (
        project_gene_means
    )

maximum_residual_project_mean = max(
    float(
        np.abs(
            np.mean(
                rna_ica_input[
                    np.flatnonzero(
                        project_labels == project_id
                    )
                ],
                axis=0,
                dtype=np.float64,
            )
        ).max()
    )
    for project_id in project_ids
)

{
    "ica_input_shape": rna_ica_input.shape,
    "ica_input_dtype": str(rna_ica_input.dtype),
    "maximum_residual_project_mean": (
        maximum_residual_project_mean
    ),
}

{'ica_input_shape': (9965, 5000),
 'ica_input_dtype': 'float32',
 'maximum_residual_project_mean': 4.76837158203125e-07}

In [39]:
# =======================================================
# Fit PCA to lineage-centered RNA-seq input
# =======================================================

RNA_ICA_PCA_COMPONENTS = 100

rna_ica_pca = PCA(
    n_components=RNA_ICA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=RANDOM_STATE,
)

rna_ica_pca_scores = rna_ica_pca.fit_transform(
    rna_ica_input
)

rna_ica_pca_cumulative_variance = np.cumsum(
    rna_ica_pca.explained_variance_ratio_
)

{
    "score_shape": rna_ica_pca_scores.shape,
    "explained_variance_100_pcs": float(
        rna_ica_pca_cumulative_variance[-1]
    ),
    "pcs_for_50_percent": int(
        np.searchsorted(
            rna_ica_pca_cumulative_variance,
            0.50,
        )
        + 1
    ),
    "pcs_for_60_percent": int(
        np.searchsorted(
            rna_ica_pca_cumulative_variance,
            0.60,
        )
        + 1
    ),
    "pcs_for_70_percent": int(
        np.searchsorted(
            rna_ica_pca_cumulative_variance,
            0.70,
        )
        + 1
    ),
}

{'score_shape': (9965, 100),
 'explained_variance_100_pcs': 0.5022695660591125,
 'pcs_for_50_percent': 99,
 'pcs_for_60_percent': 101,
 'pcs_for_70_percent': 101}

In [40]:
# =======================================================
# Summarize lineage-centered PCA dimensionality
# =======================================================

def report_pca_threshold(
    cumulative_variance,
    threshold,
):
    if cumulative_variance[-1] < threshold:
        return f">{len(cumulative_variance)}"

    return int(
        np.searchsorted(
            cumulative_variance,
            threshold,
        )
        + 1
    )


{
    "score_shape": rna_ica_pca_scores.shape,
    "explained_variance_100_pcs": float(
        rna_ica_pca_cumulative_variance[-1]
    ),
    "pcs_for_50_percent": report_pca_threshold(
        rna_ica_pca_cumulative_variance,
        0.50,
    ),
    "pcs_for_60_percent": report_pca_threshold(
        rna_ica_pca_cumulative_variance,
        0.60,
    ),
    "pcs_for_70_percent": report_pca_threshold(
        rna_ica_pca_cumulative_variance,
        0.70,
    ),
    "pcs_for_99_percent": report_pca_threshold(
        rna_ica_pca_cumulative_variance,
        0.99,
    ),
}

{'score_shape': (9965, 100),
 'explained_variance_100_pcs': 0.5022695660591125,
 'pcs_for_50_percent': 99,
 'pcs_for_60_percent': '>100',
 'pcs_for_70_percent': '>100',
 'pcs_for_99_percent': '>100'}

In [41]:
# =======================================================
# Expand lineage-centered RNA-seq PCA
# =======================================================

RNA_ICA_PCA_COMPONENTS = 500

rna_ica_pca = PCA(
    n_components=RNA_ICA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=RANDOM_STATE,
)

rna_ica_pca_scores = rna_ica_pca.fit_transform(
    rna_ica_input
)

rna_ica_pca_cumulative_variance = np.cumsum(
    rna_ica_pca.explained_variance_ratio_
)

variance_thresholds = {}

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    if rna_ica_pca_cumulative_variance[-1] >= threshold:
        variance_thresholds[
            f"pcs_for_{int(threshold * 100)}_percent"
        ] = int(
            np.searchsorted(
                rna_ica_pca_cumulative_variance,
                threshold,
            )
            + 1
        )
    else:
        variance_thresholds[
            f"pcs_for_{int(threshold * 100)}_percent"
        ] = f">{RNA_ICA_PCA_COMPONENTS}"

{
    "score_shape": rna_ica_pca_scores.shape,
    "cumulative_variance_100_pcs": float(
        rna_ica_pca_cumulative_variance[99]
    ),
    "cumulative_variance_200_pcs": float(
        rna_ica_pca_cumulative_variance[199]
    ),
    "cumulative_variance_300_pcs": float(
        rna_ica_pca_cumulative_variance[299]
    ),
    "cumulative_variance_400_pcs": float(
        rna_ica_pca_cumulative_variance[399]
    ),
    "cumulative_variance_500_pcs": float(
        rna_ica_pca_cumulative_variance[499]
    ),
    **variance_thresholds,
}

{'score_shape': (9965, 500),
 'cumulative_variance_100_pcs': 0.5025843977928162,
 'cumulative_variance_200_pcs': 0.5875036716461182,
 'cumulative_variance_300_pcs': 0.6390577554702759,
 'cumulative_variance_400_pcs': 0.6767864227294922,
 'cumulative_variance_500_pcs': 0.7057203650474548,
 'pcs_for_50_percent': 98,
 'pcs_for_60_percent': 221,
 'pcs_for_70_percent': 478,
 'pcs_for_80_percent': '>500',
 'pcs_for_90_percent': '>500'}

In [42]:
# =======================================================
# Fit primary lineage-centered RNA-seq ICA
# =======================================================

from sklearn.decomposition import FastICA

RNA_ICA_COMPONENTS = 200
RNA_ICA_MAX_ITER = 5_000

rna_ica_model_input = np.ascontiguousarray(
    rna_ica_pca_scores[:, :RNA_ICA_COMPONENTS],
    dtype=np.float64,
)

rna_ica = FastICA(
    n_components=RNA_ICA_COMPONENTS,
    algorithm="parallel",
    whiten="unit-variance",
    whiten_solver="svd",
    fun="logcosh",
    max_iter=RNA_ICA_MAX_ITER,
    tol=1e-4,
    random_state=RANDOM_STATE,
)

rna_ica_scores = rna_ica.fit_transform(
    rna_ica_model_input
)

{
    "input_shape": rna_ica_model_input.shape,
    "score_shape": rna_ica_scores.shape,
    "mixing_shape": rna_ica.mixing_.shape,
    "unmixing_shape": rna_ica.components_.shape,
    "iterations": int(rna_ica.n_iter_),
    "reached_iteration_limit": bool(
        rna_ica.n_iter_ >= RNA_ICA_MAX_ITER
    ),
}

{'input_shape': (9965, 200),
 'score_shape': (9965, 200),
 'mixing_shape': (200, 200),
 'unmixing_shape': (200, 200),
 'iterations': 133,
 'reached_iteration_limit': False}

In [43]:
# =======================================================
# Recover and orient RNA-seq ICA gene loadings
# =======================================================

rna_ica_gene_loadings = (
    rna_ica.mixing_.T
    @ rna_ica_pca.components_[:RNA_ICA_COMPONENTS]
).astype(np.float32)

largest_loading_indices = np.argmax(
    np.abs(rna_ica_gene_loadings),
    axis=1,
)

rna_ica_component_signs = np.where(
    rna_ica_gene_loadings[
        np.arange(RNA_ICA_COMPONENTS),
        largest_loading_indices,
    ] >= 0,
    1.0,
    -1.0,
).astype(np.float32)

rna_ica_scores_oriented = (
    rna_ica_scores
    * rna_ica_component_signs[None, :]
).astype(np.float32)

rna_ica_gene_loadings_oriented = (
    rna_ica_gene_loadings
    * rna_ica_component_signs[:, None]
)

rna_ica_component_names = [
    f"RNA_IC{component:03d}"
    for component in range(1, RNA_ICA_COMPONENTS + 1)
]

{
    "score_shape": rna_ica_scores_oriented.shape,
    "gene_loading_shape": (
        rna_ica_gene_loadings_oriented.shape
    ),
    "score_dtype": str(
        rna_ica_scores_oriented.dtype
    ),
    "gene_loading_dtype": str(
        rna_ica_gene_loadings_oriented.dtype
    ),
    "components_reoriented": int(
        np.sum(rna_ica_component_signs < 0)
    ),
}

{'score_shape': (9965, 200),
 'gene_loading_shape': (200, 5000),
 'score_dtype': 'float32',
 'gene_loading_dtype': 'float32',
 'components_reoriented': 91}

In [44]:
# =======================================================
# Summarize RNA-seq ICA component structure
# =======================================================

from scipy.stats import kurtosis, skew

rna_ica_absolute_loadings = np.abs(
    rna_ica_gene_loadings_oriented.astype(
        np.float64,
        copy=False,
    )
)

rna_ica_loading_power = np.square(
    rna_ica_absolute_loadings
)

rna_ica_effective_gene_count = (
    np.square(
        rna_ica_loading_power.sum(axis=1)
    )
    / np.square(
        rna_ica_loading_power
    ).sum(axis=1)
)

TOP_LOADING_COUNT = 25

rna_ica_top_loading_fraction = (
    np.partition(
        rna_ica_absolute_loadings,
        -TOP_LOADING_COUNT,
        axis=1,
    )[:, -TOP_LOADING_COUNT:].sum(axis=1)
    / rna_ica_absolute_loadings.sum(axis=1)
)

rna_ica_component_summary = pd.DataFrame(
    {
        "component": rna_ica_component_names,
        "score_skewness": skew(
            rna_ica_scores_oriented,
            axis=0,
            bias=False,
        ),
        "score_excess_kurtosis": kurtosis(
            rna_ica_scores_oriented,
            axis=0,
            fisher=True,
            bias=False,
        ),
        "effective_gene_count": (
            rna_ica_effective_gene_count
        ),
        "top_25_abs_loading_fraction": (
            rna_ica_top_loading_fraction
        ),
    }
)

rna_ica_component_summary[
    "absolute_excess_kurtosis"
] = np.abs(
    rna_ica_component_summary[
        "score_excess_kurtosis"
    ]
)

rna_ica_component_summary.sort_values(
    "absolute_excess_kurtosis",
    ascending=False,
).head(20)

,component,score_skewness,score_excess_kurtosis,effective_gene_count,top_25_abs_loading_fraction,absolute_excess_kurtosis
162,RNA_IC163,9.606014,338.353394,945.426926,0.029587,338.353394
120,RNA_IC121,6.249316,194.581741,559.255401,0.038085,194.581741
32,RNA_IC033,8.052588,161.047607,904.351023,0.030403,161.047607
76,RNA_IC077,7.085668,154.391510,1107.985358,0.026204,154.391510
54,RNA_IC055,7.485491,148.349304,1467.619604,0.022052,148.349304
114,RNA_IC115,5.756988,146.647003,850.642883,0.031115,146.647003
31,RNA_IC032,0.984452,132.920395,1196.078797,0.025570,132.920395
178,RNA_IC179,6.873561,129.079819,1005.667396,0.027992,129.079819
95,RNA_IC096,1.325132,125.134689,1365.392033,0.023070,125.134689
156,RNA_IC157,6.043588,109.149284,1546.409994,0.020641,109.149284


In [45]:
# =======================================================
# Characterize RNA-seq ICA sample concentration
# =======================================================

rna_ica_score_z = (
    rna_ica_scores_oriented
    - rna_ica_scores_oriented.mean(axis=0)
) / rna_ica_scores_oriented.std(axis=0)

rna_ica_score_power = np.square(
    rna_ica_score_z.astype(
        np.float64,
        copy=False,
    )
)

rna_ica_effective_sample_count = (
    np.square(
        rna_ica_score_power.sum(axis=0)
    )
    / np.square(
        rna_ica_score_power
    ).sum(axis=0)
)

top_sample_count = int(
    np.ceil(
        0.01 * len(sample_metadata)
    )
)

rna_ica_top_1_percent_power_fraction = (
    np.partition(
        rna_ica_score_power,
        -top_sample_count,
        axis=0,
    )[-top_sample_count:].sum(axis=0)
    / rna_ica_score_power.sum(axis=0)
)

project_score_power = np.vstack(
    [
        rna_ica_score_power[
            np.flatnonzero(
                project_labels == project_id
            )
        ].sum(axis=0)
        for project_id in project_ids
    ]
)

project_power_fraction = (
    project_score_power
    / project_score_power.sum(axis=0)
)

project_sample_fraction = (
    project_counts
    .reindex(project_ids)
    .to_numpy()
    / len(sample_metadata)
)

project_power_enrichment = (
    project_power_fraction
    / project_sample_fraction[:, None]
)

dominant_project_indices = np.argmax(
    project_power_fraction,
    axis=0,
)

enriched_project_indices = np.argmax(
    project_power_enrichment,
    axis=0,
)

rna_ica_component_diagnostics = (
    rna_ica_component_summary.copy()
)

rna_ica_component_diagnostics[
    "effective_sample_count"
] = rna_ica_effective_sample_count

rna_ica_component_diagnostics[
    "top_1_percent_score_power_fraction"
] = rna_ica_top_1_percent_power_fraction

rna_ica_component_diagnostics[
    "samples_abs_z_ge_3"
] = np.sum(
    np.abs(rna_ica_score_z) >= 3,
    axis=0,
)

rna_ica_component_diagnostics[
    "dominant_project"
] = project_ids[
    dominant_project_indices
]

rna_ica_component_diagnostics[
    "dominant_project_power_fraction"
] = project_power_fraction[
    dominant_project_indices,
    np.arange(RNA_ICA_COMPONENTS),
]

rna_ica_component_diagnostics[
    "most_enriched_project"
] = project_ids[
    enriched_project_indices
]

rna_ica_component_diagnostics[
    "maximum_project_power_enrichment"
] = project_power_enrichment[
    enriched_project_indices,
    np.arange(RNA_ICA_COMPONENTS),
]

rna_ica_component_diagnostics.sort_values(
    "absolute_excess_kurtosis",
    ascending=False,
).head(20)

,component,score_skewness,score_excess_kurtosis,effective_gene_count,top_25_abs_loading_fraction,absolute_excess_kurtosis,effective_sample_count,top_1_percent_score_power_fraction,samples_abs_z_ge_3,dominant_project,dominant_project_power_fraction,most_enriched_project,maximum_project_power_enrichment
162,RNA_IC163,9.606014,338.353394,945.426926,0.029587,338.353394,29.207199,0.866286,53,TCGA-KIRC,0.544401,TCGA-KICH,37.517525
120,RNA_IC121,6.249316,194.581741,559.255401,0.038085,194.581741,50.459912,0.640571,85,TCGA-PCPG,0.494504,TCGA-PCPG,27.529254
32,RNA_IC033,8.052588,161.047607,904.351023,0.030403,161.047607,60.774729,0.569108,106,TCGA-KIRP,0.244742,TCGA-KICH,36.546411
76,RNA_IC077,7.085668,154.391510,1107.985358,0.026204,154.391510,63.344869,0.540820,69,TCGA-THYM,0.491301,TCGA-THYM,40.798426
54,RNA_IC055,7.485491,148.349304,1467.619604,0.022052,148.349304,65.873724,0.464623,45,TCGA-SARC,0.436591,TCGA-SARC,16.797815
114,RNA_IC115,5.756988,146.647003,850.642883,0.031115,146.647003,66.623069,0.620663,63,TCGA-LAML,0.616575,TCGA-LAML,45.852024
31,RNA_IC032,0.984452,132.920395,1196.078797,0.025570,132.920395,73.351306,0.758540,61,TCGA-ACC,0.745880,TCGA-ACC,94.084749
178,RNA_IC179,6.873561,129.079819,1005.667396,0.027992,129.079819,75.484148,0.550002,102,TCGA-PAAD,0.461317,TCGA-PAAD,25.825965
95,RNA_IC096,1.325132,125.134689,1365.392033,0.023070,125.134689,77.808231,0.845978,84,TCGA-THYM,0.846399,TCGA-THYM,70.286372
156,RNA_IC157,6.043588,109.149284,1546.409994,0.020641,109.149284,88.898673,0.800776,117,TCGA-TGCT,0.837807,TCGA-TGCT,55.658313


In [46]:
# =======================================================
# Quantify RNA-seq ICA cross-project breadth
# =======================================================

project_sample_counts = (
    project_counts
    .reindex(project_ids)
    .to_numpy()
)

project_mean_score_power = (
    project_score_power
    / project_sample_counts[:, None]
)

equal_weight_project_fraction = (
    project_mean_score_power
    / project_mean_score_power.sum(axis=0)
)

rna_ica_effective_project_count = (
    1
    / np.square(
        equal_weight_project_fraction
    ).sum(axis=0)
)

rna_ica_component_diagnostics[
    "effective_project_count"
] = rna_ica_effective_project_count

rna_ica_component_diagnostics[
    "maximum_equal_weight_project_fraction"
] = equal_weight_project_fraction.max(axis=0)

rna_ica_component_diagnostics[
    [
        "absolute_excess_kurtosis",
        "effective_gene_count",
        "effective_sample_count",
        "top_1_percent_score_power_fraction",
        "dominant_project_power_fraction",
        "maximum_project_power_enrichment",
        "effective_project_count",
        "maximum_equal_weight_project_fraction",
    ]
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
    ]
).T

,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
absolute_excess_kurtosis,200.0,33.053131,41.168896,0.677311,2.207383,7.404198,21.196183,42.101101,82.359097,107.113966,338.353394
effective_gene_count,200.0,959.408902,381.310286,10.620506,340.848740,746.984652,1006.549894,1224.447102,1422.491924,1532.237301,1729.515468
effective_sample_count,200.0,713.943225,699.591416,29.207199,116.804674,221.054959,412.151744,958.193200,1914.261349,2099.308259,4312.751198
top_1_percent_score_power_fraction,200.0,0.382298,0.180041,0.056750,0.150250,0.240048,0.365527,0.505439,0.620670,0.693581,0.866286
dominant_project_power_fraction,200.0,0.421604,0.246327,0.085651,0.123881,0.198481,0.386701,0.621791,0.791292,0.845513,0.904997
maximum_project_power_enrichment,200.0,13.340270,13.881756,1.627962,2.637005,4.957536,8.759164,16.485607,26.927634,41.051105,94.084749
effective_project_count,200.0,10.491623,8.709829,1.153906,1.857074,2.948867,7.348843,16.143887,25.394650,26.805819,29.663282
maximum_equal_weight_project_fraction,200.0,0.366801,0.242624,0.049334,0.084269,0.151551,0.311989,0.566943,0.722513,0.801120,0.930837


In [47]:
# =======================================================
# Screen broadly distributed RNA-seq ICA candidates
# =======================================================

MIN_EFFECTIVE_PROJECT_COUNT = 16.0
MAX_EQUAL_WEIGHT_PROJECT_FRACTION = 0.20
MAX_TOP_1_PERCENT_POWER_FRACTION = 0.25

rna_ica_component_diagnostics[
    "broad_project_distribution"
] = (
    rna_ica_component_diagnostics[
        "effective_project_count"
    ]
    >= MIN_EFFECTIVE_PROJECT_COUNT
)

rna_ica_component_diagnostics[
    "no_single_project_dominance"
] = (
    rna_ica_component_diagnostics[
        "maximum_equal_weight_project_fraction"
    ]
    <= MAX_EQUAL_WEIGHT_PROJECT_FRACTION
)

rna_ica_component_diagnostics[
    "sample_distributed"
] = (
    rna_ica_component_diagnostics[
        "top_1_percent_score_power_fraction"
    ]
    <= MAX_TOP_1_PERCENT_POWER_FRACTION
)

rna_ica_component_diagnostics[
    "candidate_cross_project_distribution"
] = (
    rna_ica_component_diagnostics[
        [
            "broad_project_distribution",
            "no_single_project_dominance",
            "sample_distributed",
        ]
    ].all(axis=1)
)

rna_ica_candidate_components = (
    rna_ica_component_diagnostics.loc[
        rna_ica_component_diagnostics[
            "candidate_cross_project_distribution"
        ]
    ]
    .sort_values(
        [
            "absolute_excess_kurtosis",
            "effective_project_count",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(
    pd.Series(
        {
            "total_components": RNA_ICA_COMPONENTS,
            "broad_project_distribution": int(
                rna_ica_component_diagnostics[
                    "broad_project_distribution"
                ].sum()
            ),
            "no_single_project_dominance": int(
                rna_ica_component_diagnostics[
                    "no_single_project_dominance"
                ].sum()
            ),
            "sample_distributed": int(
                rna_ica_component_diagnostics[
                    "sample_distributed"
                ].sum()
            ),
            "candidate_components": len(
                rna_ica_candidate_components
            ),
        }
    )
)

rna_ica_candidate_components[
    [
        "component",
        "score_excess_kurtosis",
        "effective_sample_count",
        "top_1_percent_score_power_fraction",
        "effective_project_count",
        "maximum_equal_weight_project_fraction",
    ]
].head(20)

total_components               200
broad_project_distribution      52
no_single_project_dominance     66
sample_distributed              56
candidate_components            42
dtype: int64

,component,score_excess_kurtosis,effective_sample_count,top_1_percent_score_power_fraction,effective_project_count,maximum_equal_weight_project_fraction
0,RNA_IC097,8.578486,861.012474,0.249775,22.027721,0.090218
1,RNA_IC131,8.085031,899.337974,0.237930,17.960752,0.149336
2,RNA_IC105,6.122233,1092.826076,0.211893,25.413385,0.086636
3,RNA_IC184,5.789473,1134.195140,0.222714,17.323364,0.180446
4,RNA_IC018,5.410789,1185.255316,0.198319,22.612952,0.125885
5,RNA_IC137,5.371033,1190.883699,0.207089,18.609789,0.158284
6,RNA_IC084,4.804665,1277.293663,0.210551,16.655774,0.187997
7,RNA_IC035,4.423038,1342.952440,0.195371,16.046029,0.145862
8,RNA_IC102,4.073235,1409.359244,0.192886,18.126292,0.147777
9,RNA_IC099,4.063226,1411.356101,0.181567,19.599111,0.123230


In [48]:
# =======================================================
# Extract candidate RNA-seq ICA loading tails
# =======================================================

TOP_GENES_PER_DIRECTION = 50

component_to_index = {
    component: component_index
    for component_index, component in enumerate(
        rna_ica_component_names
    )
}

candidate_loading_tables = []

for component in rna_ica_candidate_components["component"]:
    component_index = component_to_index[component]

    component_loadings = (
        rna_ica_gene_loadings_oriented[component_index]
    )

    direction_indices = {
        "positive": np.argsort(
            component_loadings
        )[-TOP_GENES_PER_DIRECTION:][::-1],
        "negative": np.argsort(
            component_loadings
        )[:TOP_GENES_PER_DIRECTION],
    }

    for direction, feature_indices in direction_indices.items():
        loading_table = (
            rna_ica_features
            .iloc[feature_indices]
            .copy()
        )

        loading_table.insert(
            0,
            "component",
            component,
        )

        loading_table.insert(
            1,
            "direction",
            direction,
        )

        loading_table.insert(
            2,
            "direction_rank",
            np.arange(
                1,
                TOP_GENES_PER_DIRECTION + 1,
            ),
        )

        loading_table.insert(
            3,
            "gene_loading",
            component_loadings[feature_indices],
        )

        candidate_loading_tables.append(
            loading_table
        )

rna_ica_candidate_gene_loadings = pd.concat(
    candidate_loading_tables,
    ignore_index=True,
)

first_candidate = (
    rna_ica_candidate_components
    .loc[0, "component"]
)

display(
    {
        "candidate_components": len(
            rna_ica_candidate_components
        ),
        "genes_per_component": (
            2 * TOP_GENES_PER_DIRECTION
        ),
        "loading_table_rows": len(
            rna_ica_candidate_gene_loadings
        ),
        "first_candidate": first_candidate,
    }
)

rna_ica_candidate_gene_loadings.loc[
    rna_ica_candidate_gene_loadings[
        "component"
    ].eq(first_candidate)
].head(20)

{'candidate_components': 42,
 'genes_per_component': 100,
 'loading_table_rows': 4200,
 'first_candidate': 'RNA_IC097'}

,component,direction,direction_rank,gene_loading,matrix_row_index,gene_id,gene_name,gene_type,gene_id_base,gene_id_is_versioned,expressed_sample_count,filtered_matrix_row_index,mean_logcpm,variance_logcpm,median_within_project_variance
0,RNA_IC097,positive,1,0.474025,10305,ENSG00000159184.8,HOXB13,protein_coding,ENSG00000159184,True,4263,9993,-0.387100,16.938844,6.920620
1,RNA_IC097,positive,2,0.434362,14928,ENSG00000180818.5,HOXC10,protein_coding,ENSG00000180818,True,4795,14009,0.186970,14.606491,6.122840
2,RNA_IC097,positive,3,0.327564,6773,ENSG00000133636.11,NTS,protein_coding,ENSG00000133636,True,2463,6584,-1.367699,11.680587,6.618501
3,RNA_IC097,positive,4,0.310924,10317,ENSG00000159217.10,IGF2BP1,protein_coding,ENSG00000159217,True,3279,10003,-0.748860,10.956062,6.616459
4,RNA_IC097,positive,5,0.308954,5377,ENSG00000123388.4,HOXC11,protein_coding,ENSG00000123388,True,4194,5272,-0.952940,9.847234,5.110409
5,RNA_IC097,positive,6,0.293747,16041,ENSG00000185686.18,PRAME,protein_coding,ENSG00000185686,True,4906,14835,0.853685,19.613843,10.312172
6,RNA_IC097,positive,7,0.287127,15410,ENSG00000183145.9,RIPPLY3,protein_coding,ENSG00000183145,True,4175,14349,-0.515741,5.615489,3.221967
7,RNA_IC097,positive,8,0.279891,6068,ENSG00000128714.6,HOXD13,protein_coding,ENSG00000128714,True,2835,5917,-1.734334,9.301877,4.639985
8,RNA_IC097,positive,9,0.279147,6477,ENSG00000131668.14,BARX1,protein_coding,ENSG00000131668,True,2745,6301,-1.223925,8.930988,4.476027
9,RNA_IC097,positive,10,0.276694,14293,ENSG00000177459.11,ERICH5,protein_coding,ENSG00000177459,True,5112,13547,0.130784,8.264298,3.705464


In [49]:
# =======================================================
# Inspect shared methylation values for modeling
# =======================================================

METHYLATION_CHUNK_SIZE = 512

n_methylation_probes, n_methylation_samples = (
    methylation_betas.shape
)

methylation_probe_missing_counts = np.zeros(
    n_methylation_probes,
    dtype=np.int32,
)

methylation_beta_minimum = np.inf
methylation_beta_maximum = -np.inf
methylation_values_at_or_below_zero = 0
methylation_values_at_or_above_one = 0

for probe_start in range(
    0,
    n_methylation_probes,
    METHYLATION_CHUNK_SIZE,
):
    probe_stop = min(
        probe_start + METHYLATION_CHUNK_SIZE,
        n_methylation_probes,
    )

    beta_chunk = np.asarray(
        methylation_betas[probe_start:probe_stop],
        dtype=np.float32,
    )

    finite_chunk = np.isfinite(beta_chunk)

    methylation_probe_missing_counts[
        probe_start:probe_stop
    ] = np.sum(
        ~finite_chunk,
        axis=1,
    )

    methylation_beta_minimum = min(
        methylation_beta_minimum,
        float(np.nanmin(beta_chunk)),
    )

    methylation_beta_maximum = max(
        methylation_beta_maximum,
        float(np.nanmax(beta_chunk)),
    )

    methylation_values_at_or_below_zero += int(
        np.count_nonzero(
            finite_chunk & (beta_chunk <= 0)
        )
    )

    methylation_values_at_or_above_one += int(
        np.count_nonzero(
            finite_chunk & (beta_chunk >= 1)
        )
    )

methylation_missing_values = int(
    methylation_probe_missing_counts.sum()
)

pd.Series(
    {
        "matrix_shape": methylation_betas.shape,
        "matrix_dtype": str(methylation_betas.dtype),
        "missing_values": methylation_missing_values,
        "missing_fraction": (
            methylation_missing_values
            / methylation_betas.size
        ),
        "probes_with_missing_values": int(
            np.count_nonzero(
                methylation_probe_missing_counts
            )
        ),
        "maximum_probe_missing_fraction": float(
            methylation_probe_missing_counts.max()
            / n_methylation_samples
        ),
        "beta_minimum": methylation_beta_minimum,
        "beta_maximum": methylation_beta_maximum,
        "values_at_or_below_zero": (
            methylation_values_at_or_below_zero
        ),
        "values_at_or_above_one": (
            methylation_values_at_or_above_one
        ),
    }
)

matrix_shape                      (23356, 9965)
matrix_dtype                            float32
missing_values                           752180
missing_fraction                       0.003232
probes_with_missing_values                17285
maximum_probe_missing_fraction         0.193778
beta_minimum                           0.001425
beta_maximum                              0.998
values_at_or_below_zero                       0
values_at_or_above_one                        0
dtype: object

In [50]:
# =======================================================
# Summarize methylation missingness structure
# =======================================================

methylation_sample_missing_counts = np.zeros(
    n_methylation_samples,
    dtype=np.int32,
)

for probe_start in range(
    0,
    n_methylation_probes,
    METHYLATION_CHUNK_SIZE,
):
    probe_stop = min(
        probe_start + METHYLATION_CHUNK_SIZE,
        n_methylation_probes,
    )

    beta_chunk = np.asarray(
        methylation_betas[probe_start:probe_stop],
        dtype=np.float32,
    )

    methylation_sample_missing_counts += np.sum(
        ~np.isfinite(beta_chunk),
        axis=0,
    )

methylation_probe_missing_fraction = (
    methylation_probe_missing_counts
    / n_methylation_samples
)

methylation_sample_missing_fraction = (
    methylation_sample_missing_counts
    / n_methylation_probes
)

probe_missingness_summary = pd.Series(
    methylation_probe_missing_fraction
).describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

sample_missingness_by_platform = (
    sample_metadata[
        ["methylation_platform"]
    ]
    .assign(
        missing_fraction=(
            methylation_sample_missing_fraction
        )
    )
    .groupby(
        "methylation_platform",
        observed=True,
    )["missing_fraction"]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
        ]
    )
)

display(probe_missingness_summary)
display(sample_missingness_by_platform)

count    23356.000000
mean         0.003232
std          0.012442
min          0.000000
50%          0.000201
75%          0.000602
90%          0.006222
95%          0.017687
99%          0.063884
max          0.193778
dtype: float64

,count,mean,std,min,50%,90%,95%,99%,max
methylation_platform,,,,,,,,,
Illumina Human Methylation 27,1620.0,0.006631,0.013300,0.0,0.002141,0.017216,0.028367,0.065939,0.214848
Illumina Human Methylation 450,8345.0,0.002572,0.005825,0.0,0.000514,0.007107,0.012716,0.029349,0.110036


In [51]:
# =======================================================
# Quantify probe missingness by methylation platform
# =======================================================

methylation_platform_labels = (
    sample_metadata["methylation_platform"].to_numpy()
)

methylation_platforms = np.sort(
    sample_metadata["methylation_platform"].unique()
)

methylation_probe_platform_missingness = {}

for platform in methylation_platforms:
    platform_sample_indices = np.flatnonzero(
        methylation_platform_labels == platform
    )

    platform_missing_counts = np.zeros(
        n_methylation_probes,
        dtype=np.int32,
    )

    for probe_start in range(
        0,
        n_methylation_probes,
        METHYLATION_CHUNK_SIZE,
    ):
        probe_stop = min(
            probe_start + METHYLATION_CHUNK_SIZE,
            n_methylation_probes,
        )

        beta_chunk = np.asarray(
            methylation_betas[
                probe_start:probe_stop,
                platform_sample_indices,
            ],
            dtype=np.float32,
        )

        platform_missing_counts[
            probe_start:probe_stop
        ] = np.sum(
            ~np.isfinite(beta_chunk),
            axis=1,
        )

    methylation_probe_platform_missingness[
        platform
    ] = (
        platform_missing_counts
        / len(platform_sample_indices)
    )

methylation_probe_platform_missingness = (
    pd.DataFrame(
        methylation_probe_platform_missingness
    )
)

probe_platform_missingness_summary = (
    methylation_probe_platform_missingness
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)

probe_missingness_threshold_counts = pd.DataFrame(
    {
        f"probes_above_{int(threshold * 100)}_percent": (
            methylation_probe_platform_missingness
            .gt(threshold)
            .sum(axis=0)
        )
        for threshold in [0.01, 0.05, 0.10]
    }
)

probe_missingness_threshold_counts.loc[
    "any_platform"
] = {
    f"probes_above_{int(threshold * 100)}_percent": int(
        methylation_probe_platform_missingness
        .max(axis=1)
        .gt(threshold)
        .sum()
    )
    for threshold in [0.01, 0.05, 0.10]
}

display(probe_platform_missingness_summary)
probe_missingness_threshold_counts

,count,mean,std,min,50%,75%,90%,95%,99%,max
Illumina Human Methylation 27,23356.0,0.006631,0.023837,0.0,0.00000,0.000617,0.011728,0.040741,0.140741,0.200000
Illumina Human Methylation 450,23356.0,0.002572,0.012788,0.0,0.00012,0.000359,0.002397,0.010306,0.065003,0.197963


,probes_above_1_percent,probes_above_5_percent,probes_above_10_percent
Illumina Human Methylation 27,2446,1010,489
Illumina Human Methylation 450,1189,317,118
any_platform,3024,1200,578


In [52]:
# =======================================================
# Select methylation probes for modeling
# =======================================================

MAX_PLATFORM_PROBE_MISSING_FRACTION = 0.05

methylation_model_probe_mask = (
    methylation_probe_platform_missingness
    .max(axis=1)
    .le(MAX_PLATFORM_PROBE_MISSING_FRACTION)
    .to_numpy()
)

methylation_model_probe_indices = np.flatnonzero(
    methylation_model_probe_mask
)

methylation_model_probe_mapping = (
    shared_probe_mapping
    .iloc[methylation_model_probe_indices]
    .copy()
    .reset_index(drop=True)
)

methylation_model_probe_mapping.insert(
    0,
    "model_matrix_row_index",
    np.arange(
        len(methylation_model_probe_mapping),
        dtype=np.int32,
    ),
)

methylation_model_probe_mapping[
    "shared_matrix_row_index"
] = methylation_model_probe_indices

{
    "input_probes": n_methylation_probes,
    "retained_probes": int(
        methylation_model_probe_mask.sum()
    ),
    "excluded_probes": int(
        (~methylation_model_probe_mask).sum()
    ),
    "retained_fraction": float(
        methylation_model_probe_mask.mean()
    ),
    "maximum_retained_hm27_missing_fraction": float(
        methylation_probe_platform_missingness.loc[
            methylation_model_probe_mask,
            "Illumina Human Methylation 27",
        ].max()
    ),
    "maximum_retained_hm450_missing_fraction": float(
        methylation_probe_platform_missingness.loc[
            methylation_model_probe_mask,
            "Illumina Human Methylation 450",
        ].max()
    ),
}

{'input_probes': 23356,
 'retained_probes': 22156,
 'excluded_probes': 1200,
 'retained_fraction': 0.948621339270423,
 'maximum_retained_hm27_missing_fraction': 0.05,
 'maximum_retained_hm450_missing_fraction': 0.049850209706411026}

In [53]:
# =======================================================
# Build imputed methylation M-value matrix
# =======================================================

METHYLATION_M_VALUES_PATH = (
    Paths.methylation
    / "tcga_primary_tumor_shared_methylation_program_discovery_m_values.npy"
)

methylation_m_values_write = np.lib.format.open_memmap(
    METHYLATION_M_VALUES_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(
        len(methylation_model_probe_indices),
        n_methylation_samples,
    ),
)

methylation_imputed_values = 0
methylation_m_value_minimum = np.inf
methylation_m_value_maximum = -np.inf

for model_start in range(
    0,
    len(methylation_model_probe_indices),
    METHYLATION_CHUNK_SIZE,
):
    model_stop = min(
        model_start + METHYLATION_CHUNK_SIZE,
        len(methylation_model_probe_indices),
    )

    source_probe_indices = (
        methylation_model_probe_indices[
            model_start:model_stop
        ]
    )

    beta_chunk = np.asarray(
        methylation_betas[source_probe_indices],
        dtype=np.float32,
    )

    for platform in methylation_platforms:
        platform_sample_indices = np.flatnonzero(
            methylation_platform_labels == platform
        )

        platform_beta_values = beta_chunk[
            :,
            platform_sample_indices,
        ]

        missing_mask = ~np.isfinite(
            platform_beta_values
        )

        probe_platform_medians = np.nanmedian(
            platform_beta_values,
            axis=1,
        )

        missing_rows, missing_columns = np.where(
            missing_mask
        )

        platform_beta_values[
            missing_rows,
            missing_columns,
        ] = probe_platform_medians[missing_rows]

        beta_chunk[
            :,
            platform_sample_indices,
        ] = platform_beta_values

        methylation_imputed_values += int(
            missing_mask.sum()
        )

    m_value_chunk = np.log2(
        beta_chunk
        / (1.0 - beta_chunk)
    ).astype(np.float32)

    methylation_m_values_write[
        model_start:model_stop
    ] = m_value_chunk

    methylation_m_value_minimum = min(
        methylation_m_value_minimum,
        float(m_value_chunk.min()),
    )

    methylation_m_value_maximum = max(
        methylation_m_value_maximum,
        float(m_value_chunk.max()),
    )

methylation_m_values_write.flush()
del methylation_m_values_write

methylation_m_values = np.load(
    METHYLATION_M_VALUES_PATH,
    mmap_mode="r",
)

{
    "matrix_shape": methylation_m_values.shape,
    "matrix_dtype": str(methylation_m_values.dtype),
    "imputed_values": methylation_imputed_values,
    "imputed_fraction": (
        methylation_imputed_values
        / methylation_m_values.size
    ),
    "m_value_minimum": methylation_m_value_minimum,
    "m_value_maximum": methylation_m_value_maximum,
}

{'matrix_shape': (22156, 9965),
 'matrix_dtype': 'float32',
 'imputed_values': 264814,
 'imputed_fraction': 0.0011994227494370757,
 'm_value_minimum': -9.452821731567383,
 'm_value_maximum': 8.962828636169434}

In [54]:
# =======================================================
# Summarize methylation platform overlap by project
# =======================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM450_PLATFORM = "Illumina Human Methylation 450"

methylation_project_platform_counts = (
    sample_metadata
    .groupby(
        ["project_id", "methylation_platform"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=[
            HM27_PLATFORM,
            HM450_PLATFORM,
        ],
        fill_value=0,
    )
)

methylation_cross_platform_projects = (
    methylation_project_platform_counts.loc[
        (
            methylation_project_platform_counts[
                HM27_PLATFORM
            ]
            > 0
        )
        & (
            methylation_project_platform_counts[
                HM450_PLATFORM
            ]
            > 0
        )
    ]
    .copy()
)

methylation_cross_platform_projects[
    "total_samples"
] = (
    methylation_cross_platform_projects.sum(axis=1)
)

display(
    pd.Series(
        {
            "total_projects": len(
                methylation_project_platform_counts
            ),
            "projects_with_hm27": int(
                (
                    methylation_project_platform_counts[
                        HM27_PLATFORM
                    ]
                    > 0
                ).sum()
            ),
            "projects_with_hm450": int(
                (
                    methylation_project_platform_counts[
                        HM450_PLATFORM
                    ]
                    > 0
                ).sum()
            ),
            "projects_with_both_platforms": len(
                methylation_cross_platform_projects
            ),
        }
    )
)

methylation_cross_platform_projects.sort_values(
    "total_samples",
    ascending=False,
)

total_projects                  33
projects_with_hm27              11
projects_with_hm450             33
projects_with_both_platforms    11
dtype: int64

methylation_platform,Illumina Human Methylation 27,Illumina Human Methylation 450,total_samples
project_id,,,
TCGA-BRCA,312,777,1089
TCGA-UCEC,115,428,543
TCGA-LUAD,59,453,512
TCGA-LUSC,130,370,500
TCGA-KIRC,169,318,487
TCGA-COAD,160,293,453
TCGA-OV,413,9,422
TCGA-STAD,39,373,412
TCGA-KIRP,6,274,280


In [55]:
# =======================================================
# Quantify within-project methylation platform effects
# =======================================================

MIN_PLATFORM_SAMPLES_PER_PROJECT = 20

platform_diagnostic_projects = (
    methylation_cross_platform_projects.loc[
        (
            methylation_cross_platform_projects[
                HM27_PLATFORM
            ]
            >= MIN_PLATFORM_SAMPLES_PER_PROJECT
        )
        & (
            methylation_cross_platform_projects[
                HM450_PLATFORM
            ]
            >= MIN_PLATFORM_SAMPLES_PER_PROJECT
        )
    ]
    .index
    .to_numpy()
)

methylation_platform_standardized_differences = np.empty(
    (
        len(platform_diagnostic_projects),
        len(methylation_model_probe_indices),
    ),
    dtype=np.float32,
)

for project_index, project_id in enumerate(
    platform_diagnostic_projects
):
    hm27_sample_indices = np.flatnonzero(
        (project_labels == project_id)
        & (
            methylation_platform_labels
            == HM27_PLATFORM
        )
    )

    hm450_sample_indices = np.flatnonzero(
        (project_labels == project_id)
        & (
            methylation_platform_labels
            == HM450_PLATFORM
        )
    )

    for probe_start in range(
        0,
        len(methylation_model_probe_indices),
        METHYLATION_CHUNK_SIZE,
    ):
        probe_stop = min(
            probe_start + METHYLATION_CHUNK_SIZE,
            len(methylation_model_probe_indices),
        )

        hm27_values = np.asarray(
            methylation_m_values[
                probe_start:probe_stop,
                hm27_sample_indices,
            ],
            dtype=np.float64,
        )

        hm450_values = np.asarray(
            methylation_m_values[
                probe_start:probe_stop,
                hm450_sample_indices,
            ],
            dtype=np.float64,
        )

        hm27_means = hm27_values.mean(axis=1)
        hm450_means = hm450_values.mean(axis=1)

        hm27_variances = hm27_values.var(
            axis=1,
            ddof=1,
        )

        hm450_variances = hm450_values.var(
            axis=1,
            ddof=1,
        )

        pooled_standard_deviation = np.sqrt(
            (
                (len(hm27_sample_indices) - 1)
                * hm27_variances
                + (len(hm450_sample_indices) - 1)
                * hm450_variances
            )
            / (
                len(hm27_sample_indices)
                + len(hm450_sample_indices)
                - 2
            )
        )

        standardized_difference = np.divide(
            hm450_means - hm27_means,
            pooled_standard_deviation,
            out=np.zeros_like(hm450_means),
            where=pooled_standard_deviation > 0,
        )

        methylation_platform_standardized_differences[
            project_index,
            probe_start:probe_stop,
        ] = standardized_difference.astype(
            np.float32
        )

methylation_median_absolute_platform_difference = (
    np.median(
        np.abs(
            methylation_platform_standardized_differences
        ),
        axis=0,
    )
)

methylation_platform_direction_consistency = np.maximum(
    np.mean(
        methylation_platform_standardized_differences
        > 0,
        axis=0,
    ),
    np.mean(
        methylation_platform_standardized_differences
        < 0,
        axis=0,
    ),
)

display(
    pd.Series(
        {
            "cross_platform_projects": len(
                methylation_cross_platform_projects
            ),
            "diagnostic_projects": len(
                platform_diagnostic_projects
            ),
            "excluded_from_diagnostic": (
                len(methylation_cross_platform_projects)
                - len(platform_diagnostic_projects)
            ),
        }
    )
)

display(
    pd.Series(
        methylation_median_absolute_platform_difference
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

pd.Series(
    {
        "probes_median_abs_difference_ge_0.10": int(
            np.sum(
                methylation_median_absolute_platform_difference
                >= 0.10
            )
        ),
        "probes_median_abs_difference_ge_0.20": int(
            np.sum(
                methylation_median_absolute_platform_difference
                >= 0.20
            )
        ),
        "probes_median_abs_difference_ge_0.50": int(
            np.sum(
                methylation_median_absolute_platform_difference
                >= 0.50
            )
        ),
        "probes_direction_consistency_ge_0.80": int(
            np.sum(
                methylation_platform_direction_consistency
                >= 0.80
            )
        ),
    }
)

cross_platform_projects     11
diagnostic_projects          9
excluded_from_diagnostic     2
dtype: int64

count    22156.000000
mean         1.896697
std          1.865417
min          0.026911
50%          1.151419
75%          2.881347
90%          4.720225
95%          5.710890
99%          7.660015
max         17.281050
dtype: float64

probes_median_abs_difference_ge_0.10    21857
probes_median_abs_difference_ge_0.20    20456
probes_median_abs_difference_ge_0.50    16099
probes_direction_consistency_ge_0.80    18313
dtype: int64

In [56]:
# =======================================================
# Define primary HM450 methylation cohort
# =======================================================

methylation_hm450_sample_indices = np.flatnonzero(
    methylation_platform_labels == HM450_PLATFORM
)

methylation_hm450_sample_metadata = (
    sample_metadata
    .iloc[methylation_hm450_sample_indices]
    .copy()
    .reset_index(drop=True)
)

methylation_hm450_project_labels = (
    methylation_hm450_sample_metadata[
        "project_id"
    ].to_numpy()
)

methylation_hm450_project_ids = np.sort(
    methylation_hm450_sample_metadata[
        "project_id"
    ].unique()
)

{
    "hm450_samples": len(
        methylation_hm450_sample_indices
    ),
    "hm450_projects": len(
        methylation_hm450_project_ids
    ),
    "modeling_probes": len(
        methylation_model_probe_indices
    ),
    "primary_matrix_shape": (
        len(methylation_hm450_sample_indices),
        len(methylation_model_probe_indices),
    ),
}

{'hm450_samples': 8345,
 'hm450_projects': 33,
 'modeling_probes': 22156,
 'primary_matrix_shape': (8345, 22156)}

In [57]:
# =======================================================
# Rank HM450 probes by within-project variability
# =======================================================

methylation_hm450_project_variances = np.empty(
    (
        len(methylation_hm450_project_ids),
        len(methylation_model_probe_indices),
    ),
    dtype=np.float32,
)

for project_index, project_id in enumerate(
    methylation_hm450_project_ids
):
    project_local_sample_indices = np.flatnonzero(
        methylation_hm450_project_labels == project_id
    )

    project_global_sample_indices = (
        methylation_hm450_sample_indices[
            project_local_sample_indices
        ]
    )

    for probe_start in range(
        0,
        len(methylation_model_probe_indices),
        METHYLATION_CHUNK_SIZE,
    ):
        probe_stop = min(
            probe_start + METHYLATION_CHUNK_SIZE,
            len(methylation_model_probe_indices),
        )

        project_m_values = np.asarray(
            methylation_m_values[
                probe_start:probe_stop,
                project_global_sample_indices,
            ],
            dtype=np.float64,
        )

        methylation_hm450_project_variances[
            project_index,
            probe_start:probe_stop,
        ] = np.var(
            project_m_values,
            axis=1,
            ddof=1,
        ).astype(np.float32)

methylation_hm450_median_within_project_variance = (
    np.median(
        methylation_hm450_project_variances,
        axis=0,
    )
)

pd.Series(
    methylation_hm450_median_within_project_variance
).describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

count    22156.000000
mean         0.647409
std          0.714811
min          0.027069
25%          0.098661
50%          0.352709
75%          1.009699
90%          1.595558
95%          1.998127
99%          3.148979
max          9.985904
dtype: float64

In [58]:
# =======================================================
# Select variable HM450 methylation probes
# =======================================================

N_METHYLATION_VARIABLE_PROBES = 10_000

methylation_variable_probe_indices = np.sort(
    np.argsort(
        methylation_hm450_median_within_project_variance
    )[-N_METHYLATION_VARIABLE_PROBES:]
)

methylation_variable_probe_mapping = (
    methylation_model_probe_mapping
    .iloc[methylation_variable_probe_indices]
    .copy()
    .reset_index(drop=True)
)

methylation_variable_probe_mapping[
    "median_within_project_variance"
] = (
    methylation_hm450_median_within_project_variance[
        methylation_variable_probe_indices
    ]
)

{
    "selected_probes": len(
        methylation_variable_probe_indices
    ),
    "selected_fraction": (
        len(methylation_variable_probe_indices)
        / len(methylation_model_probe_indices)
    ),
    "minimum_selected_median_variance": float(
        methylation_variable_probe_mapping[
            "median_within_project_variance"
        ].min()
    ),
}

{'selected_probes': 10000,
 'selected_fraction': 0.45134500812421013,
 'minimum_selected_median_variance': 0.48317381739616394}

In [59]:
# =======================================================
# Prepare variable HM450 methylation matrix
# =======================================================

methylation_hm450_variable_matrix = np.empty(
    (
        len(methylation_hm450_sample_indices),
        len(methylation_variable_probe_indices),
    ),
    dtype=np.float32,
)

for variable_start in range(
    0,
    len(methylation_variable_probe_indices),
    METHYLATION_CHUNK_SIZE,
):
    variable_stop = min(
        variable_start + METHYLATION_CHUNK_SIZE,
        len(methylation_variable_probe_indices),
    )

    source_probe_indices = (
        methylation_variable_probe_indices[
            variable_start:variable_stop
        ]
    )

    methylation_hm450_variable_matrix[
        :,
        variable_start:variable_stop,
    ] = np.asarray(
        methylation_m_values[
            np.ix_(
                source_probe_indices,
                methylation_hm450_sample_indices,
            )
        ].T,
        dtype=np.float32,
    )

{
    "matrix_shape": methylation_hm450_variable_matrix.shape,
    "matrix_dtype": str(
        methylation_hm450_variable_matrix.dtype
    ),
    "memory_gib": (
        methylation_hm450_variable_matrix.nbytes
        / 1024**3
    ),
}

{'matrix_shape': (8345, 10000),
 'matrix_dtype': 'float32',
 'memory_gib': 0.31087547540664673}

In [60]:
# =======================================================
# Fit diagnostic HM450 methylation PCA
# =======================================================

METHYLATION_PCA_COMPONENTS = 100

methylation_hm450_pca = PCA(
    n_components=METHYLATION_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=RANDOM_STATE,
)

methylation_hm450_pca_scores = (
    methylation_hm450_pca.fit_transform(
        methylation_hm450_variable_matrix
    )
)

methylation_hm450_pca_cumulative_variance = np.cumsum(
    methylation_hm450_pca.explained_variance_ratio_
)

{
    "score_shape": methylation_hm450_pca_scores.shape,
    "loading_shape": (
        methylation_hm450_pca.components_.shape
    ),
    "explained_variance_10_pcs": float(
        methylation_hm450_pca_cumulative_variance[9]
    ),
    "explained_variance_20_pcs": float(
        methylation_hm450_pca_cumulative_variance[19]
    ),
    "explained_variance_50_pcs": float(
        methylation_hm450_pca_cumulative_variance[49]
    ),
    "explained_variance_100_pcs": float(
        methylation_hm450_pca_cumulative_variance[99]
    ),
}

{'score_shape': (8345, 100),
 'loading_shape': (100, 10000),
 'explained_variance_10_pcs': 0.5042604207992554,
 'explained_variance_20_pcs': 0.5824429988861084,
 'explained_variance_50_pcs': 0.6528515815734863,
 'explained_variance_100_pcs': 0.6909075975418091}

In [61]:
# =======================================================
# Quantify project-associated HM450 PCA structure
# =======================================================

methylation_hm450_pca_score_columns = [
    f"PC{component}"
    for component in range(
        1,
        METHYLATION_PCA_COMPONENTS + 1,
    )
]

methylation_hm450_pca_score_metadata = pd.concat(
    [
        methylation_hm450_sample_metadata,
        pd.DataFrame(
            methylation_hm450_pca_scores,
            columns=methylation_hm450_pca_score_columns,
        ),
    ],
    axis=1,
)

methylation_hm450_project_counts = (
    methylation_hm450_pca_score_metadata
    .groupby("project_id", observed=True)
    .size()
)

methylation_hm450_project_pc_means = (
    methylation_hm450_pca_score_metadata
    .groupby(
        "project_id",
        observed=True,
    )[methylation_hm450_pca_score_columns]
    .mean()
    .reindex(methylation_hm450_project_ids)
)

methylation_hm450_pca_scores_float64 = (
    methylation_hm450_pca_scores.astype(
        np.float64,
        copy=False,
    )
)

methylation_hm450_pc_grand_means = (
    methylation_hm450_pca_scores_float64.mean(
        axis=0
    )
)

methylation_hm450_pc_total_sum_squares = np.sum(
    (
        methylation_hm450_pca_scores_float64
        - methylation_hm450_pc_grand_means
    ) ** 2,
    axis=0,
)

methylation_hm450_pc_between_project_sum_squares = (
    np.sum(
        methylation_hm450_project_counts
        .reindex(methylation_hm450_project_ids)
        .to_numpy()[:, None]
        * (
            methylation_hm450_project_pc_means.to_numpy()
            - methylation_hm450_pc_grand_means
        ) ** 2,
        axis=0,
    )
)

methylation_hm450_pca_project_association = pd.DataFrame(
    {
        "principal_component": np.arange(
            1,
            METHYLATION_PCA_COMPONENTS + 1,
        ),
        "project_eta_squared": (
            methylation_hm450_pc_between_project_sum_squares
            / methylation_hm450_pc_total_sum_squares
        ),
        "explained_variance_percent": (
            methylation_hm450_pca
            .explained_variance_ratio_
            * 100
        ),
    }
)

methylation_hm450_pca_project_association.head(20)

,principal_component,project_eta_squared,explained_variance_percent
0,1,0.766697,14.537102
1,2,0.412329,10.480583
2,3,0.518265,5.926501
3,4,0.620134,4.980820
4,5,0.582289,4.288681
5,6,0.778234,2.868407
6,7,0.752069,2.639043
7,8,0.657258,1.683479
8,9,0.714284,1.604252
9,10,0.747141,1.417170


In [62]:
# =======================================================
# Summarize project-associated HM450 variance
# =======================================================

methylation_hm450_project_variance_summary = pd.DataFrame(
    [
        {
            "n_components": n_components,
            "captured_variance_percent": (
                methylation_hm450_pca
                .explained_variance_ratio_[:n_components]
                .sum()
                * 100
            ),
            "project_associated_variance_percent": (
                np.sum(
                    methylation_hm450_pca
                    .explained_variance_ratio_[:n_components]
                    * methylation_hm450_pca_project_association[
                        "project_eta_squared"
                    ].to_numpy()[:n_components]
                )
                * 100
            ),
        }
        for n_components in [10, 20, 50, 100]
    ]
)

methylation_hm450_project_variance_summary[
    "weighted_project_eta_squared"
] = (
    methylation_hm450_project_variance_summary[
        "project_associated_variance_percent"
    ]
    / methylation_hm450_project_variance_summary[
        "captured_variance_percent"
    ]
)

methylation_hm450_project_variance_summary

,n_components,captured_variance_percent,project_associated_variance_percent,weighted_project_eta_squared
0,10,50.426041,31.652758,0.627707
1,20,58.244289,36.161902,0.620866
2,50,65.285149,38.243266,0.585788
3,100,69.090744,38.435515,0.556305


In [63]:
# =======================================================
# Prepare lineage-centered HM450 ICA input
# =======================================================

methylation_hm450_ica_input = np.array(
    methylation_hm450_variable_matrix,
    dtype=np.float32,
    order="C",
    copy=True,
)

for project_id in methylation_hm450_project_ids:
    project_sample_indices = np.flatnonzero(
        methylation_hm450_project_labels == project_id
    )

    project_probe_means = np.mean(
        methylation_hm450_ica_input[
            project_sample_indices
        ],
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    methylation_hm450_ica_input[
        project_sample_indices
    ] -= project_probe_means

maximum_residual_project_mean = max(
    float(
        np.abs(
            np.mean(
                methylation_hm450_ica_input[
                    np.flatnonzero(
                        methylation_hm450_project_labels
                        == project_id
                    )
                ],
                axis=0,
                dtype=np.float64,
            )
        ).max()
    )
    for project_id in methylation_hm450_project_ids
)

{
    "ica_input_shape": (
        methylation_hm450_ica_input.shape
    ),
    "ica_input_dtype": str(
        methylation_hm450_ica_input.dtype
    ),
    "maximum_residual_project_mean": (
        maximum_residual_project_mean
    ),
}

{'ica_input_shape': (8345, 10000),
 'ica_input_dtype': 'float32',
 'maximum_residual_project_mean': 2.4579234958923966e-07}

In [64]:
# =======================================================
# Expand lineage-centered HM450 PCA
# =======================================================

METHYLATION_ICA_PCA_COMPONENTS = 500

methylation_hm450_ica_pca = PCA(
    n_components=METHYLATION_ICA_PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=RANDOM_STATE,
)

methylation_hm450_ica_pca_scores = (
    methylation_hm450_ica_pca.fit_transform(
        methylation_hm450_ica_input
    )
)

methylation_hm450_ica_pca_cumulative_variance = (
    np.cumsum(
        methylation_hm450_ica_pca
        .explained_variance_ratio_
    )
)

methylation_variance_thresholds = {}

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    threshold_name = (
        f"pcs_for_{int(threshold * 100)}_percent"
    )

    if (
        methylation_hm450_ica_pca_cumulative_variance[-1]
        >= threshold
    ):
        methylation_variance_thresholds[
            threshold_name
        ] = int(
            np.searchsorted(
                methylation_hm450_ica_pca_cumulative_variance,
                threshold,
            )
            + 1
        )
    else:
        methylation_variance_thresholds[
            threshold_name
        ] = f">{METHYLATION_ICA_PCA_COMPONENTS}"

{
    "score_shape": (
        methylation_hm450_ica_pca_scores.shape
    ),
    "cumulative_variance_100_pcs": float(
        methylation_hm450_ica_pca_cumulative_variance[
            99
        ]
    ),
    "cumulative_variance_200_pcs": float(
        methylation_hm450_ica_pca_cumulative_variance[
            199
        ]
    ),
    "cumulative_variance_300_pcs": float(
        methylation_hm450_ica_pca_cumulative_variance[
            299
        ]
    ),
    "cumulative_variance_400_pcs": float(
        methylation_hm450_ica_pca_cumulative_variance[
            399
        ]
    ),
    "cumulative_variance_500_pcs": float(
        methylation_hm450_ica_pca_cumulative_variance[
            499
        ]
    ),
    **methylation_variance_thresholds,
}

{'score_shape': (8345, 500),
 'cumulative_variance_100_pcs': 0.5060708522796631,
 'cumulative_variance_200_pcs': 0.5610719323158264,
 'cumulative_variance_300_pcs': 0.5972272157669067,
 'cumulative_variance_400_pcs': 0.6260970830917358,
 'cumulative_variance_500_pcs': 0.6502165198326111,
 'pcs_for_50_percent': 93,
 'pcs_for_60_percent': 309,
 'pcs_for_70_percent': '>500',
 'pcs_for_80_percent': '>500',
 'pcs_for_90_percent': '>500'}

In [65]:
# =======================================================
# Fit primary lineage-centered HM450 ICA
# =======================================================

METHYLATION_ICA_COMPONENTS = 300
METHYLATION_ICA_MAX_ITER = 5_000

methylation_hm450_ica_model_input = (
    np.ascontiguousarray(
        methylation_hm450_ica_pca_scores[
            :,
            :METHYLATION_ICA_COMPONENTS,
        ],
        dtype=np.float64,
    )
)

methylation_hm450_ica = FastICA(
    n_components=METHYLATION_ICA_COMPONENTS,
    algorithm="parallel",
    whiten="unit-variance",
    whiten_solver="svd",
    fun="logcosh",
    max_iter=METHYLATION_ICA_MAX_ITER,
    tol=1e-4,
    random_state=RANDOM_STATE,
)

methylation_hm450_ica_scores = (
    methylation_hm450_ica.fit_transform(
        methylation_hm450_ica_model_input
    )
)

{
    "input_shape": (
        methylation_hm450_ica_model_input.shape
    ),
    "score_shape": (
        methylation_hm450_ica_scores.shape
    ),
    "mixing_shape": (
        methylation_hm450_ica.mixing_.shape
    ),
    "unmixing_shape": (
        methylation_hm450_ica.components_.shape
    ),
    "iterations": int(
        methylation_hm450_ica.n_iter_
    ),
    "reached_iteration_limit": bool(
        methylation_hm450_ica.n_iter_
        >= METHYLATION_ICA_MAX_ITER
    ),
}

{'input_shape': (8345, 300),
 'score_shape': (8345, 300),
 'mixing_shape': (300, 300),
 'unmixing_shape': (300, 300),
 'iterations': 157,
 'reached_iteration_limit': False}

In [66]:
# =======================================================
# Recover and orient HM450 ICA probe loadings
# =======================================================

methylation_hm450_ica_probe_loadings = (
    methylation_hm450_ica.mixing_.T
    @ methylation_hm450_ica_pca.components_[
        :METHYLATION_ICA_COMPONENTS
    ]
).astype(np.float32)

largest_probe_loading_indices = np.argmax(
    np.abs(methylation_hm450_ica_probe_loadings),
    axis=1,
)

methylation_hm450_ica_component_signs = np.where(
    methylation_hm450_ica_probe_loadings[
        np.arange(METHYLATION_ICA_COMPONENTS),
        largest_probe_loading_indices,
    ] >= 0,
    1.0,
    -1.0,
).astype(np.float32)

methylation_hm450_ica_scores_oriented = (
    methylation_hm450_ica_scores
    * methylation_hm450_ica_component_signs[None, :]
).astype(np.float32)

methylation_hm450_ica_probe_loadings_oriented = (
    methylation_hm450_ica_probe_loadings
    * methylation_hm450_ica_component_signs[:, None]
)

methylation_hm450_ica_component_names = [
    f"METH_IC{component:03d}"
    for component in range(
        1,
        METHYLATION_ICA_COMPONENTS + 1,
    )
]

{
    "score_shape": (
        methylation_hm450_ica_scores_oriented.shape
    ),
    "probe_loading_shape": (
        methylation_hm450_ica_probe_loadings_oriented.shape
    ),
    "score_dtype": str(
        methylation_hm450_ica_scores_oriented.dtype
    ),
    "probe_loading_dtype": str(
        methylation_hm450_ica_probe_loadings_oriented.dtype
    ),
    "components_reoriented": int(
        np.sum(
            methylation_hm450_ica_component_signs < 0
        )
    ),
}

{'score_shape': (8345, 300),
 'probe_loading_shape': (300, 10000),
 'score_dtype': 'float32',
 'probe_loading_dtype': 'float32',
 'components_reoriented': 156}

In [67]:
# =======================================================
# Characterize HM450 ICA component structure
# =======================================================

methylation_ica_absolute_loadings = np.abs(
    methylation_hm450_ica_probe_loadings_oriented.astype(
        np.float64,
        copy=False,
    )
)

methylation_ica_loading_power = np.square(
    methylation_ica_absolute_loadings
)

methylation_ica_effective_probe_count = (
    np.square(
        methylation_ica_loading_power.sum(axis=1)
    )
    / np.square(
        methylation_ica_loading_power
    ).sum(axis=1)
)

TOP_PROBE_LOADING_COUNT = 25

methylation_ica_top_loading_fraction = (
    np.partition(
        methylation_ica_absolute_loadings,
        -TOP_PROBE_LOADING_COUNT,
        axis=1,
    )[:, -TOP_PROBE_LOADING_COUNT:].sum(axis=1)
    / methylation_ica_absolute_loadings.sum(axis=1)
)

methylation_ica_score_z = (
    methylation_hm450_ica_scores_oriented
    - methylation_hm450_ica_scores_oriented.mean(axis=0)
) / methylation_hm450_ica_scores_oriented.std(axis=0)

methylation_ica_score_power = np.square(
    methylation_ica_score_z.astype(
        np.float64,
        copy=False,
    )
)

methylation_ica_effective_sample_count = (
    np.square(
        methylation_ica_score_power.sum(axis=0)
    )
    / np.square(
        methylation_ica_score_power
    ).sum(axis=0)
)

top_hm450_sample_count = int(
    np.ceil(
        0.01
        * len(methylation_hm450_sample_metadata)
    )
)

methylation_ica_top_1_percent_power_fraction = (
    np.partition(
        methylation_ica_score_power,
        -top_hm450_sample_count,
        axis=0,
    )[-top_hm450_sample_count:].sum(axis=0)
    / methylation_ica_score_power.sum(axis=0)
)

methylation_hm450_project_sample_counts = np.array(
    [
        np.sum(
            methylation_hm450_project_labels
            == project_id
        )
        for project_id in methylation_hm450_project_ids
    ]
)

methylation_ica_project_score_power = np.vstack(
    [
        methylation_ica_score_power[
            methylation_hm450_project_labels
            == project_id
        ].sum(axis=0)
        for project_id in methylation_hm450_project_ids
    ]
)

methylation_ica_project_mean_score_power = (
    methylation_ica_project_score_power
    / methylation_hm450_project_sample_counts[:, None]
)

methylation_ica_equal_weight_project_fraction = (
    methylation_ica_project_mean_score_power
    / methylation_ica_project_mean_score_power.sum(
        axis=0
    )
)

methylation_ica_effective_project_count = (
    1
    / np.square(
        methylation_ica_equal_weight_project_fraction
    ).sum(axis=0)
)

methylation_ica_component_summary = pd.DataFrame(
    {
        "component": (
            methylation_hm450_ica_component_names
        ),
        "score_skewness": skew(
            methylation_hm450_ica_scores_oriented,
            axis=0,
            bias=False,
        ),
        "score_excess_kurtosis": kurtosis(
            methylation_hm450_ica_scores_oriented,
            axis=0,
            fisher=True,
            bias=False,
        ),
        "effective_probe_count": (
            methylation_ica_effective_probe_count
        ),
        "top_25_abs_loading_fraction": (
            methylation_ica_top_loading_fraction
        ),
        "effective_sample_count": (
            methylation_ica_effective_sample_count
        ),
        "top_1_percent_score_power_fraction": (
            methylation_ica_top_1_percent_power_fraction
        ),
        "effective_project_count": (
            methylation_ica_effective_project_count
        ),
        "maximum_equal_weight_project_fraction": (
            methylation_ica_equal_weight_project_fraction
            .max(axis=0)
        ),
    }
)

methylation_ica_component_summary[
    "absolute_excess_kurtosis"
] = np.abs(
    methylation_ica_component_summary[
        "score_excess_kurtosis"
    ]
)

methylation_ica_component_summary[
    [
        "absolute_excess_kurtosis",
        "effective_probe_count",
        "top_25_abs_loading_fraction",
        "effective_sample_count",
        "top_1_percent_score_power_fraction",
        "effective_project_count",
        "maximum_equal_weight_project_fraction",
    ]
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
    ]
).T

,count,mean,std,min,10%,25%,50%,75%,90%,95%,max
absolute_excess_kurtosis,300.0,64.786201,111.698891,0.901228,6.259590,12.226561,30.517807,73.023899,145.942522,223.444822,1302.409058
effective_probe_count,300.0,2086.686988,772.529414,26.420175,1122.778666,1617.713225,2137.138999,2575.445344,3002.198852,3321.710526,4268.837138
top_25_abs_loading_fraction,300.0,0.015912,0.005148,0.008037,0.011575,0.013052,0.014921,0.017533,0.019937,0.024274,0.049021
effective_sample_count,300.0,383.617279,368.120809,6.396459,56.061649,109.832476,249.146288,548.554187,901.675779,1017.533110,2139.760328
top_1_percent_score_power_fraction,300.0,0.414399,0.176324,0.107692,0.216342,0.259293,0.385477,0.540883,0.676866,0.756395,0.894780
effective_project_count,300.0,10.727531,8.473736,1.123196,1.830736,2.993689,7.741818,18.849039,24.283959,25.573514,28.254575
maximum_equal_weight_project_fraction,300.0,0.363803,0.245419,0.053351,0.087183,0.145923,0.299790,0.560049,0.736610,0.814474,0.943481


In [68]:
# =======================================================
# Screen broadly distributed HM450 ICA candidates
# =======================================================

MIN_METHYLATION_EFFECTIVE_PROJECT_COUNT = 18.0
MAX_METHYLATION_PROJECT_FRACTION = 0.20
MAX_METHYLATION_TOP_1_PERCENT_POWER = 0.30

methylation_ica_component_summary[
    "broad_project_distribution"
] = (
    methylation_ica_component_summary[
        "effective_project_count"
    ]
    >= MIN_METHYLATION_EFFECTIVE_PROJECT_COUNT
)

methylation_ica_component_summary[
    "no_single_project_dominance"
] = (
    methylation_ica_component_summary[
        "maximum_equal_weight_project_fraction"
    ]
    <= MAX_METHYLATION_PROJECT_FRACTION
)

methylation_ica_component_summary[
    "sample_distributed"
] = (
    methylation_ica_component_summary[
        "top_1_percent_score_power_fraction"
    ]
    <= MAX_METHYLATION_TOP_1_PERCENT_POWER
)

methylation_ica_component_summary[
    "candidate_cross_project_distribution"
] = (
    methylation_ica_component_summary[
        [
            "broad_project_distribution",
            "no_single_project_dominance",
            "sample_distributed",
        ]
    ].all(axis=1)
)

methylation_ica_candidate_components = (
    methylation_ica_component_summary.loc[
        methylation_ica_component_summary[
            "candidate_cross_project_distribution"
        ]
    ]
    .sort_values(
        [
            "absolute_excess_kurtosis",
            "effective_project_count",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(
    pd.Series(
        {
            "total_components": (
                METHYLATION_ICA_COMPONENTS
            ),
            "broad_project_distribution": int(
                methylation_ica_component_summary[
                    "broad_project_distribution"
                ].sum()
            ),
            "no_single_project_dominance": int(
                methylation_ica_component_summary[
                    "no_single_project_dominance"
                ].sum()
            ),
            "sample_distributed": int(
                methylation_ica_component_summary[
                    "sample_distributed"
                ].sum()
            ),
            "candidate_components": len(
                methylation_ica_candidate_components
            ),
        }
    )
)

methylation_ica_candidate_components[
    [
        "component",
        "score_excess_kurtosis",
        "effective_probe_count",
        "effective_sample_count",
        "top_1_percent_score_power_fraction",
        "effective_project_count",
        "maximum_equal_weight_project_fraction",
    ]
].head(20)

total_components               300
broad_project_distribution      81
no_single_project_dominance    111
sample_distributed             105
candidate_components            76
dtype: int64

,component,score_excess_kurtosis,effective_probe_count,effective_sample_count,top_1_percent_score_power_fraction,effective_project_count,maximum_equal_weight_project_fraction
0,METH_IC259,80.016121,1886.293563,100.581602,0.286797,20.431672,0.113799
1,METH_IC129,64.883553,2430.798956,123.002869,0.268649,23.202070,0.092535
2,METH_IC139,24.564909,1143.397994,302.909641,0.256012,24.854525,0.069695
3,METH_IC220,24.138020,1396.600296,307.674293,0.261591,23.182490,0.093263
4,METH_IC289,23.983105,1489.571488,309.440602,0.271582,22.480843,0.109650
5,METH_IC250,23.645899,2452.933189,313.356446,0.238985,26.407331,0.064799
6,METH_IC193,22.617952,1894.608527,325.929648,0.285122,18.464479,0.135733
7,METH_IC024,19.404812,2192.526213,372.669941,0.280966,22.296153,0.086386
8,METH_IC052,19.063881,1072.009561,378.428194,0.296056,22.542081,0.115152
9,METH_IC130,15.091724,2200.477032,461.509605,0.292914,19.232957,0.146181


In [69]:
# =======================================================
# Align candidate ICA scores on HM450 samples
# =======================================================

rna_candidate_component_names = (
    rna_ica_candidate_components[
        "component"
    ].to_list()
)

methylation_candidate_component_names = (
    methylation_ica_candidate_components[
        "component"
    ].to_list()
)

rna_component_to_index = {
    component: component_index
    for component_index, component in enumerate(
        rna_ica_component_names
    )
}

methylation_component_to_index = {
    component: component_index
    for component_index, component in enumerate(
        methylation_hm450_ica_component_names
    )
}

rna_candidate_component_indices = np.array(
    [
        rna_component_to_index[component]
        for component in rna_candidate_component_names
    ],
    dtype=np.int32,
)

methylation_candidate_component_indices = np.array(
    [
        methylation_component_to_index[component]
        for component
        in methylation_candidate_component_names
    ],
    dtype=np.int32,
)

rna_candidate_scores_hm450 = np.ascontiguousarray(
    rna_ica_scores_oriented[
        np.ix_(
            methylation_hm450_sample_indices,
            rna_candidate_component_indices,
        )
    ],
    dtype=np.float32,
)

methylation_candidate_scores_hm450 = (
    np.ascontiguousarray(
        methylation_hm450_ica_scores_oriented[
            :,
            methylation_candidate_component_indices,
        ],
        dtype=np.float32,
    )
)

{
    "shared_samples": len(
        methylation_hm450_sample_indices
    ),
    "rna_candidate_score_shape": (
        rna_candidate_scores_hm450.shape
    ),
    "methylation_candidate_score_shape": (
        methylation_candidate_scores_hm450.shape
    ),
}

{'shared_samples': 8345,
 'rna_candidate_score_shape': (8345, 42),
 'methylation_candidate_score_shape': (8345, 76)}

In [70]:
# =======================================================
# Quantify project-wise cross-omic ICA associations
# =======================================================

MIN_CROSS_OMIC_PROJECT_SAMPLES = 30

cross_omic_project_counts = pd.Series(
    {
        project_id: int(
            np.sum(
                methylation_hm450_project_labels
                == project_id
            )
        )
        for project_id in methylation_hm450_project_ids
    }
)

cross_omic_project_ids = (
    cross_omic_project_counts.loc[
        cross_omic_project_counts
        >= MIN_CROSS_OMIC_PROJECT_SAMPLES
    ]
    .index
    .to_numpy()
)

cross_omic_project_correlations = np.empty(
    (
        len(cross_omic_project_ids),
        len(rna_candidate_component_names),
        len(methylation_candidate_component_names),
    ),
    dtype=np.float32,
)

for project_index, project_id in enumerate(
    cross_omic_project_ids
):
    project_sample_indices = np.flatnonzero(
        methylation_hm450_project_labels
        == project_id
    )

    rna_project_scores = (
        rna_candidate_scores_hm450[
            project_sample_indices
        ].astype(
            np.float64,
            copy=False,
        )
    )

    methylation_project_scores = (
        methylation_candidate_scores_hm450[
            project_sample_indices
        ].astype(
            np.float64,
            copy=False,
        )
    )

    rna_centered = (
        rna_project_scores
        - rna_project_scores.mean(axis=0)
    )

    methylation_centered = (
        methylation_project_scores
        - methylation_project_scores.mean(axis=0)
    )

    correlation_numerator = (
        rna_centered.T
        @ methylation_centered
    )

    correlation_denominator = np.outer(
        np.sqrt(
            np.square(rna_centered).sum(axis=0)
        ),
        np.sqrt(
            np.square(
                methylation_centered
            ).sum(axis=0)
        ),
    )

    cross_omic_project_correlations[
        project_index
    ] = np.divide(
        correlation_numerator,
        correlation_denominator,
        out=np.full_like(
            correlation_numerator,
            np.nan,
        ),
        where=correlation_denominator > 0,
    ).astype(np.float32)

valid_project_correlations = np.isfinite(
    cross_omic_project_correlations
)

valid_project_counts = valid_project_correlations.sum(
    axis=0
)

positive_project_counts = np.sum(
    cross_omic_project_correlations > 0,
    axis=0,
)

negative_project_counts = np.sum(
    cross_omic_project_correlations < 0,
    axis=0,
)

cross_omic_pair_summary = pd.DataFrame(
    {
        "rna_component": np.repeat(
            rna_candidate_component_names,
            len(
                methylation_candidate_component_names
            ),
        ),
        "methylation_component": np.tile(
            methylation_candidate_component_names,
            len(rna_candidate_component_names),
        ),
        "median_project_correlation": np.nanmedian(
            cross_omic_project_correlations,
            axis=0,
        ).ravel(),
        "median_absolute_project_correlation": (
            np.nanmedian(
                np.abs(
                    cross_omic_project_correlations
                ),
                axis=0,
            ).ravel()
        ),
        "project_correlation_iqr": (
            (
                np.nanquantile(
                    cross_omic_project_correlations,
                    0.75,
                    axis=0,
                )
                - np.nanquantile(
                    cross_omic_project_correlations,
                    0.25,
                    axis=0,
                )
            ).ravel()
        ),
        "direction_consistency": (
            np.maximum(
                positive_project_counts,
                negative_project_counts,
            )
            / valid_project_counts
        ).ravel(),
        "valid_project_count": (
            valid_project_counts.ravel()
        ),
    }
)

display(
    pd.Series(
        {
            "eligible_projects": len(
                cross_omic_project_ids
            ),
            "minimum_project_samples": int(
                cross_omic_project_counts.loc[
                    cross_omic_project_ids
                ].min()
            ),
            "component_pairs": len(
                cross_omic_pair_summary
            ),
        }
    )
)

cross_omic_pair_summary[
    [
        "median_project_correlation",
        "median_absolute_project_correlation",
        "project_correlation_iqr",
        "direction_consistency",
        "valid_project_count",
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
).T

eligible_projects            32
minimum_project_samples      35
component_pairs            3192
dtype: int64

,count,mean,std,min,50%,75%,90%,95%,99%,max
median_project_correlation,3192.0,-0.000768,0.034093,-0.497353,-0.001133,0.020434,0.039746,0.052106,0.075692,0.321344
median_absolute_project_correlation,3192.0,0.064749,0.017855,0.027315,0.063079,0.073943,0.084912,0.092084,0.107289,0.497353
project_correlation_iqr,3192.0,0.117337,0.026943,0.052516,0.115072,0.133454,0.151989,0.162827,0.184587,0.482722
direction_consistency,3192.0,0.610021,0.080378,0.500000,0.593750,0.656250,0.718750,0.750000,0.843750,0.937500
valid_project_count,3192.0,32.000000,0.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000


In [71]:
# =======================================================
# Characterize cross-omic association tails
# =======================================================

cross_omic_pair_summary[
    "absolute_median_project_correlation"
] = np.abs(
    cross_omic_pair_summary[
        "median_project_correlation"
    ]
)

pair_median_correlations = np.nanmedian(
    cross_omic_project_correlations,
    axis=0,
)

pair_directions = np.where(
    pair_median_correlations >= 0,
    1.0,
    -1.0,
)

same_direction_project_correlations = (
    cross_omic_project_correlations
    * pair_directions[None, :, :]
)

cross_omic_pair_summary[
    "same_direction_ge_0_10_project_count"
] = np.sum(
    same_direction_project_correlations >= 0.10,
    axis=0,
).ravel()

cross_omic_pair_summary[
    "same_direction_ge_0_20_project_count"
] = np.sum(
    same_direction_project_correlations >= 0.20,
    axis=0,
).ravel()

cross_omic_tail_counts = pd.Series(
    {
        "pairs_abs_median_ge_0.05": int(
            (
                cross_omic_pair_summary[
                    "absolute_median_project_correlation"
                ]
                >= 0.05
            ).sum()
        ),
        "pairs_abs_median_ge_0.10": int(
            (
                cross_omic_pair_summary[
                    "absolute_median_project_correlation"
                ]
                >= 0.10
            ).sum()
        ),
        "pairs_abs_median_ge_0.15": int(
            (
                cross_omic_pair_summary[
                    "absolute_median_project_correlation"
                ]
                >= 0.15
            ).sum()
        ),
        "pairs_abs_median_ge_0.20": int(
            (
                cross_omic_pair_summary[
                    "absolute_median_project_correlation"
                ]
                >= 0.20
            ).sum()
        ),
        "pairs_abs_median_ge_0.10_direction_ge_0.75": int(
            (
                (
                    cross_omic_pair_summary[
                        "absolute_median_project_correlation"
                    ]
                    >= 0.10
                )
                & (
                    cross_omic_pair_summary[
                        "direction_consistency"
                    ]
                    >= 0.75
                )
            ).sum()
        ),
    }
)

cross_omic_top_pairs = (
    cross_omic_pair_summary
    .sort_values(
        [
            "absolute_median_project_correlation",
            "direction_consistency",
            "same_direction_ge_0_20_project_count",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(cross_omic_tail_counts)

cross_omic_top_pairs[
    [
        "rna_component",
        "methylation_component",
        "median_project_correlation",
        "absolute_median_project_correlation",
        "direction_consistency",
        "same_direction_ge_0_10_project_count",
        "same_direction_ge_0_20_project_count",
        "project_correlation_iqr",
    ]
].head(20)

pairs_abs_median_ge_0.05                      387
pairs_abs_median_ge_0.10                       14
pairs_abs_median_ge_0.15                        3
pairs_abs_median_ge_0.20                        2
pairs_abs_median_ge_0.10_direction_ge_0.75     14
dtype: int64

,rna_component,methylation_component,median_project_correlation,absolute_median_project_correlation,direction_consistency,same_direction_ge_0_10_project_count,same_direction_ge_0_20_project_count,project_correlation_iqr
0,RNA_IC158,METH_IC152,-0.497353,0.497353,0.87500,24,24,0.482722
1,RNA_IC083,METH_IC232,0.321344,0.321344,0.90625,28,24,0.250640
2,RNA_IC184,METH_IC169,-0.179417,0.179417,0.87500,21,14,0.213754
3,RNA_IC150,METH_IC128,0.133099,0.133099,0.84375,19,7,0.146604
4,RNA_IC169,METH_IC023,-0.130233,0.130233,0.84375,20,6,0.125145
5,RNA_IC175,METH_IC013,0.112888,0.112888,0.78125,17,5,0.108907
6,RNA_IC050,METH_IC234,0.110080,0.110080,0.78125,17,6,0.157540
7,RNA_IC151,METH_IC050,0.109256,0.109256,0.87500,18,2,0.089380
8,RNA_IC001,METH_IC107,0.107940,0.107940,0.75000,16,6,0.169181
9,RNA_IC001,METH_IC241,0.107284,0.107284,0.90625,17,3,0.109725


In [72]:
# =======================================================
# Define exploratory cross-omic candidate pairs
# =======================================================

MIN_ABSOLUTE_MEDIAN_CORRELATION = 0.10
MIN_DIRECTION_CONSISTENCY = 0.75
MIN_SAME_DIRECTION_PROJECT_COUNT = 16

cross_omic_candidate_pairs = (
    cross_omic_pair_summary.loc[
        (
            cross_omic_pair_summary[
                "absolute_median_project_correlation"
            ]
            >= MIN_ABSOLUTE_MEDIAN_CORRELATION
        )
        & (
            cross_omic_pair_summary[
                "direction_consistency"
            ]
            >= MIN_DIRECTION_CONSISTENCY
        )
        & (
            cross_omic_pair_summary[
                "same_direction_ge_0_10_project_count"
            ]
            >= MIN_SAME_DIRECTION_PROJECT_COUNT
        )
    ]
    .copy()
    .sort_values(
        [
            "absolute_median_project_correlation",
            "direction_consistency",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

cross_omic_candidate_pairs[
    "association_direction"
] = np.where(
    cross_omic_candidate_pairs[
        "median_project_correlation"
    ]
    > 0,
    "positive",
    "negative",
)

cross_omic_candidate_pairs[
    "magnitude_stratum"
] = pd.cut(
    cross_omic_candidate_pairs[
        "absolute_median_project_correlation"
    ],
    bins=[0.10, 0.15, 0.20, np.inf],
    labels=[
        "0.10_to_below_0.15",
        "0.15_to_below_0.20",
        "0.20_or_higher",
    ],
    right=False,
)

display(
    pd.Series(
        {
            "candidate_pairs": len(
                cross_omic_candidate_pairs
            ),
            "unique_rna_components": (
                cross_omic_candidate_pairs[
                    "rna_component"
                ].nunique()
            ),
            "unique_methylation_components": (
                cross_omic_candidate_pairs[
                    "methylation_component"
                ].nunique()
            ),
            "positive_pairs": int(
                (
                    cross_omic_candidate_pairs[
                        "association_direction"
                    ]
                    == "positive"
                ).sum()
            ),
            "negative_pairs": int(
                (
                    cross_omic_candidate_pairs[
                        "association_direction"
                    ]
                    == "negative"
                ).sum()
            ),
        }
    )
)

cross_omic_candidate_pairs[
    [
        "rna_component",
        "methylation_component",
        "median_project_correlation",
        "direction_consistency",
        "same_direction_ge_0_10_project_count",
        "same_direction_ge_0_20_project_count",
        "project_correlation_iqr",
        "magnitude_stratum",
    ]
]

candidate_pairs                  14
unique_rna_components            10
unique_methylation_components    12
positive_pairs                    8
negative_pairs                    6
dtype: int64

,rna_component,methylation_component,median_project_correlation,direction_consistency,same_direction_ge_0_10_project_count,same_direction_ge_0_20_project_count,project_correlation_iqr,magnitude_stratum
0,RNA_IC158,METH_IC152,-0.497353,0.87500,24,24,0.482722,0.20_or_higher
1,RNA_IC083,METH_IC232,0.321344,0.90625,28,24,0.250640,0.20_or_higher
2,RNA_IC184,METH_IC169,-0.179417,0.87500,21,14,0.213754,0.15_to_below_0.20
3,RNA_IC150,METH_IC128,0.133099,0.84375,19,7,0.146604,0.10_to_below_0.15
4,RNA_IC169,METH_IC023,-0.130233,0.84375,20,6,0.125145,0.10_to_below_0.15
5,RNA_IC175,METH_IC013,0.112888,0.78125,17,5,0.108907,0.10_to_below_0.15
6,RNA_IC050,METH_IC234,0.110080,0.78125,17,6,0.157540,0.10_to_below_0.15
7,RNA_IC151,METH_IC050,0.109256,0.87500,18,2,0.089380,0.10_to_below_0.15
8,RNA_IC001,METH_IC107,0.107940,0.75000,16,6,0.169181,0.10_to_below_0.15
9,RNA_IC001,METH_IC241,0.107284,0.90625,17,3,0.109725,0.10_to_below_0.15


In [73]:
# =======================================================
# Extract candidate-pair project correlations
# =======================================================

rna_candidate_local_index = {
    component: component_index
    for component_index, component in enumerate(
        rna_candidate_component_names
    )
}

methylation_candidate_local_index = {
    component: component_index
    for component_index, component in enumerate(
        methylation_candidate_component_names
    )
}

candidate_pair_project_tables = []

for pair_index, pair in cross_omic_candidate_pairs.iterrows():
    rna_index = rna_candidate_local_index[
        pair["rna_component"]
    ]

    methylation_index = (
        methylation_candidate_local_index[
            pair["methylation_component"]
        ]
    )

    project_correlations = (
        cross_omic_project_correlations[
            :,
            rna_index,
            methylation_index,
        ]
    )

    direction_sign = np.sign(
        pair["median_project_correlation"]
    )

    candidate_pair_project_tables.append(
        pd.DataFrame(
            {
                "candidate_pair": (
                    f"CROSS_OMIC_PAIR_{pair_index + 1:02d}"
                ),
                "rna_component": pair["rna_component"],
                "methylation_component": (
                    pair["methylation_component"]
                ),
                "association_direction": (
                    pair["association_direction"]
                ),
                "project_id": cross_omic_project_ids,
                "project_sample_count": (
                    cross_omic_project_counts
                    .reindex(cross_omic_project_ids)
                    .to_numpy()
                ),
                "project_correlation": (
                    project_correlations
                ),
                "aligned_project_correlation": (
                    project_correlations
                    * direction_sign
                ),
            }
        )
    )

cross_omic_candidate_project_correlations = pd.concat(
    candidate_pair_project_tables,
    ignore_index=True,
)

cross_omic_candidate_project_correlations[
    "same_direction_ge_0_10"
] = (
    cross_omic_candidate_project_correlations[
        "aligned_project_correlation"
    ]
    >= 0.10
)

cross_omic_candidate_project_correlations[
    "same_direction_ge_0_20"
] = (
    cross_omic_candidate_project_correlations[
        "aligned_project_correlation"
    ]
    >= 0.20
)

first_candidate_pair = (
    cross_omic_candidate_project_correlations[
        "candidate_pair"
    ].iloc[0]
)

display(
    {
        "table_rows": len(
            cross_omic_candidate_project_correlations
        ),
        "candidate_pairs": (
            cross_omic_candidate_project_correlations[
                "candidate_pair"
            ].nunique()
        ),
        "projects_per_pair": (
            cross_omic_candidate_project_correlations
            .groupby("candidate_pair")
            .size()
            .unique()
            .tolist()
        ),
        "first_candidate_pair": first_candidate_pair,
    }
)

cross_omic_candidate_project_correlations.loc[
    cross_omic_candidate_project_correlations[
        "candidate_pair"
    ].eq(first_candidate_pair)
].sort_values(
    "aligned_project_correlation",
    ascending=False,
).head(15)

{'table_rows': 448,
 'candidate_pairs': 14,
 'projects_per_pair': [32],
 'first_candidate_pair': 'CROSS_OMIC_PAIR_01'}

,candidate_pair,rna_component,methylation_component,association_direction,project_id,project_sample_count,project_correlation,aligned_project_correlation,same_direction_ge_0_10,same_direction_ge_0_20
28,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-THYM,120,-0.910900,0.910900,True,True
19,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-PAAD,178,-0.809064,0.809064,True,True
20,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-PCPG,179,-0.804643,0.804643,True,True
27,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-THCA,505,-0.804105,0.804105,True,True
11,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-KIRC,318,-0.708772,0.708772,True,True
18,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-MESO,87,-0.700724,0.700724,True,True
16,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-LUAD,453,-0.686755,0.686755,True,True
13,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-LAML,134,-0.654790,0.654790,True,True
10,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-KICH,66,-0.645572,0.645572,True,True
17,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-LUSC,370,-0.631170,0.631170,True,True


In [74]:
# =======================================================
# Inspect HM450 probe annotation fields
# =======================================================

display(
    pd.Series(
        {
            "selected_probes": len(
                methylation_variable_probe_mapping
            ),
            "annotation_columns": (
                methylation_variable_probe_mapping
                .columns
                .to_list()
            ),
        }
    )
)

methylation_variable_probe_mapping.head()

selected_probes                                                   10000
annotation_columns    [model_matrix_row_index, representation, methy...
dtype: object

,model_matrix_row_index,representation,methylation_platform,matrix_row_index,probe_id,source_platform_probe_order,source_shared_probe_order,probe_missing_beta_fraction,probe_qc_eligibility_scope,probe_qc_eligible,hm27_missing_beta_fraction,hm450_missing_beta_fraction,hm27_probe_qc_eligible,hm450_probe_qc_eligible,shared_probe_qc_eligible,shared_matrix_row_index,median_within_project_variance
0,0,shared_hm27_hm450,shared_hm27_hm450,0,cg00000292,NaN,0.0,NaN,shared_both_platforms,True,0.001234,0.000000,True,True,True,0,1.398366
1,1,shared_hm27_hm450,shared_hm27_hm450,1,cg00002426,NaN,1.0,NaN,shared_both_platforms,True,0.007403,0.000000,True,True,True,1,1.411903
2,2,shared_hm27_hm450,shared_hm27_hm450,2,cg00003994,NaN,2.0,NaN,shared_both_platforms,True,0.003701,0.023292,True,True,True,2,2.202043
3,3,shared_hm27_hm450,shared_hm27_hm450,3,cg00005847,NaN,3.0,NaN,shared_both_platforms,True,0.000000,0.000000,True,True,True,3,2.003737
4,9,shared_hm27_hm450,shared_hm27_hm450,9,cg00011459,NaN,9.0,NaN,shared_both_platforms,True,0.024059,0.000119,True,True,True,9,0.499001


In [75]:
# =======================================================
# Extract top probes from candidate HM450 components
# =======================================================

TOP_METHYLATION_PROBES_PER_DIRECTION = 50

candidate_methylation_components = (
    cross_omic_candidate_pairs[
        "methylation_component"
    ]
    .drop_duplicates()
    .to_list()
)

selected_probe_metadata = (
    methylation_variable_probe_mapping
    .reset_index(drop=True)
    .copy()
)

selected_probe_metadata.insert(
    0,
    "ica_probe_column_index",
    np.arange(
        len(selected_probe_metadata),
        dtype=np.int32,
    ),
)

candidate_methylation_probe_tables = []

for component in candidate_methylation_components:
    component_index = methylation_component_to_index[
        component
    ]

    component_loadings = (
        methylation_hm450_ica_probe_loadings_oriented[
            component_index
        ]
    )

    positive_probe_indices = np.argsort(
        component_loadings
    )[-TOP_METHYLATION_PROBES_PER_DIRECTION:][::-1]

    negative_probe_indices = np.argsort(
        component_loadings
    )[:TOP_METHYLATION_PROBES_PER_DIRECTION]

    for direction, probe_indices in [
        ("positive", positive_probe_indices),
        ("negative", negative_probe_indices),
    ]:
        probe_table = selected_probe_metadata.iloc[
            probe_indices
        ].copy()

        probe_table.insert(
            0,
            "component",
            component,
        )

        probe_table.insert(
            1,
            "loading_direction",
            direction,
        )

        probe_table.insert(
            2,
            "directional_rank",
            np.arange(
                1,
                len(probe_indices) + 1,
                dtype=np.int32,
            ),
        )

        probe_table.insert(
            3,
            "probe_loading",
            component_loadings[
                probe_indices
            ],
        )

        candidate_methylation_probe_tables.append(
            probe_table
        )

candidate_methylation_top_probes = pd.concat(
    candidate_methylation_probe_tables,
    ignore_index=True,
)

first_methylation_component = (
    candidate_methylation_components[0]
)

display(
    {
        "candidate_components": len(
            candidate_methylation_components
        ),
        "top_probes_per_direction": (
            TOP_METHYLATION_PROBES_PER_DIRECTION
        ),
        "table_rows": len(
            candidate_methylation_top_probes
        ),
        "first_component": (
            first_methylation_component
        ),
    }
)

candidate_methylation_top_probes.loc[
    candidate_methylation_top_probes[
        "component"
    ].eq(first_methylation_component),
    [
        "component",
        "loading_direction",
        "directional_rank",
        "probe_id",
        "probe_loading",
        "median_within_project_variance",
        "hm450_missing_beta_fraction",
    ],
].groupby(
    "loading_direction",
    sort=False,
).head(15)

{'candidate_components': 12,
 'top_probes_per_direction': 50,
 'table_rows': 1200,
 'first_component': 'METH_IC152'}

,component,loading_direction,directional_rank,probe_id,probe_loading,median_within_project_variance,hm450_missing_beta_fraction
0,METH_IC152,positive,1,cg09229960,1.338381,7.849493,0.000119
1,METH_IC152,positive,2,cg05935584,1.233302,6.465564,0.003446
2,METH_IC152,positive,3,cg08798116,1.225417,7.062622,0.000119
3,METH_IC152,positive,4,cg11233153,1.141885,5.823558,0.000000
4,METH_IC152,positive,5,cg20085077,1.135170,5.650967,0.000119
5,METH_IC152,positive,6,cg26745032,1.132677,6.426073,0.000119
6,METH_IC152,positive,7,cg15977272,1.095760,5.404728,0.003684
7,METH_IC152,positive,8,cg09018810,1.074658,5.016266,0.000000
8,METH_IC152,positive,9,cg26246138,1.057589,5.974684,0.000000
9,METH_IC152,positive,10,cg11653864,1.048695,4.691922,0.000000


In [76]:
# =======================================================
# Inspect exported HM450 annotation
# =======================================================

hm450_annotation_path = (
    Paths.metadata
    / "illumina_hm450_ilmn12_hg19_probe_annotation.csv"
)

hm450_annotation_preview = pd.read_csv(
    hm450_annotation_path,
    nrows=5,
    low_memory=False,
)

display(
    {
        "annotation_path": str(
            hm450_annotation_path
        ),
        "annotation_columns": (
            hm450_annotation_preview.columns.to_list()
        ),
    }
)

hm450_annotation_preview

{'annotation_path': 'C:\\Users\\paula\\OneDrive\\Documentos\\Proyectos\\pancancer-epigenetics\\data\\interim\\metadata\\illumina_hm450_ilmn12_hg19_probe_annotation.csv',
 'annotation_columns': ['probe_id',
  'chr',
  'pos',
  'strand',
  'Name',
  'AddressA',
  'AddressB',
  'ProbeSeqA',
  'ProbeSeqB',
  'Type',
  'NextBase',
  'Color',
  'Probe_rs',
  'Probe_maf',
  'CpG_rs',
  'CpG_maf',
  'SBE_rs',
  'SBE_maf',
  'Islands_Name',
  'Relation_to_Island',
  'Forward_Sequence',
  'SourceSeq',
  'Random_Loci',
  'Methyl27_Loci',
  'UCSC_RefGene_Name',
  'UCSC_RefGene_Accession',
  'UCSC_RefGene_Group',
  'Phantom',
  'DMR',
  'Enhancer',
  'HMM_Island',
  'Regulatory_Feature_Name',
  'Regulatory_Feature_Group',
  'DHS']}

,probe_id,chr,pos,strand,Name,AddressA,AddressB,ProbeSeqA,ProbeSeqB,Type,...,UCSC_RefGene_Name,UCSC_RefGene_Accession,UCSC_RefGene_Group,Phantom,DMR,Enhancer,HMM_Island,Regulatory_Feature_Name,Regulatory_Feature_Group,DHS
0,cg00050873,chrY,9363356,-,cg00050873,32735311,31717405,ACAAAAAAACAACACACAACTATAATAATTTTTAAAATAAATAAAC...,ACGAAAAAACAACGCACAACTATAATAATTTTTAAAATAAATAAAC...,I,...,TSPY4;FAM197Y2,NM_001164471;NR_001553,Body;TSS1500,NaN,NaN,NaN,Y:9973136-9976273,NaN,NaN,NaN
1,cg00212031,chrY,21239348,-,cg00212031,29674443,38703326,CCCAATTAACCACAAAAACTAAACAAATTATACAATCAAAAAAACA...,CCCAATTAACCGCAAAAACTAAACAAATTATACGATCGAAAAAACG...,I,...,TTTY14,NR_001543,TSS200,NaN,NaN,NaN,Y:19697854-19699393,NaN,NaN,NaN
2,cg00213748,chrY,8148233,-,cg00213748,30703409,36767301,TTTTAACACCTAACACCATTTTAACAATAAAAATTCTACAAAAAAA...,TTTTAACGCCTAACACCGTTTTAACGATAAAAATTCTACAAAAAAA...,I,...,NaN,NaN,NaN,NaN,NaN,NaN,Y:8207555-8208234,NaN,NaN,NaN
3,cg00214611,chrY,15815688,-,cg00214611,69792329,46723459,CTAACTTCCAAACCACACTTTATATACTAAACTACAATATAACACA...,CTAACTTCCGAACCGCGCTTTATATACTAAACTACAATATAACGCG...,I,...,TMSB4Y;TMSB4Y,NM_004202;NM_004202,1stExon;5'UTR,NaN,NaN,NaN,Y:14324883-14325218,Y:15815422-15815706,Promoter_Associated_Cell_type_specific,NaN
4,cg00455876,chrY,9385539,-,cg00455876,27653438,69732350,AACTCTAAACTACCCAACACAAACTCCAAAAACTTCTCAAAAAAAA...,AACTCTAAACTACCCGACACAAACTCCAAAAACTTCTCGAAAAAAA...,I,...,NaN,NaN,NaN,NaN,NaN,NaN,Y:9993394-9995882,NaN,NaN,NaN


In [77]:
# =======================================================
# Annotate candidate HM450 probes
# =======================================================

hm450_annotation_columns = [
    "probe_id",
    "chr",
    "pos",
    "strand",
    "Type",
    "Probe_rs",
    "Probe_maf",
    "CpG_rs",
    "CpG_maf",
    "SBE_rs",
    "SBE_maf",
    "Islands_Name",
    "Relation_to_Island",
    "UCSC_RefGene_Name",
    "UCSC_RefGene_Accession",
    "UCSC_RefGene_Group",
    "Enhancer",
    "Regulatory_Feature_Name",
    "Regulatory_Feature_Group",
    "DHS",
]

hm450_probe_annotation = pd.read_csv(
    hm450_annotation_path,
    usecols=hm450_annotation_columns,
    low_memory=False,
)

candidate_methylation_top_probes_annotated = (
    candidate_methylation_top_probes.merge(
        hm450_probe_annotation,
        on="probe_id",
        how="left",
    )
)

display(
    {
        "table_rows": len(
            candidate_methylation_top_probes_annotated
        ),
        "unique_probes": (
            candidate_methylation_top_probes_annotated[
                "probe_id"
            ].nunique()
        ),
        "probes_with_coordinates": int(
            candidate_methylation_top_probes_annotated[
                "chr"
            ].notna().sum()
        ),
        "probes_with_gene_annotation": int(
            candidate_methylation_top_probes_annotated[
                "UCSC_RefGene_Name"
            ].notna().sum()
        ),
        "probes_with_island_relation": int(
            candidate_methylation_top_probes_annotated[
                "Relation_to_Island"
            ].notna().sum()
        ),
        "probes_with_probe_or_cpg_snp": int(
            (
                candidate_methylation_top_probes_annotated[
                    ["Probe_rs", "CpG_rs"]
                ]
                .notna()
                .any(axis=1)
            ).sum()
        ),
    }
)

candidate_methylation_top_probes_annotated.loc[
    candidate_methylation_top_probes_annotated[
        "component"
    ].eq(first_methylation_component),
    [
        "component",
        "loading_direction",
        "directional_rank",
        "probe_id",
        "probe_loading",
        "chr",
        "pos",
        "Relation_to_Island",
        "UCSC_RefGene_Name",
        "UCSC_RefGene_Group",
        "Enhancer",
        "DHS",
    ],
].head(30)

{'table_rows': 1200,
 'unique_probes': 890,
 'probes_with_coordinates': 1200,
 'probes_with_gene_annotation': 1192,
 'probes_with_island_relation': 1200,
 'probes_with_probe_or_cpg_snp': 190}

,component,loading_direction,directional_rank,probe_id,probe_loading,chr,pos,Relation_to_Island,UCSC_RefGene_Name,UCSC_RefGene_Group,Enhancer,DHS
0,METH_IC152,positive,1,cg09229960,1.338381,chrX,153607856,Island,EMD,1stExon,NaN,NaN
1,METH_IC152,positive,2,cg05935584,1.233302,chrX,150151823,Island,HMGB3;HMGB3,1stExon;5'UTR,NaN,NaN
2,METH_IC152,positive,3,cg08798116,1.225417,chrX,132548882,Island,GPC4,1stExon,NaN,NaN
3,METH_IC152,positive,4,cg11233153,1.141885,chrX,153718691,Island,SLC10A3;SLC10A3;SLC10A3;SLC10A3,5'UTR;1stExon;5'UTR;5'UTR,NaN,NaN
4,METH_IC152,positive,5,cg20085077,1.135170,chrX,100740421,Island,ARMCX4,Body,NaN,True
5,METH_IC152,positive,6,cg26745032,1.132677,chrX,16964743,Island,REPS2;REPS2,TSS200;TSS200,NaN,NaN
6,METH_IC152,positive,7,cg15977272,1.095760,chrX,68048909,Island,EFNB1;EFNB1,5'UTR;1stExon,NaN,NaN
7,METH_IC152,positive,8,cg09018810,1.074658,chrX,148586931,Island,IDS;IDS;IDS,TSS200;TSS200;TSS200,NaN,NaN
8,METH_IC152,positive,9,cg26246138,1.057589,chrX,18372612,Island,SCML2,5'UTR,NaN,NaN
9,METH_IC152,positive,10,cg11653864,1.048695,chrX,47509884,Island,ELK1;ELK1;ELK1;ELK1,5'UTR;1stExon;1stExon;5'UTR,NaN,NaN


In [78]:
# =======================================================
# Quantify chromosome concentration in HM450 ICA
# =======================================================

methylation_variable_probe_annotation = (
    methylation_variable_probe_mapping[
        ["probe_id"]
    ]
    .merge(
        hm450_probe_annotation[
            ["probe_id", "chr"]
        ],
        on="probe_id",
        how="left",
    )
)

methylation_chromosome_indicator = pd.get_dummies(
    methylation_variable_probe_annotation["chr"],
    dtype=np.float64,
)

methylation_chromosome_names = (
    methylation_chromosome_indicator
    .columns
    .to_numpy()
)

methylation_ica_chromosome_power = (
    methylation_ica_loading_power
    @ methylation_chromosome_indicator.to_numpy()
)

methylation_ica_chromosome_power_fraction = (
    methylation_ica_chromosome_power
    / methylation_ica_loading_power.sum(
        axis=1,
        keepdims=True,
    )
)

dominant_chromosome_indices = np.argmax(
    methylation_ica_chromosome_power_fraction,
    axis=1,
)

sex_chromosome_mask = np.isin(
    methylation_chromosome_names,
    ["chrX", "chrY"],
)

methylation_ica_chromosome_summary = pd.DataFrame(
    {
        "component": (
            methylation_hm450_ica_component_names
        ),
        "dominant_chromosome": (
            methylation_chromosome_names[
                dominant_chromosome_indices
            ]
        ),
        "dominant_chromosome_power_fraction": (
            methylation_ica_chromosome_power_fraction[
                np.arange(
                    METHYLATION_ICA_COMPONENTS
                ),
                dominant_chromosome_indices,
            ]
        ),
        "sex_chromosome_power_fraction": (
            methylation_ica_chromosome_power_fraction[
                :,
                sex_chromosome_mask,
            ].sum(axis=1)
        ),
    }
)

cross_omic_methylation_chromosome_summary = (
    methylation_ica_chromosome_summary.loc[
        methylation_ica_chromosome_summary[
            "component"
        ].isin(candidate_methylation_components)
    ]
    .sort_values(
        "sex_chromosome_power_fraction",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    {
        "selected_probes": len(
            methylation_variable_probe_annotation
        ),
        "chromosomes represented": len(
            methylation_chromosome_names
        ),
        "all_components_dominant_chr_ge_0.50": int(
            (
                methylation_ica_chromosome_summary[
                    "dominant_chromosome_power_fraction"
                ]
                >= 0.50
            ).sum()
        ),
        "cross_omic_components_dominant_chr_ge_0.50": int(
            (
                cross_omic_methylation_chromosome_summary[
                    "dominant_chromosome_power_fraction"
                ]
                >= 0.50
            ).sum()
        ),
    }
)

cross_omic_methylation_chromosome_summary

{'selected_probes': 10000,
 'chromosomes represented': 23,
 'all_components_dominant_chr_ge_0.50': 7,
 'cross_omic_components_dominant_chr_ge_0.50': 1}

,component,dominant_chromosome,dominant_chromosome_power_fraction,sex_chromosome_power_fraction
0,METH_IC152,chrX,0.973241,0.973241
1,METH_IC241,chrX,0.251706,0.251706
2,METH_IC023,chrX,0.232662,0.232662
3,METH_IC013,chrX,0.161364,0.161364
4,METH_IC050,chrX,0.134567,0.134567
5,METH_IC033,chrX,0.122568,0.122568
6,METH_IC107,chr6,0.121868,0.095282
7,METH_IC169,chr1,0.101146,0.075671
8,METH_IC234,chr1,0.113123,0.043709
9,METH_IC109,chr1,0.130182,0.039936


In [79]:
# =======================================================
# Quantify sex-chromosome loading enrichment
# =======================================================

selected_probe_chromosome_counts = (
    methylation_variable_probe_annotation[
        "chr"
    ].value_counts()
)

selected_probe_chromosome_fractions = (
    selected_probe_chromosome_counts
    / selected_probe_chromosome_counts.sum()
)

selected_sex_chromosome_fraction = float(
    selected_probe_chromosome_fractions.reindex(
        ["chrX", "chrY"],
        fill_value=0.0,
    ).sum()
)

methylation_ica_chromosome_summary[
    "sex_chromosome_enrichment"
] = (
    methylation_ica_chromosome_summary[
        "sex_chromosome_power_fraction"
    ]
    / selected_sex_chromosome_fraction
)

cross_omic_methylation_chromosome_summary = (
    methylation_ica_chromosome_summary.loc[
        methylation_ica_chromosome_summary[
            "component"
        ].isin(candidate_methylation_components)
    ]
    .sort_values(
        "sex_chromosome_enrichment",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    {
        "selected_probe_sex_chromosome_fraction": (
            selected_sex_chromosome_fraction
        ),
        "selected_chrX_probe_count": int(
            selected_probe_chromosome_counts.get(
                "chrX",
                0,
            )
        ),
        "selected_chrY_probe_count": int(
            selected_probe_chromosome_counts.get(
                "chrY",
                0,
            )
        ),
    }
)

cross_omic_methylation_chromosome_summary[
    [
        "component",
        "dominant_chromosome",
        "dominant_chromosome_power_fraction",
        "sex_chromosome_power_fraction",
        "sex_chromosome_enrichment",
    ]
]

{'selected_probe_sex_chromosome_fraction': 0.0697,
 'selected_chrX_probe_count': 697,
 'selected_chrY_probe_count': 0}

,component,dominant_chromosome,dominant_chromosome_power_fraction,sex_chromosome_power_fraction,sex_chromosome_enrichment
0,METH_IC152,chrX,0.973241,0.973241,13.963286
1,METH_IC241,chrX,0.251706,0.251706,3.611279
2,METH_IC023,chrX,0.232662,0.232662,3.338053
3,METH_IC013,chrX,0.161364,0.161364,2.315118
4,METH_IC050,chrX,0.134567,0.134567,1.930661
5,METH_IC033,chrX,0.122568,0.122568,1.758511
6,METH_IC107,chr6,0.121868,0.095282,1.367029
7,METH_IC169,chr1,0.101146,0.075671,1.085661
8,METH_IC234,chr1,0.113123,0.043709,0.627102
9,METH_IC109,chr1,0.130182,0.039936,0.572973


In [80]:
# =======================================================
# Flag chrX enrichment in cross-omic candidates
# =======================================================

cross_omic_candidate_pairs_chr = (
    cross_omic_candidate_pairs.merge(
        methylation_ica_chromosome_summary[
            [
                "component",
                "dominant_chromosome",
                "dominant_chromosome_power_fraction",
                "sex_chromosome_power_fraction",
                "sex_chromosome_enrichment",
            ]
        ].rename(
            columns={
                "component": "methylation_component",
            }
        ),
        on="methylation_component",
        how="left",
    )
)

cross_omic_candidate_pairs_chr[
    "chrX_loading_status"
] = np.select(
    [
        (
            cross_omic_candidate_pairs_chr[
                "sex_chromosome_power_fraction"
            ]
            >= 0.50
        ),
        (
            cross_omic_candidate_pairs_chr[
                "sex_chromosome_enrichment"
            ]
            >= 2.00
        ),
        (
            cross_omic_candidate_pairs_chr[
                "sex_chromosome_enrichment"
            ]
            >= 1.50
        ),
    ],
    [
        "extreme_chrX_concentration",
        "notable_chrX_enrichment",
        "mild_chrX_enrichment",
    ],
    default="no_major_chrX_enrichment",
)

cross_omic_candidate_pairs_chr[
    "candidate_review_status"
] = np.select(
    [
        cross_omic_candidate_pairs_chr[
            "chrX_loading_status"
        ].eq("extreme_chrX_concentration"),
        cross_omic_candidate_pairs_chr[
            "chrX_loading_status"
        ].eq("notable_chrX_enrichment"),
        cross_omic_candidate_pairs_chr[
            "chrX_loading_status"
        ].eq("mild_chrX_enrichment"),
    ],
    [
        "unresolved_sex_chromosome_confounding",
        "prioritized_sex_sensitivity",
        "secondary_sex_sensitivity",
    ],
    default="retained_for_further_characterization",
)

display(
    cross_omic_candidate_pairs_chr[
        "candidate_review_status"
    ].value_counts()
)

cross_omic_candidate_pairs_chr[
    [
        "rna_component",
        "methylation_component",
        "median_project_correlation",
        "sex_chromosome_power_fraction",
        "sex_chromosome_enrichment",
        "chrX_loading_status",
        "candidate_review_status",
    ]
]

candidate_review_status
retained_for_further_characterization    7
prioritized_sex_sensitivity              4
secondary_sex_sensitivity                2
unresolved_sex_chromosome_confounding    1
Name: count, dtype: int64

,rna_component,methylation_component,median_project_correlation,sex_chromosome_power_fraction,sex_chromosome_enrichment,chrX_loading_status,candidate_review_status
0,RNA_IC158,METH_IC152,-0.497353,0.973241,13.963286,extreme_chrX_concentration,unresolved_sex_chromosome_confounding
1,RNA_IC083,METH_IC232,0.321344,0.033873,0.485978,no_major_chrX_enrichment,retained_for_further_characterization
2,RNA_IC184,METH_IC169,-0.179417,0.075671,1.085661,no_major_chrX_enrichment,retained_for_further_characterization
3,RNA_IC150,METH_IC128,0.133099,0.034392,0.493434,no_major_chrX_enrichment,retained_for_further_characterization
4,RNA_IC169,METH_IC023,-0.130233,0.232662,3.338053,notable_chrX_enrichment,prioritized_sex_sensitivity
5,RNA_IC175,METH_IC013,0.112888,0.161364,2.315118,notable_chrX_enrichment,prioritized_sex_sensitivity
6,RNA_IC050,METH_IC234,0.110080,0.043709,0.627102,no_major_chrX_enrichment,retained_for_further_characterization
7,RNA_IC151,METH_IC050,0.109256,0.134567,1.930661,mild_chrX_enrichment,secondary_sex_sensitivity
8,RNA_IC001,METH_IC107,0.107940,0.095282,1.367029,no_major_chrX_enrichment,retained_for_further_characterization
9,RNA_IC001,METH_IC241,0.107284,0.251706,3.611279,notable_chrX_enrichment,prioritized_sex_sensitivity


## Sex-at-birth sensitivity assessment

The confounder-covariate artifact was extended in notebook 204 with
case-level `sex_at_birth` retrieved from the NCI Genomic Data Commons.

This section evaluates whether candidate methylation components with chrX
loading enrichment are also associated with observed demographic sex and
whether the exploratory cross-omic correlations persist after within-project
sex adjustment.

The analysis is restricted to sensitivity assessment. Sex-associated
component structure is not interpreted as a tumor karyotype measurement or
as evidence of a causal biological mechanism.

In [81]:
# =======================================================
# Load updated sex-at-birth covariate
# =======================================================

CONFOUNDER_COVARIATES_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_confounder_covariates.csv"
)

updated_confounder_covariates = pd.read_csv(
    CONFOUNDER_COVARIATES_PATH
)

print(
    "Updated confounder-covariate table: "
    f"{updated_confounder_covariates.shape}"
)

print(
    "Unique cases: "
    f"{updated_confounder_covariates['case_submitter_id'].nunique():,}"
)

display(
    updated_confounder_covariates[
        "sex_at_birth"
    ].value_counts(
        dropna=False
    )
)

Updated confounder-covariate table: (9965, 18)
Unique cases: 9,965


sex_at_birth
female     5210
male       4749
unknown       4
NaN           2
Name: count, dtype: int64

In [82]:
# =======================================================
# Align sex-at-birth with HM450 samples
# =======================================================

methylation_hm450_sex_metadata = (
    methylation_hm450_sample_metadata
    .merge(
        updated_confounder_covariates[
            [
                "case_submitter_id",
                "sex_at_birth",
            ]
        ],
        on="case_submitter_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

methylation_hm450_sex_metadata[
    "binary_sex_available"
] = methylation_hm450_sex_metadata[
    "sex_at_birth"
].isin(
    [
        "female",
        "male",
    ]
)

print(
    "HM450 sex-aligned metadata: "
    f"{methylation_hm450_sex_metadata.shape}"
)

print(
    "Binary sex-at-birth available: "
    f"{methylation_hm450_sex_metadata['binary_sex_available'].sum():,}"
)

print(
    "Unknown or unavailable: "
    f"{(~methylation_hm450_sex_metadata['binary_sex_available']).sum():,}"
)

display(
    methylation_hm450_sex_metadata[
        "sex_at_birth"
    ].value_counts(
        dropna=False
    )
)

display(
    methylation_hm450_sex_metadata.head()
)

HM450 sex-aligned metadata: (8345, 7)
Binary sex-at-birth available: 8,339
Unknown or unavailable: 6


sex_at_birth
male       4269
female     4070
unknown       4
NaN           2
Name: count, dtype: int64

,final_sample_column_index,case_submitter_id,sample_submitter_id,project_id,methylation_platform,sex_at_birth,binary_sex_available
0,38,TCGA-05-4384,TCGA-05-4384-01A,TCGA-LUAD,Illumina Human Methylation 450,male,True
1,40,TCGA-05-4390,TCGA-05-4390-01A,TCGA-LUAD,Illumina Human Methylation 450,female,True
2,42,TCGA-05-4396,TCGA-05-4396-01A,TCGA-LUAD,Illumina Human Methylation 450,male,True
3,47,TCGA-05-4405,TCGA-05-4405-01A,TCGA-LUAD,Illumina Human Methylation 450,female,True
4,48,TCGA-05-4410,TCGA-05-4410-01A,TCGA-LUAD,Illumina Human Methylation 450,male,True


In [83]:
# =======================================================
# Assess within-project sex representation
# =======================================================

MIN_SEX_GROUP_SAMPLES = 20

binary_sex_metadata = (
    methylation_hm450_sex_metadata[
        methylation_hm450_sex_metadata[
            "binary_sex_available"
        ]
    ]
    .copy()
)

project_sex_counts = (
    binary_sex_metadata
    .groupby(
        [
            "project_id",
            "sex_at_birth",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=[
            "female",
            "male",
        ],
        fill_value=0,
    )
    .reset_index()
)

project_sex_counts[
    "binary_total"
] = (
    project_sex_counts["female"]
    + project_sex_counts["male"]
)

project_sex_counts[
    "minimum_sex_group_count"
] = project_sex_counts[
    [
        "female",
        "male",
    ]
].min(axis=1)

project_sex_counts[
    "both_sexes_present"
] = (
    project_sex_counts[
        "minimum_sex_group_count"
    ]
    > 0
)

project_sex_counts[
    "eligible_for_sex_sensitivity"
] = (
    project_sex_counts[
        "minimum_sex_group_count"
    ]
    >= MIN_SEX_GROUP_SAMPLES
)

eligible_sex_sensitivity_projects = (
    project_sex_counts.loc[
        project_sex_counts[
            "eligible_for_sex_sensitivity"
        ],
        "project_id",
    ]
    .to_list()
)

print(
    "HM450 projects represented: "
    f"{project_sex_counts.shape[0]}"
)

print(
    "Projects with both sexes present: "
    f"{project_sex_counts['both_sexes_present'].sum()}"
)

print(
    "Projects eligible for sex sensitivity: "
    f"{project_sex_counts['eligible_for_sex_sensitivity'].sum()}"
)

print(
    "Samples in eligible projects: "
    f"{binary_sex_metadata['project_id'].isin(eligible_sex_sensitivity_projects).sum():,}"
)

display(
    project_sex_counts
    .sort_values(
        [
            "eligible_for_sex_sensitivity",
            "minimum_sex_group_count",
            "binary_total",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

HM450 projects represented: 33
Projects with both sexes present: 27
Projects eligible for sex sensitivity: 24
Samples in eligible projects: 5,997


sex_at_birth,project_id,female,male,binary_total,minimum_sex_group_count,both_sexes_present,eligible_for_sex_sensitivity
0,TCGA-LGG,229,283,512,229,True,True
1,TCGA-LUAD,244,209,453,209,True,True
2,TCGA-HNSC,136,384,520,136,True,True
3,TCGA-THCA,369,136,505,136,True,True
4,TCGA-COAD,135,156,291,135,True,True
5,TCGA-STAD,127,246,373,127,True,True
6,TCGA-LIHC,121,250,371,121,True,True
7,TCGA-SARC,141,118,259,118,True,True
8,TCGA-KIRC,113,205,318,113,True,True
9,TCGA-BLCA,106,299,405,106,True,True


In [84]:
# =======================================================
# Inspect methylation candidate score labels
# =======================================================

candidate_label_inventory = []

for object_name, analysis_object in list(
    globals().items()
):
    if object_name.startswith("_"):
        continue

    if (
        "methylation" not in object_name.lower()
        or "candidate" not in object_name.lower()
        or callable(analysis_object)
    ):
        continue

    object_shape = getattr(
        analysis_object,
        "shape",
        None,
    )

    object_length = None
    object_preview = None

    if isinstance(
        analysis_object,
        (list, tuple, pd.Index),
    ):
        object_length = len(analysis_object)
        object_preview = list(
            analysis_object[:10]
        )

    elif isinstance(
        analysis_object,
        np.ndarray,
    ):
        object_length = len(analysis_object)
        object_preview = (
            analysis_object[:10].tolist()
            if analysis_object.ndim == 1
            else None
        )

    elif isinstance(
        analysis_object,
        pd.Series,
    ):
        object_length = len(analysis_object)
        object_preview = (
            analysis_object
            .head(10)
            .to_list()
        )

    candidate_label_inventory.append(
        {
            "object_name": object_name,
            "object_type": type(
                analysis_object
            ).__name__,
            "shape": (
                object_shape
                if isinstance(object_shape, tuple)
                else None
            ),
            "length": object_length,
            "preview": object_preview,
        }
    )

candidate_label_inventory = (
    pd.DataFrame(
        candidate_label_inventory
    )
    .sort_values("object_name")
    .reset_index(drop=True)
)

display(candidate_label_inventory)

display(
    cross_omic_candidate_pairs[
        [
            "rna_component",
            "methylation_component",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

,object_name,object_type,shape,length,preview
0,candidate_methylation_components,list,None,12.0,"[METH_IC152, METH_IC232, METH_IC169, METH_IC12..."
1,candidate_methylation_probe_tables,list,None,24.0,"[[component, loading_direction, directional_ra..."
2,candidate_methylation_top_probes,DataFrame,"(1200, 22)",NaN,None
3,candidate_methylation_top_probes_annotated,DataFrame,"(1200, 41)",NaN,None
4,methylation_candidate_component_indices,ndarray,"(76,)",76.0,"[258, 128, 138, 219, 288, 249, 192, 23, 51, 129]"
5,methylation_candidate_component_names,list,None,76.0,"[METH_IC259, METH_IC129, METH_IC139, METH_IC22..."
6,methylation_candidate_local_index,dict,None,NaN,None
7,methylation_candidate_scores_hm450,ndarray,"(8345, 76)",8345.0,None
8,methylation_ica_candidate_components,DataFrame,"(76, 14)",NaN,None


,rna_component,methylation_component
0,RNA_IC158,METH_IC152
1,RNA_IC083,METH_IC232
2,RNA_IC184,METH_IC169
3,RNA_IC150,METH_IC128
4,RNA_IC169,METH_IC023
5,RNA_IC175,METH_IC013
6,RNA_IC050,METH_IC234
7,RNA_IC151,METH_IC050
8,RNA_IC001,METH_IC107
9,RNA_IC001,METH_IC241


In [85]:
# =======================================================
# Quantify within-project sex association
# =======================================================

methylation_candidate_score_table = pd.DataFrame(
    methylation_candidate_scores_hm450,
    columns=methylation_candidate_component_names,
)

methylation_candidate_score_table[
    "project_id"
] = methylation_hm450_sex_metadata[
    "project_id"
].to_numpy()

methylation_candidate_score_table[
    "sex_at_birth"
] = methylation_hm450_sex_metadata[
    "sex_at_birth"
].to_numpy()

sex_association_rows = []

for project_id in eligible_sex_sensitivity_projects:
    project_mask = (
        methylation_candidate_score_table[
            "project_id"
        ]
        == project_id
    )

    female_mask = (
        project_mask
        & (
            methylation_candidate_score_table[
                "sex_at_birth"
            ]
            == "female"
        )
    )

    male_mask = (
        project_mask
        & (
            methylation_candidate_score_table[
                "sex_at_birth"
            ]
            == "male"
        )
    )

    for component in methylation_candidate_component_names:
        female_scores = (
            methylation_candidate_score_table.loc[
                female_mask,
                component,
            ]
            .to_numpy(dtype=float)
        )

        male_scores = (
            methylation_candidate_score_table.loc[
                male_mask,
                component,
            ]
            .to_numpy(dtype=float)
        )

        n_female = female_scores.size
        n_male = male_scores.size

        female_variance = female_scores.var(
            ddof=1
        )

        male_variance = male_scores.var(
            ddof=1
        )

        pooled_variance = (
            (
                (n_female - 1)
                * female_variance
            )
            + (
                (n_male - 1)
                * male_variance
            )
        ) / (
            n_female
            + n_male
            - 2
        )

        if (
            not np.isfinite(pooled_variance)
            or pooled_variance <= 0
        ):
            continue

        cohen_d = (
            female_scores.mean()
            - male_scores.mean()
        ) / np.sqrt(pooled_variance)

        degrees_of_freedom = (
            n_female
            + n_male
            - 2
        )

        hedges_correction = (
            1
            - (
                3
                / (
                    4
                    * degrees_of_freedom
                    - 1
                )
            )
        )

        hedges_g = (
            hedges_correction
            * cohen_d
        )

        combined_scores = np.concatenate(
            [
                female_scores,
                male_scores,
            ]
        )

        grand_mean = combined_scores.mean()

        between_group_ss = (
            n_female
            * (
                female_scores.mean()
                - grand_mean
            ) ** 2
            + n_male
            * (
                male_scores.mean()
                - grand_mean
            ) ** 2
        )

        total_ss = np.sum(
            (
                combined_scores
                - grand_mean
            ) ** 2
        )

        eta_squared = (
            between_group_ss / total_ss
            if total_ss > 0
            else np.nan
        )

        sex_association_rows.append(
            {
                "project_id": project_id,
                "component": component,
                "n_female": n_female,
                "n_male": n_male,
                "female_mean": (
                    female_scores.mean()
                ),
                "male_mean": (
                    male_scores.mean()
                ),
                "hedges_g_female_minus_male": (
                    hedges_g
                ),
                "abs_hedges_g": abs(
                    hedges_g
                ),
                "eta_squared": eta_squared,
            }
        )

methylation_sex_project_associations = (
    pd.DataFrame(
        sex_association_rows
    )
)

methylation_sex_component_summary = (
    methylation_sex_project_associations
    .groupby("component")
    .agg(
        n_projects=(
            "project_id",
            "nunique",
        ),
        median_hedges_g=(
            "hedges_g_female_minus_male",
            "median",
        ),
        median_abs_hedges_g=(
            "abs_hedges_g",
            "median",
        ),
        q75_abs_hedges_g=(
            "abs_hedges_g",
            lambda values: values.quantile(
                0.75
            ),
        ),
        maximum_abs_hedges_g=(
            "abs_hedges_g",
            "max",
        ),
        median_eta_squared=(
            "eta_squared",
            "median",
        ),
        positive_project_fraction=(
            "hedges_g_female_minus_male",
            lambda values: (
                values > 0
            ).mean(),
        ),
    )
    .reset_index()
)

cross_omic_methylation_sex_summary = (
    methylation_sex_component_summary[
        methylation_sex_component_summary[
            "component"
        ].isin(
            candidate_methylation_components
        )
    ]
    .sort_values(
        "median_abs_hedges_g",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    "Project-component associations: "
    f"{methylation_sex_project_associations.shape}"
)

print(
    "Components summarized: "
    f"{methylation_sex_component_summary.shape[0]}"
)

print(
    "Cross-omic methylation components: "
    f"{cross_omic_methylation_sex_summary.shape[0]}"
)

display(
    cross_omic_methylation_sex_summary
)

Project-component associations: (1824, 9)
Components summarized: 76
Cross-omic methylation components: 12


,component,n_projects,median_hedges_g,median_abs_hedges_g,q75_abs_hedges_g,maximum_abs_hedges_g,median_eta_squared,positive_project_fraction
0,METH_IC152,24,1.612999,1.612999,2.425133,4.663262,0.399152,1.000000
1,METH_IC013,24,0.180758,0.216430,0.289967,0.467731,0.011617,0.625000
2,METH_IC232,24,-0.016590,0.206855,0.259546,0.422647,0.009727,0.500000
3,METH_IC241,24,-0.128821,0.199966,0.322180,0.410111,0.009467,0.250000
4,METH_IC234,24,0.099995,0.165162,0.258582,0.540167,0.006181,0.625000
5,METH_IC023,24,0.147437,0.164835,0.264704,0.650626,0.005663,0.708333
6,METH_IC169,24,0.060228,0.132809,0.258456,0.523523,0.004447,0.583333
7,METH_IC107,24,-0.040061,0.129036,0.191644,0.434205,0.003359,0.416667
8,METH_IC128,24,0.041435,0.117755,0.225571,0.451922,0.003335,0.625000
9,METH_IC109,24,-0.029779,0.110817,0.180200,0.491898,0.002770,0.375000


In [86]:
# =======================================================
# Prepare sex-sensitivity subset
# =======================================================

sex_sensitivity_mask = (
    methylation_hm450_sex_metadata["project_id"].isin(
        eligible_sex_sensitivity_projects
    )
    & methylation_hm450_sex_metadata["sex_at_birth"].isin(
        ["female", "male"]
    )
)

sex_sensitivity_metadata = (
    methylation_hm450_sex_metadata.loc[
        sex_sensitivity_mask,
        ["project_id", "sex_at_birth"],
    ]
    .reset_index(drop=True)
)

rna_candidate_scores_sex = (
    rna_candidate_scores_hm450[
        sex_sensitivity_mask
    ]
)

methylation_candidate_scores_sex = (
    methylation_candidate_scores_hm450[
        sex_sensitivity_mask
    ]
)

print(
    "Sex-sensitivity samples: "
    f"{sex_sensitivity_metadata.shape[0]:,}"
)
print(
    "RNA candidate scores: "
    f"{rna_candidate_scores_sex.shape}"
)
print(
    "Methylation candidate scores: "
    f"{methylation_candidate_scores_sex.shape}"
)

Sex-sensitivity samples: 5,997
RNA candidate scores: (5997, 42)
Methylation candidate scores: (5997, 76)


In [87]:
# =======================================================
# Residualize candidate scores on sex within project
# =======================================================

sex_group_keys = [
    sex_sensitivity_metadata["project_id"],
    sex_sensitivity_metadata["sex_at_birth"],
]

rna_candidate_scores_sex = pd.DataFrame(
    rna_candidate_scores_sex,
    columns=rna_candidate_component_names,
)

methylation_candidate_scores_sex = pd.DataFrame(
    methylation_candidate_scores_sex,
    columns=methylation_candidate_component_names,
)

rna_sex_residuals = (
    rna_candidate_scores_sex
    - rna_candidate_scores_sex
    .groupby(sex_group_keys)
    .transform("mean")
)

methylation_sex_residuals = (
    methylation_candidate_scores_sex
    - methylation_candidate_scores_sex
    .groupby(sex_group_keys)
    .transform("mean")
)

print(
    "RNA sex-adjusted scores: "
    f"{rna_sex_residuals.shape}"
)
print(
    "Methylation sex-adjusted scores: "
    f"{methylation_sex_residuals.shape}"
)

RNA sex-adjusted scores: (5997, 42)
Methylation sex-adjusted scores: (5997, 76)


In [88]:
# =======================================================
# Recompute sex-adjusted cross-omic correlations
# =======================================================

sex_adjusted_pair_rows = []

candidate_pairs = (
    cross_omic_candidate_pairs[
        [
            "rna_component",
            "methylation_component",
        ]
    ]
    .drop_duplicates()
)

for project_id in eligible_sex_sensitivity_projects:
    project_mask = (
        sex_sensitivity_metadata["project_id"]
        == project_id
    )

    for pair in candidate_pairs.itertuples(
        index=False
    ):
        correlation = np.corrcoef(
            rna_sex_residuals.loc[
                project_mask,
                pair.rna_component,
            ],
            methylation_sex_residuals.loc[
                project_mask,
                pair.methylation_component,
            ],
        )[0, 1]

        sex_adjusted_pair_rows.append(
            {
                "project_id": project_id,
                "rna_component": pair.rna_component,
                "methylation_component": (
                    pair.methylation_component
                ),
                "sex_adjusted_correlation": correlation,
            }
        )

sex_adjusted_project_correlations = pd.DataFrame(
    sex_adjusted_pair_rows
)

print(
    "Sex-adjusted project correlations: "
    f"{sex_adjusted_project_correlations.shape}"
)
print(
    "Projects represented: "
    f"{sex_adjusted_project_correlations['project_id'].nunique()}"
)
print(
    "Candidate pairs represented: "
    f"{candidate_pairs.shape[0]}"
)

Sex-adjusted project correlations: (336, 4)
Projects represented: 24
Candidate pairs represented: 14


In [89]:
# =======================================================
# Inspect baseline correlation columns
# =======================================================

print(
    cross_omic_candidate_project_correlations.columns.to_list()
)

display(
    cross_omic_candidate_project_correlations.head()
)

['candidate_pair', 'rna_component', 'methylation_component', 'association_direction', 'project_id', 'project_sample_count', 'project_correlation', 'aligned_project_correlation', 'same_direction_ge_0_10', 'same_direction_ge_0_20']


,candidate_pair,rna_component,methylation_component,association_direction,project_id,project_sample_count,project_correlation,aligned_project_correlation,same_direction_ge_0_10,same_direction_ge_0_20
0,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-ACC,79,-0.570736,0.570736,True,True
1,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-BLCA,405,-0.367996,0.367996,True,True
2,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-BRCA,777,-0.042101,0.042101,False,False
3,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-CESC,304,-0.051861,0.051861,False,False
4,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-CHOL,35,-0.545653,0.545653,True,True


In [90]:
# =======================================================
# Compare baseline and sex-adjusted correlations
# =======================================================

baseline_sex_correlations = (
    cross_omic_candidate_project_correlations.loc[
        cross_omic_candidate_project_correlations[
            "project_id"
        ].isin(eligible_sex_sensitivity_projects),
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "association_direction",
            "project_id",
            "project_correlation",
        ],
    ]
)

sex_sensitivity_pair_comparison = (
    baseline_sex_correlations
    .merge(
        sex_adjusted_project_correlations,
        on=[
            "project_id",
            "rna_component",
            "methylation_component",
        ],
        how="inner",
        validate="one_to_one",
    )
)

sex_sensitivity_pair_comparison[
    "correlation_change"
] = (
    sex_sensitivity_pair_comparison[
        "sex_adjusted_correlation"
    ]
    - sex_sensitivity_pair_comparison[
        "project_correlation"
    ]
)

print(
    "Baseline-adjusted comparisons: "
    f"{sex_sensitivity_pair_comparison.shape}"
)

display(
    sex_sensitivity_pair_comparison.head()
)

Baseline-adjusted comparisons: (336, 8)


,candidate_pair,rna_component,methylation_component,association_direction,project_id,project_correlation,sex_adjusted_correlation,correlation_change
0,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-ACC,-0.570736,-0.191556,0.379180
1,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-BLCA,-0.367996,-0.038660,0.329336
2,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-COAD,-0.031210,-0.045427,-0.014217
3,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-DLBC,-0.449053,-0.071322,0.377731
4,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,negative,TCGA-ESCA,-0.202920,-0.088804,0.114116


In [91]:
# =======================================================
# Summarize sex-adjusted pair associations
# =======================================================

direction_sign = (
    sex_sensitivity_pair_comparison[
        "association_direction"
    ]
    .map(
        {
            "positive": 1,
            "negative": -1,
        }
    )
)

sex_sensitivity_pair_comparison[
    "baseline_aligned"
] = (
    sex_sensitivity_pair_comparison[
        "project_correlation"
    ]
    * direction_sign
)

sex_sensitivity_pair_comparison[
    "sex_adjusted_aligned"
] = (
    sex_sensitivity_pair_comparison[
        "sex_adjusted_correlation"
    ]
    * direction_sign
)

sex_adjusted_pair_summary = (
    sex_sensitivity_pair_comparison
    .groupby(
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ]
    )
    .agg(
        n_projects=("project_id", "nunique"),
        median_baseline=("baseline_aligned", "median"),
        median_sex_adjusted=("sex_adjusted_aligned", "median"),
        adjusted_ge_0_10=(
            "sex_adjusted_aligned",
            lambda x: (x >= 0.10).mean(),
        ),
        adjusted_ge_0_20=(
            "sex_adjusted_aligned",
            lambda x: (x >= 0.20).mean(),
        ),
    )
    .reset_index()
)

sex_adjusted_pair_summary[
    "median_change"
] = (
    sex_adjusted_pair_summary["median_sex_adjusted"]
    - sex_adjusted_pair_summary["median_baseline"]
)

display(
    sex_adjusted_pair_summary
    .sort_values(
        "median_sex_adjusted",
        ascending=False,
    )
    .reset_index(drop=True)
)

,candidate_pair,rna_component,methylation_component,n_projects,median_baseline,median_sex_adjusted,adjusted_ge_0_10,adjusted_ge_0_20,median_change
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,24,0.280760,0.294488,0.833333,0.708333,0.013728
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,24,0.158485,0.149719,0.583333,0.416667,-0.008766
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,24,0.139989,0.141620,0.583333,0.250000,0.001631
3,CROSS_OMIC_PAIR_13,RNA_IC193,METH_IC033,24,0.129068,0.122890,0.541667,0.250000,-0.006179
4,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,24,0.130233,0.122005,0.541667,0.125000,-0.008228
5,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,24,0.107284,0.116651,0.541667,0.083333,0.009367
6,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,24,0.110080,0.109819,0.541667,0.208333,-0.000261
7,CROSS_OMIC_PAIR_12,RNA_IC184,METH_IC128,24,0.095765,0.089580,0.458333,0.125000,-0.006185
8,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,24,0.613628,0.088816,0.458333,0.041667,-0.524812
9,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,24,0.081394,0.088709,0.500000,0.166667,0.007316


In [92]:
# =======================================================
# Consolidate sex-sensitivity metrics
# =======================================================

pair_sex_sensitivity_summary = (
    sex_adjusted_pair_summary
    .merge(
        cross_omic_methylation_sex_summary[
            [
                "component",
                "median_abs_hedges_g",
                "median_eta_squared",
            ]
        ],
        left_on="methylation_component",
        right_on="component",
        how="left",
        validate="many_to_one",
    )
    .drop(columns="component")
)

pair_sex_sensitivity_summary[
    "median_retained_fraction"
] = (
    pair_sex_sensitivity_summary[
        "median_sex_adjusted"
    ]
    / pair_sex_sensitivity_summary[
        "median_baseline"
    ]
)

display(
    pair_sex_sensitivity_summary
    .sort_values(
        "median_retained_fraction"
    )
    .reset_index(drop=True)
)

,candidate_pair,rna_component,methylation_component,n_projects,median_baseline,median_sex_adjusted,adjusted_ge_0_10,adjusted_ge_0_20,median_change,median_abs_hedges_g,median_eta_squared,median_retained_fraction
0,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,24,0.613628,0.088816,0.458333,0.041667,-0.524812,1.612999,0.399152,0.144739
1,CROSS_OMIC_PAIR_14,RNA_IC158,METH_IC013,24,0.090308,0.065612,0.375000,0.083333,-0.024695,0.216430,0.011617,0.726542
2,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,24,0.093996,0.087733,0.458333,0.208333,-0.006263,0.110817,0.002770,0.933371
3,CROSS_OMIC_PAIR_12,RNA_IC184,METH_IC128,24,0.095765,0.089580,0.458333,0.125000,-0.006185,0.117755,0.003335,0.935411
4,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,24,0.130233,0.122005,0.541667,0.125000,-0.008228,0.164835,0.005663,0.936823
5,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,24,0.158485,0.149719,0.583333,0.416667,-0.008766,0.132809,0.004447,0.944686
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,24,0.091609,0.086916,0.500000,0.083333,-0.004693,0.109406,0.002620,0.948774
7,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,24,0.080662,0.076692,0.458333,0.125000,-0.003969,0.129036,0.003359,0.950788
8,CROSS_OMIC_PAIR_13,RNA_IC193,METH_IC033,24,0.129068,0.122890,0.541667,0.250000,-0.006179,0.098777,0.002158,0.952130
9,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,24,0.110080,0.109819,0.541667,0.208333,-0.000261,0.165162,0.006181,0.997630


In [93]:
# =======================================================
# Record sex-sensitivity interpretation
# =======================================================

pair_sex_sensitivity_summary[
    "sex_sensitivity_status"
] = "retained_after_sex_sensitivity"

pair_sex_sensitivity_summary.loc[
    pair_sex_sensitivity_summary["candidate_pair"]
    == "CROSS_OMIC_PAIR_01",
    "sex_sensitivity_status",
] = "excluded_sex_at_birth_sensitive"

pair_sex_sensitivity_summary[
    "sex_sensitivity_reason"
] = ""

pair_sex_sensitivity_summary.loc[
    pair_sex_sensitivity_summary["candidate_pair"]
    == "CROSS_OMIC_PAIR_01",
    "sex_sensitivity_reason",
] = (
    "Strong methylation-component association with sex_at_birth "
    "and marked attenuation after within-project adjustment."
)

display(
    pair_sex_sensitivity_summary[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "median_baseline",
            "median_sex_adjusted",
            "median_retained_fraction",
            "sex_sensitivity_status",
        ]
    ]
)

,candidate_pair,rna_component,methylation_component,median_baseline,median_sex_adjusted,median_retained_fraction,sex_sensitivity_status
0,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,0.613628,0.088816,0.144739,excluded_sex_at_birth_sensitive
1,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,0.280760,0.294488,1.048894,retained_after_sex_sensitivity
2,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,0.158485,0.149719,0.944686,retained_after_sex_sensitivity
3,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,0.139989,0.141620,1.011652,retained_after_sex_sensitivity
4,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,0.130233,0.122005,0.936823,retained_after_sex_sensitivity
5,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,0.081394,0.088709,1.089879,retained_after_sex_sensitivity
6,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,0.110080,0.109819,0.997630,retained_after_sex_sensitivity
7,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,0.091609,0.086916,0.948774,retained_after_sex_sensitivity
8,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,0.080662,0.076692,0.950788,retained_after_sex_sensitivity
9,CROSS_OMIC_PAIR_10,RNA_IC001,METH_IC241,0.107284,0.116651,1.087310,retained_after_sex_sensitivity


In [94]:
# =======================================================
# Finalize sex-sensitive candidate-pair set
# =======================================================

sex_sensitivity_columns = [
    "candidate_pair",
    "rna_component",
    "methylation_component",
    "median_sex_adjusted",
    "median_retained_fraction",
    "sex_sensitivity_status",
    "sex_sensitivity_reason",
]

cross_omic_candidates_final = (
    cross_omic_candidate_pairs
    .merge(
        pair_sex_sensitivity_summary[
            sex_sensitivity_columns
        ],
        on=[
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_omic_candidates_retained = (
    cross_omic_candidates_final[
        cross_omic_candidates_final[
            "sex_sensitivity_status"
        ]
        == "retained_after_sex_sensitivity"
    ]
    .reset_index(drop=True)
)

print(
    "Candidate pairs before sensitivity: "
    f"{cross_omic_candidates_final.shape[0]}"
)
print(
    "Candidate pairs retained: "
    f"{cross_omic_candidates_retained.shape[0]}"
)
print(
    "Candidate pairs excluded: "
    f"{cross_omic_candidates_final.shape[0] - cross_omic_candidates_retained.shape[0]}"
)

Candidate pairs before sensitivity: 14
Candidate pairs retained: 13
Candidate pairs excluded: 1


## Complementary NMF sensitivity analysis

Non-negative matrix factorization is used as a secondary,  targeted sensitivity analysis of the retained ICA-derived candidate-program space.

For each modality, the NMF feature space is restricted to the union of the strongest positive and negative feature loadings from the unique ICA components represented among the retained cross-omic candidate pairs. The NMF rank is fixed to the corresponding number of unique retained ICA components.

Because the lineage-centered RNA-seq and HM450 matrices contain signed values, each selected feature is translated by its observed minimum before factorization. This preserves sample-to-sample differences within each feature while satisfying the non-negativity constraint.

This analysis evaluates whether the retained ICA candidate structure admits a comparable non-negative representation. Because feature selection is conditioned on the ICA results and translation alters the factorization geometry, NMF concordance is not treated as
independent validation and cannot increase the evidence status of a candidate program.

Factor stability across initializations, sensitivity to factorization rank, and broader methodological robustness remain reserved for notebook 206.

In [97]:
# =======================================================
# Prepare targeted NMF inputs
# =======================================================

NMF_FEATURES_PER_DIRECTION = 50

rna_nmf_component_names = (
    cross_omic_candidates_retained[
        "rna_component"
    ]
    .drop_duplicates()
    .to_list()
)

methylation_nmf_component_names = (
    cross_omic_candidates_retained[
        "methylation_component"
    ]
    .drop_duplicates()
    .to_list()
)

rna_nmf_component_indices = np.array(
    [
        rna_component_to_index[component]
        for component in rna_nmf_component_names
    ],
    dtype=np.int32,
)

methylation_nmf_component_indices = np.array(
    [
        methylation_component_to_index[component]
        for component in methylation_nmf_component_names
    ],
    dtype=np.int32,
)


def select_loading_tail_union(
    loading_matrix,
    component_indices,
    features_per_direction,
):
    selected_indices = []

    for component_index in component_indices:
        loading_order = np.argsort(
            loading_matrix[component_index]
        )

        selected_indices.extend(
            loading_order[:features_per_direction]
        )
        selected_indices.extend(
            loading_order[-features_per_direction:]
        )

    return np.unique(
        selected_indices
    ).astype(np.int32)


rna_nmf_feature_indices = select_loading_tail_union(
    rna_ica_gene_loadings_oriented,
    rna_nmf_component_indices,
    NMF_FEATURES_PER_DIRECTION,
)

methylation_nmf_feature_indices = (
    select_loading_tail_union(
        methylation_hm450_ica_probe_loadings_oriented,
        methylation_nmf_component_indices,
        NMF_FEATURES_PER_DIRECTION,
    )
)

rna_nmf_input = np.array(
    rna_ica_input[:, rna_nmf_feature_indices],
    dtype=np.float32,
    order="C",
    copy=True,
)

methylation_nmf_input = np.array(
    methylation_hm450_ica_input[
        :,
        methylation_nmf_feature_indices,
    ],
    dtype=np.float32,
    order="C",
    copy=True,
)

rna_nmf_input -= rna_nmf_input.min(
    axis=0
)

methylation_nmf_input -= (
    methylation_nmf_input.min(
        axis=0
    )
)

RNA_NMF_COMPONENTS = len(
    rna_nmf_component_names
)

METHYLATION_NMF_COMPONENTS = len(
    methylation_nmf_component_names
)

display(
    pd.Series(
        {
            "retained_rna_ica_components": (
                RNA_NMF_COMPONENTS
            ),
            "retained_methylation_ica_components": (
                METHYLATION_NMF_COMPONENTS
            ),
            "targeted_rna_genes": len(
                rna_nmf_feature_indices
            ),
            "targeted_methylation_probes": len(
                methylation_nmf_feature_indices
            ),
            "rna_nmf_input_shape": (
                rna_nmf_input.shape
            ),
            "methylation_nmf_input_shape": (
                methylation_nmf_input.shape
            ),
            "rna_nmf_minimum": float(
                rna_nmf_input.min()
            ),
            "methylation_nmf_minimum": float(
                methylation_nmf_input.min()
            ),
        }
    )
)

retained_rna_ica_components                     10
retained_methylation_ica_components             11
targeted_rna_genes                             761
targeted_methylation_probes                    828
rna_nmf_input_shape                    (9965, 761)
methylation_nmf_input_shape            (8345, 828)
rna_nmf_minimum                                0.0
methylation_nmf_minimum                        0.0
dtype: object

In [99]:
# =======================================================
# Fit targeted NMF models
# =======================================================

from sklearn.decomposition import NMF


NMF_MAX_ITER = 2_000

rna_nmf = NMF(
    n_components=RNA_NMF_COMPONENTS,
    init="nndsvda",
    solver="cd",
    tol=1e-4,
    max_iter=NMF_MAX_ITER,
    shuffle=True,
    random_state=RANDOM_STATE,
)

methylation_nmf = NMF(
    n_components=METHYLATION_NMF_COMPONENTS,
    init="nndsvda",
    solver="cd",
    tol=1e-4,
    max_iter=NMF_MAX_ITER,
    shuffle=True,
    random_state=RANDOM_STATE,
)

rna_nmf_scores = rna_nmf.fit_transform(
    rna_nmf_input
).astype(np.float32)

methylation_nmf_scores = (
    methylation_nmf.fit_transform(
        methylation_nmf_input
    )
    .astype(np.float32)
)

rna_nmf_feature_loadings = (
    rna_nmf.components_.astype(np.float32)
)

methylation_nmf_feature_loadings = (
    methylation_nmf.components_.astype(np.float32)
)

display(
    {
        "rna_score_shape": rna_nmf_scores.shape,
        "rna_loading_shape": (
            rna_nmf_feature_loadings.shape
        ),
        "rna_iterations": int(
            rna_nmf.n_iter_
        ),
        "rna_reached_iteration_limit": bool(
            rna_nmf.n_iter_ >= NMF_MAX_ITER
        ),
        "rna_relative_reconstruction_error": float(
            rna_nmf.reconstruction_err_
            / np.linalg.norm(rna_nmf_input)
        ),
        "methylation_score_shape": (
            methylation_nmf_scores.shape
        ),
        "methylation_loading_shape": (
            methylation_nmf_feature_loadings.shape
        ),
        "methylation_iterations": int(
            methylation_nmf.n_iter_
        ),
        "methylation_reached_iteration_limit": bool(
            methylation_nmf.n_iter_
            >= NMF_MAX_ITER
        ),
        "methylation_relative_reconstruction_error": float(
            methylation_nmf.reconstruction_err_
            / np.linalg.norm(
                methylation_nmf_input
            )
        ),
    }
)

{'rna_score_shape': (9965, 10),
 'rna_loading_shape': (10, 761),
 'rna_iterations': 95,
 'rna_reached_iteration_limit': False,
 'rna_relative_reconstruction_error': 0.20764853060245514,
 'methylation_score_shape': (8345, 11),
 'methylation_loading_shape': (11, 828),
 'methylation_iterations': 154,
 'methylation_reached_iteration_limit': False,
 'methylation_relative_reconstruction_error': 0.21140623092651367}

In [100]:
# =======================================================
# Quantify lineage-aware ICA-NMF concordance
# =======================================================

MIN_NMF_CONCORDANCE_PROJECT_SAMPLES = (
    MIN_CROSS_OMIC_PROJECT_SAMPLES
)


def median_project_absolute_correlations(
    ica_scores,
    nmf_scores,
    project_labels,
):
    project_correlations = []
    eligible_projects = []

    for project_id in np.unique(project_labels):
        project_mask = project_labels == project_id

        if (
            project_mask.sum()
            < MIN_NMF_CONCORDANCE_PROJECT_SAMPLES
        ):
            continue

        correlation_matrix = np.corrcoef(
            ica_scores[project_mask].T,
            nmf_scores[project_mask].T,
        )

        n_ica_components = ica_scores.shape[1]

        project_correlations.append(
            np.abs(
                correlation_matrix[
                    :n_ica_components,
                    n_ica_components:,
                ]
            )
        )
        eligible_projects.append(project_id)

    return (
        np.nanmedian(
            np.stack(project_correlations),
            axis=0,
        ),
        np.array(eligible_projects),
    )


rna_nmf_concordance, rna_nmf_project_ids = (
    median_project_absolute_correlations(
        rna_ica_scores_oriented[
            :,
            rna_nmf_component_indices,
        ],
        rna_nmf_scores,
        sample_metadata["project_id"].to_numpy(),
    )
)

methylation_nmf_concordance, methylation_nmf_project_ids = (
    median_project_absolute_correlations(
        methylation_hm450_ica_scores_oriented[
            :,
            methylation_nmf_component_indices,
        ],
        methylation_nmf_scores,
        methylation_hm450_project_labels,
    )
)

display(
    {
        "rna_concordance_shape": (
            rna_nmf_concordance.shape
        ),
        "rna_projects": len(
            rna_nmf_project_ids
        ),
        "methylation_concordance_shape": (
            methylation_nmf_concordance.shape
        ),
        "methylation_projects": len(
            methylation_nmf_project_ids
        ),
    }
)

{'rna_concordance_shape': (10, 10),
 'rna_projects': 33,
 'methylation_concordance_shape': (11, 11),
 'methylation_projects': 32}

In [101]:
# =======================================================
# Match ICA components to NMF factors
# =======================================================

from scipy.optimize import linear_sum_assignment


def match_ica_to_nmf(
    concordance_matrix,
    ica_component_names,
    nmf_prefix,
):
    ica_indices, nmf_indices = (
        linear_sum_assignment(
            -concordance_matrix
        )
    )

    return pd.DataFrame(
        {
            "ica_component": [
                ica_component_names[index]
                for index in ica_indices
            ],
            "nmf_factor": [
                f"{nmf_prefix}{index + 1:03d}"
                for index in nmf_indices
            ],
            "nmf_factor_index": nmf_indices,
            "assigned_median_abs_correlation": (
                concordance_matrix[
                    ica_indices,
                    nmf_indices,
                ]
            ),
            "row_best_median_abs_correlation": (
                concordance_matrix.max(axis=1)[
                    ica_indices
                ]
            ),
        }
    )


rna_ica_nmf_matches = match_ica_to_nmf(
    rna_nmf_concordance,
    rna_nmf_component_names,
    "RNA_NMF",
)

methylation_ica_nmf_matches = match_ica_to_nmf(
    methylation_nmf_concordance,
    methylation_nmf_component_names,
    "METH_NMF",
)

rna_ica_nmf_matches["modality"] = "RNA"
methylation_ica_nmf_matches["modality"] = (
    "methylation"
)

ica_nmf_matches = pd.concat(
    [
        rna_ica_nmf_matches,
        methylation_ica_nmf_matches,
    ],
    ignore_index=True,
)

display(
    ica_nmf_matches[
        [
            "modality",
            "ica_component",
            "nmf_factor",
            "assigned_median_abs_correlation",
            "row_best_median_abs_correlation",
        ]
    ]
    .sort_values(
        [
            "modality",
            "assigned_median_abs_correlation",
        ],
        ascending=[True, False],
    )
)

,modality,ica_component,nmf_factor,assigned_median_abs_correlation,row_best_median_abs_correlation
9,RNA,RNA_IC158,RNA_NMF003,0.744618,0.744618
2,RNA,RNA_IC150,RNA_NMF009,0.316420,0.316420
4,RNA,RNA_IC175,RNA_NMF008,0.179778,0.179778
8,RNA,RNA_IC193,RNA_NMF006,0.175624,0.175624
6,RNA,RNA_IC151,RNA_NMF002,0.144965,0.144965
0,RNA,RNA_IC083,RNA_NMF010,0.121404,0.121404
1,RNA,RNA_IC184,RNA_NMF007,0.119982,0.126391
5,RNA,RNA_IC050,RNA_NMF004,0.093799,0.117112
3,RNA,RNA_IC169,RNA_NMF001,0.068668,0.095154
7,RNA,RNA_IC001,RNA_NMF005,0.066818,0.103166


In [ ]:
# =======================================================
# Summarize pair-level ICA-NMF concordance
# =======================================================

rna_nmf_match_summary = (
    rna_ica_nmf_matches[
        [
            "ica_component",
            "nmf_factor",
            "assigned_median_abs_correlation",
        ]
    ]
    .rename(
        columns={
            "ica_component": "rna_component",
            "nmf_factor": "rna_nmf_factor",
            "assigned_median_abs_correlation": (
                "rna_nmf_concordance"
            ),
        }
    )
)

methylation_nmf_match_summary = (
    methylation_ica_nmf_matches[
        [
            "ica_component",
            "nmf_factor",
            "assigned_median_abs_correlation",
        ]
    ]
    .rename(
        columns={
            "ica_component": "methylation_component",
            "nmf_factor": "methylation_nmf_factor",
            "assigned_median_abs_correlation": (
                "methylation_nmf_concordance"
            ),
        }
    )
)

cross_omic_candidates_nmf = (
    cross_omic_candidates_retained
    .merge(
        rna_nmf_match_summary,
        on="rna_component",
        how="left",
        validate="many_to_one",
    )
    .merge(
        methylation_nmf_match_summary,
        on="methylation_component",
        how="left",
        validate="many_to_one",
    )
)

cross_omic_candidates_nmf[
    "minimum_nmf_concordance"
] = cross_omic_candidates_nmf[
    [
        "rna_nmf_concordance",
        "methylation_nmf_concordance",
    ]
].min(axis=1)

cross_omic_candidates_nmf[
    "mean_nmf_concordance"
] = cross_omic_candidates_nmf[
    [
        "rna_nmf_concordance",
        "methylation_nmf_concordance",
    ]
].mean(axis=1)

display(
    cross_omic_candidates_nmf[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "rna_nmf_factor",
            "methylation_nmf_factor",
            "rna_nmf_concordance",
            "methylation_nmf_concordance",
            "minimum_nmf_concordance",
            "mean_nmf_concordance",
        ]
    ]
    .sort_values(
        "mean_nmf_concordance",
        ascending=False,
    )
)

,candidate_pair,rna_component,methylation_component,rna_nmf_factor,methylation_nmf_factor,rna_nmf_concordance,methylation_nmf_concordance,minimum_nmf_concordance,mean_nmf_concordance
12,CROSS_OMIC_PAIR_14,RNA_IC158,METH_IC013,RNA_NMF003,METH_NMF007,0.744618,0.079366,0.079366,0.411992
2,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,RNA_NMF009,METH_NMF008,0.316420,0.372153,0.316420,0.344287
10,CROSS_OMIC_PAIR_12,RNA_IC184,METH_IC128,RNA_NMF007,METH_NMF008,0.119982,0.372153,0.119982,0.246068
11,CROSS_OMIC_PAIR_13,RNA_IC193,METH_IC033,RNA_NMF006,METH_NMF005,0.175624,0.243880,0.175624,0.209752
9,CROSS_OMIC_PAIR_11,RNA_IC193,METH_IC109,RNA_NMF006,METH_NMF009,0.175624,0.177229,0.175624,0.176426
4,CROSS_OMIC_PAIR_06,RNA_IC175,METH_IC013,RNA_NMF008,METH_NMF007,0.179778,0.079366,0.079366,0.129572
6,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,RNA_NMF002,METH_NMF004,0.144965,0.089150,0.089150,0.117057
1,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,RNA_NMF007,METH_NMF003,0.119982,0.103034,0.103034,0.111508
0,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,RNA_NMF010,METH_NMF011,0.121404,0.072895,0.072895,0.097149
3,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,RNA_NMF001,METH_NMF001,0.068668,0.106897,0.068668,0.087782


In [107]:
# =======================================================
# Compute matched-sample baseline correlations
# =======================================================

matched_baseline_rows = []

for project_id in eligible_sex_sensitivity_projects:
    project_mask = (
        sex_sensitivity_metadata["project_id"]
        .eq(project_id)
        .to_numpy()
    )

    for pair in candidate_pairs.itertuples(
        index=False
    ):
        matched_baseline_rows.append(
            {
                "project_id": project_id,
                "rna_component": pair.rna_component,
                "methylation_component": (
                    pair.methylation_component
                ),
                "matched_baseline_correlation": (
                    np.corrcoef(
                        rna_candidate_scores_sex.loc[
                            project_mask,
                            pair.rna_component,
                        ],
                        methylation_candidate_scores_sex.loc[
                            project_mask,
                            pair.methylation_component,
                        ],
                    )[0, 1]
                ),
            }
        )

matched_baseline_project_correlations = (
    pd.DataFrame(
        matched_baseline_rows
    )
)

print(
    "Matched baseline correlations: "
    f"{matched_baseline_project_correlations.shape}"
)

print(
    "Projects: "
    f"{matched_baseline_project_correlations['project_id'].nunique()}"
)

print(
    "Candidate pairs: "
    f"{matched_baseline_project_correlations[['rna_component', 'methylation_component']].drop_duplicates().shape[0]}"
)

Matched baseline correlations: (336, 4)
Projects: 24
Candidate pairs: 14


In [109]:
# =======================================================
# Rebuild matched sex-sensitivity comparison
# =======================================================

candidate_pair_lookup = (
    cross_omic_candidates_final[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ]
    ]
    .drop_duplicates()
)

sex_adjusted_project_correlations_matched = (
    sex_adjusted_project_correlations.rename(
        columns={
            "adjusted_correlation": (
                "sex_adjusted_correlation"
            ),
            "correlation": (
                "sex_adjusted_correlation"
            ),
        }
    )
)

sex_sensitivity_pair_comparison = (
    matched_baseline_project_correlations
    .merge(
        sex_adjusted_project_correlations_matched,
        on=[
            "project_id",
            "rna_component",
            "methylation_component",
        ],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        candidate_pair_lookup,
        on=[
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="many_to_one",
    )
)

sex_sensitivity_pair_comparison[
    "matched_baseline_abs_correlation"
] = sex_sensitivity_pair_comparison[
    "matched_baseline_correlation"
].abs()

sex_sensitivity_pair_comparison[
    "sex_adjusted_abs_correlation"
] = sex_sensitivity_pair_comparison[
    "sex_adjusted_correlation"
].abs()

sex_adjusted_pair_summary = (
    sex_sensitivity_pair_comparison
    .groupby(
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ],
        as_index=False,
    )
    .agg(
        projects=("project_id", "nunique"),
        matched_baseline_median_abs_correlation=(
            "matched_baseline_abs_correlation",
            "median",
        ),
        sex_adjusted_median_abs_correlation=(
            "sex_adjusted_abs_correlation",
            "median",
        ),
    )
)

sex_adjusted_pair_summary[
    "retained_correlation_fraction"
] = (
    sex_adjusted_pair_summary[
        "sex_adjusted_median_abs_correlation"
    ]
    / sex_adjusted_pair_summary[
        "matched_baseline_median_abs_correlation"
    ]
)

sex_adjusted_pair_summary[
    "absolute_correlation_change"
] = (
    sex_adjusted_pair_summary[
        "sex_adjusted_median_abs_correlation"
    ]
    - sex_adjusted_pair_summary[
        "matched_baseline_median_abs_correlation"
    ]
)

display(
    sex_adjusted_pair_summary.sort_values(
        "matched_baseline_median_abs_correlation",
        ascending=False,
    )
)

,candidate_pair,rna_component,methylation_component,projects,matched_baseline_median_abs_correlation,sex_adjusted_median_abs_correlation,retained_correlation_fraction,absolute_correlation_change
0,CROSS_OMIC_PAIR_01,RNA_IC158,METH_IC152,24,0.612484,0.102236,0.166921,-0.510248
1,CROSS_OMIC_PAIR_02,RNA_IC083,METH_IC232,24,0.280760,0.294488,1.048894,0.013728
3,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,24,0.179260,0.175568,0.979402,-0.003692
2,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,24,0.159571,0.149719,0.938255,-0.009853
4,CROSS_OMIC_PAIR_05,RNA_IC169,METH_IC023,24,0.134911,0.132311,0.980722,-0.002601
12,CROSS_OMIC_PAIR_13,RNA_IC193,METH_IC033,24,0.128530,0.122890,0.956119,-0.005640
8,CROSS_OMIC_PAIR_09,RNA_IC001,METH_IC107,24,0.126050,0.122000,0.967870,-0.004050
6,CROSS_OMIC_PAIR_07,RNA_IC050,METH_IC234,24,0.118829,0.111113,0.935064,-0.007716
13,CROSS_OMIC_PAIR_14,RNA_IC158,METH_IC013,24,0.108812,0.079045,0.726441,-0.029766
7,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,24,0.107550,0.106589,0.991059,-0.000962


In [ ]:
# =======================================================
# Finalize matched sex-sensitivity interpretation
# =======================================================

pair_sex_sensitivity_summary = (
    sex_adjusted_pair_summary
    .rename(
        columns={
            "projects": "matched_project_count",
            "matched_baseline_median_abs_correlation": (
                "median_baseline"
            ),
            "sex_adjusted_median_abs_correlation": (
                "median_sex_adjusted"
            ),
            "retained_correlation_fraction": (
                "median_retained_fraction"
            ),
            "absolute_correlation_change": (
                "median_absolute_correlation_change"
            ),
        }
    )
    .copy()
)

pair_sex_sensitivity_summary[
    "sex_sensitivity_status"
] = "retained_after_sex_sensitivity"

excluded_pair_mask = (
    pair_sex_sensitivity_summary["candidate_pair"]
    == "CROSS_OMIC_PAIR_01"
)

pair_sex_sensitivity_summary.loc[
    excluded_pair_mask,
    "sex_sensitivity_status",
] = "excluded_sex_at_birth_sensitive"

pair_sex_sensitivity_summary[
    "sex_sensitivity_reason"
] = ""

pair_sex_sensitivity_summary.loc[
    excluded_pair_mask,
    "sex_sensitivity_reason",
] = (
    "Strong methylation-component association with "
    "sex_at_birth and marked attenuation in the "
    "matched-sample within-project sensitivity analysis."
)

pair_sex_sensitivity_summary[
    "sex_sensitivity_analysis_scope"
] = "matched_binary_sex_subset_5997_samples"

sex_sensitivity_columns = [
    "candidate_pair",
    "rna_component",
    "methylation_component",
    "matched_project_count",
    "median_baseline",
    "median_sex_adjusted",
    "median_retained_fraction",
    "median_absolute_correlation_change",
    "sex_sensitivity_status",
    "sex_sensitivity_reason",
    "sex_sensitivity_analysis_scope",
]


In [ ]:
# =======================================================
# Finalize cross-omic candidate tables
# =======================================================

sex_result_columns = sex_sensitivity_columns[3:]

cross_omic_candidates_final = (
    cross_omic_candidates_final
    .drop(
        columns=sex_result_columns,
        errors="ignore",
    )
    .merge(
        pair_sex_sensitivity_summary[
            sex_sensitivity_columns
        ],
        on=[
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_omic_candidates_retained = (
    cross_omic_candidates_nmf
    .drop(
        columns=sex_result_columns,
        errors="ignore",
    )
    .merge(
        pair_sex_sensitivity_summary[
            sex_sensitivity_columns
        ],
        on=[
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="one_to_one",
    )
    .query(
        "sex_sensitivity_status == "
        "'retained_after_sex_sensitivity'"
    )
    .reset_index(drop=True)
)

In [117]:
# =======================================================
# Assemble authoritative candidate-pair catalog
# =======================================================

nmf_catalog_columns = [
    "rna_nmf_factor",
    "methylation_nmf_factor",
    "rna_nmf_concordance",
    "methylation_nmf_concordance",
    "minimum_nmf_concordance",
    "mean_nmf_concordance",
    "nmf_analysis_scope",
    "nmf_candidate_status",
    "nmf_interpretation",
]

cross_omic_candidate_catalog = (
    cross_omic_candidates_final
    .drop(
        columns=nmf_catalog_columns,
        errors="ignore",
    )
    .merge(
        cross_omic_candidates_nmf[
            [
                "candidate_pair",
                *nmf_catalog_columns,
            ]
        ],
        on="candidate_pair",
        how="left",
        validate="one_to_one",
    )
)

nmf_not_evaluated_mask = (
    cross_omic_candidate_catalog[
        "nmf_candidate_status"
    ].isna()
)

cross_omic_candidate_catalog.loc[
    nmf_not_evaluated_mask,
    "nmf_analysis_scope",
] = "not_evaluated_after_sex_sensitivity_exclusion"

cross_omic_candidate_catalog.loc[
    nmf_not_evaluated_mask,
    "nmf_candidate_status",
] = "not_evaluated"

cross_omic_candidate_catalog.loc[
    nmf_not_evaluated_mask,
    "nmf_interpretation",
] = (
    "Targeted NMF was not evaluated because the pair "
    "was excluded by the sex-at-birth sensitivity analysis."
)

In [118]:
# =======================================================
# Define tumor-program artifact paths
# =======================================================

PROGRAM_DISCOVERY_DIR = Paths.tumor_programs

PROGRAM_DISCOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CROSS_OMIC_CANDIDATE_CATALOG_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_candidate_pair_catalog.csv"
)

RNA_CANDIDATE_SCORES_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_rna_ica_candidate_scores.csv"
)

RNA_CANDIDATE_LOADINGS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_rna_ica_candidate_gene_loadings.csv"
)

METHYLATION_CANDIDATE_SCORES_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_hm450_ica_candidate_scores.csv"
)

METHYLATION_CANDIDATE_LOADINGS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_hm450_ica_candidate_probe_loadings.csv"
)

CROSS_OMIC_PROJECT_CORRELATIONS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_candidate_project_correlations.csv"
)

SEX_SENSITIVITY_CORRELATIONS_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_cross_omic_matched_sex_sensitivity.csv"
)

PROGRAM_DISCOVERY_METADATA_PATH = (
    PROGRAM_DISCOVERY_DIR
    / "tcga_primary_tumor_program_discovery_metadata.json"
)

In [119]:
# =======================================================
# Prepare candidate component export tables
# =======================================================

catalog_rna_components = (
    cross_omic_candidate_catalog[
        "rna_component"
    ]
    .drop_duplicates()
    .to_list()
)

catalog_methylation_components = (
    cross_omic_candidate_catalog[
        "methylation_component"
    ]
    .drop_duplicates()
    .to_list()
)

catalog_rna_component_indices = [
    rna_component_to_index[component]
    for component in catalog_rna_components
]

catalog_methylation_component_indices = [
    methylation_component_to_index[component]
    for component in catalog_methylation_components
]

rna_candidate_score_table = pd.concat(
    [
        sample_metadata.reset_index(drop=True),
        pd.DataFrame(
            rna_ica_scores_oriented[
                :,
                catalog_rna_component_indices,
            ],
            columns=catalog_rna_components,
        ),
    ],
    axis=1,
)

methylation_candidate_score_table = pd.concat(
    [
        methylation_hm450_sample_metadata.reset_index(
            drop=True
        ),
        pd.DataFrame(
            methylation_hm450_ica_scores_oriented[
                :,
                catalog_methylation_component_indices,
            ],
            columns=catalog_methylation_components,
        ),
    ],
    axis=1,
)

rna_candidate_loading_table = pd.concat(
    [
        rna_ica_features.reset_index(drop=True),
        pd.DataFrame(
            rna_ica_gene_loadings_oriented[
                catalog_rna_component_indices,
                :,
            ].T,
            columns=catalog_rna_components,
        ),
    ],
    axis=1,
)

methylation_candidate_loading_table = pd.concat(
    [
        methylation_variable_probe_mapping.reset_index(
            drop=True
        ),
        pd.DataFrame(
            methylation_hm450_ica_probe_loadings_oriented[
                catalog_methylation_component_indices,
                :,
            ].T,
            columns=catalog_methylation_components,
        ),
    ],
    axis=1,
)

In [121]:
# =======================================================
# Prepare project-level correlation export tables
# =======================================================

candidate_status_lookup = (
    cross_omic_candidate_catalog[
        [
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "sex_sensitivity_status",
            "sex_sensitivity_reason",
        ]
    ]
)

cross_omic_project_correlation_table = (
    cross_omic_candidate_project_correlations
    .drop(
        columns=[
            "sex_sensitivity_status",
            "sex_sensitivity_reason",
        ],
        errors="ignore",
    )
    .merge(
        candidate_status_lookup,
        on=[
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="many_to_one",
    )
)

sex_sensitivity_correlation_table = (
    sex_sensitivity_pair_comparison
    .drop(
        columns=[
            "sex_sensitivity_status",
            "sex_sensitivity_reason",
        ],
        errors="ignore",
    )
    .merge(
        candidate_status_lookup,
        on=[
            "candidate_pair",
            "rna_component",
            "methylation_component",
        ],
        how="left",
        validate="many_to_one",
    )
)

In [122]:
# =======================================================
# Assemble program-discovery metadata
# =======================================================

retained_candidate_count = int(
    cross_omic_candidate_catalog[
        "sex_sensitivity_status"
    ]
    .eq("retained_after_sex_sensitivity")
    .sum()
)

excluded_candidate_count = int(
    cross_omic_candidate_catalog[
        "sex_sensitivity_status"
    ]
    .eq("excluded_sex_at_birth_sensitive")
    .sum()
)

program_discovery_metadata = {
    "analysis_id": (
        "205_tcga_epigenetic_transcriptomic_"
        "program_discovery"
    ),
    "analysis_scope": (
        "exploratory_candidate_program_discovery"
    ),
    "scientific_positioning": (
        "Candidate epigenetic-transcriptomic programs "
        "identified through computational association."
    ),
    "cohort": {
        "rna_samples": int(
            rna_candidate_score_table.shape[0]
        ),
        "hm450_samples": int(
            methylation_candidate_score_table.shape[0]
        ),
    },
    "ica": {
        "rna_fitted_components": int(
            RNA_ICA_COMPONENTS
        ),
        "methylation_fitted_components": int(
            METHYLATION_ICA_COMPONENTS
        ),
        "catalog_rna_components": int(
            len(catalog_rna_components)
        ),
        "catalog_methylation_components": int(
            len(catalog_methylation_components)
        ),
    },
    "cross_omic_candidates": {
        "evaluated_pairs": int(
            len(cross_omic_candidate_catalog)
        ),
        "retained_pairs": retained_candidate_count,
        "excluded_pairs": excluded_candidate_count,
        "excluded_pair": "CROSS_OMIC_PAIR_01",
        "exclusion_basis": (
            "Marked attenuation after matched-sample "
            "within-project adjustment for sex_at_birth."
        ),
    },
    "nmf_sensitivity": {
        "scope": (
            "targeted_ica_conditioned_sensitivity"
        ),
        "rna_rank": int(RNA_NMF_COMPONENTS),
        "methylation_rank": int(
            METHYLATION_NMF_COMPONENTS
        ),
        "interpretation": (
            "Descriptive representation concordance only; "
            "no candidate filtering was based on NMF."
        ),
    },
    "sex_at_birth_sensitivity": {
        "eligible_projects": int(
            sex_sensitivity_pair_comparison[
                "project_id"
            ].nunique()
        ),
        "analysis_scope": (
            "matched_binary_sex_subset_5997_samples"
        ),
        "interpretation": (
            "GDC sex_at_birth was used only as a "
            "sensitivity covariate and does not represent "
            "tumor karyotype or sex-chromosome state."
        ),
    },
    "limitations": [
        (
            "Notebook 205 identifies exploratory candidate "
            "programs and does not establish recurrence, "
            "robustness, causality, or clinical utility."
        ),
        (
            "Cross-dataset and lineage-aware robustness "
            "evaluation is deferred to notebook 206."
        ),
        (
            "Targeted NMF is conditioned on ICA-derived "
            "feature and rank selection and is not an "
            "independent validation."
        ),
    ],
    "artifacts": {
        "candidate_pair_catalog": project_relative_path(
            CROSS_OMIC_CANDIDATE_CATALOG_PATH
        ),
        "rna_candidate_scores": project_relative_path(
            RNA_CANDIDATE_SCORES_PATH
        ),
        "rna_candidate_loadings": project_relative_path(
            RNA_CANDIDATE_LOADINGS_PATH
        ),
        "methylation_candidate_scores": (
            project_relative_path(
                METHYLATION_CANDIDATE_SCORES_PATH
            )
        ),
        "methylation_candidate_loadings": (
            project_relative_path(
                METHYLATION_CANDIDATE_LOADINGS_PATH
            )
        ),
        "project_correlations": project_relative_path(
            CROSS_OMIC_PROJECT_CORRELATIONS_PATH
        ),
        "sex_sensitivity_correlations": (
            project_relative_path(
                SEX_SENSITIVITY_CORRELATIONS_PATH
            )
        ),
    },
}

In [123]:
# =======================================================
# Write program-discovery artifacts
# =======================================================

program_discovery_metadata["artifacts"]["metadata"] = (
    project_relative_path(
        PROGRAM_DISCOVERY_METADATA_PATH
    )
)

cross_omic_candidate_catalog.to_csv(
    CROSS_OMIC_CANDIDATE_CATALOG_PATH,
    index=False,
)

rna_candidate_score_table.to_csv(
    RNA_CANDIDATE_SCORES_PATH,
    index=False,
)

rna_candidate_loading_table.to_csv(
    RNA_CANDIDATE_LOADINGS_PATH,
    index=False,
)

methylation_candidate_score_table.to_csv(
    METHYLATION_CANDIDATE_SCORES_PATH,
    index=False,
)

methylation_candidate_loading_table.to_csv(
    METHYLATION_CANDIDATE_LOADINGS_PATH,
    index=False,
)

cross_omic_project_correlation_table.to_csv(
    CROSS_OMIC_PROJECT_CORRELATIONS_PATH,
    index=False,
)

sex_sensitivity_correlation_table.to_csv(
    SEX_SENSITIVITY_CORRELATIONS_PATH,
    index=False,
)

PROGRAM_DISCOVERY_METADATA_PATH.write_text(
    json.dumps(
        program_discovery_metadata,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Program-discovery artifacts written: 8\n"
    f"Directory: "
    f"{project_relative_path(PROGRAM_DISCOVERY_DIR)}"
)

Program-discovery artifacts written: 8
Directory: data/processed/tumor_programs


In [124]:
# =======================================================
# Verify published program-discovery artifacts
# =======================================================

published_artifact_paths = [
    CROSS_OMIC_CANDIDATE_CATALOG_PATH,
    RNA_CANDIDATE_SCORES_PATH,
    RNA_CANDIDATE_LOADINGS_PATH,
    METHYLATION_CANDIDATE_SCORES_PATH,
    METHYLATION_CANDIDATE_LOADINGS_PATH,
    CROSS_OMIC_PROJECT_CORRELATIONS_PATH,
    SEX_SENSITIVITY_CORRELATIONS_PATH,
    PROGRAM_DISCOVERY_METADATA_PATH,
]

published_catalog = pd.read_csv(
    CROSS_OMIC_CANDIDATE_CATALOG_PATH
)

published_metadata = json.loads(
    PROGRAM_DISCOVERY_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

publication_verified = (
    all(
        path.is_file() and path.stat().st_size > 0
        for path in published_artifact_paths
    )
    and published_catalog.shape
    == cross_omic_candidate_catalog.shape
    and published_catalog.columns.to_list()
    == cross_omic_candidate_catalog.columns.to_list()
    and published_metadata
    == program_discovery_metadata
)

print(
    "Program-discovery artifact verification: "
    f"{publication_verified}"
)

Program-discovery artifact verification: True


## Notebook closure

This notebook completed the exploratory discovery of TCGA primary-tumor epigenetic-transcriptomic candidate programs.

RNA-seq and HM450 methylation representations were analyzed separately through lineage-centered ICA and subsequently integrated using within-project cross-omic correlations. Fourteen exploratory candidate pairs were identified.

One pair, `CROSS_OMIC_PAIR_01`, was excluded because its association was strongly attenuated after matched-sample, within-project adjustment for GDC `sex_at_birth`. The remaining 13 pairs were retained as exploratory candidates.

Targeted NMF provided a secondary, ICA-conditioned representation-concordance assessment. It was used descriptively and did not define or filter the retained candidates.

The candidate catalog, component scores, feature loadings, project-level correlations, matched sex-sensitivity results, and analysis metadata were published to:

`data/processed/tumor_programs`

These outputs represent computationally associated candidate programs only. They do not establish biological causality, clinical relevance, cross-dataset robustness, or cross-cancer recurrence. Stability, lineage-aware robustness, and external validation are deferred to notebook 206.